# Case Studies, SSP Future Projections, and Texture Analysis

Produces the case study spatial reconstructions (Figure 7), the QDM-DDPM pipeline
bias reduction analysis (Figure 11), Sobel texture analysis (Figure 12), and
future SSP warming projections (Figure 13).

**Before running:** Update all path variables marked `<SET YOUR PATH HERE>` to your local data directories.

**Paper reference:** Jha, S.K., Gupta, V., et al. (2026). A Generative AI Framework for High-Fidelity Probabilistic Downscaling of CMIP6 Daily Temperature Projections over India. *Communications Earth & Environment.*

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset 
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --- FAST IMPORTS ---
try:
    import rasterio.features
    from rasterio.transform import from_origin
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# --- 1. CONFIGURATION ---
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"
hist_file = "<SET YOUR PATH HERE>"
model_name = "INM_CM5_0"
N_CORES = 64  

scenarios = {
    'SSP 245': os.path.join(base_dir, f"DDPM_ssp_245_{model_name}"),
    'SSP 370': os.path.join(base_dir, f"DDPM_ssp_370_{model_name}"),
    'SSP 585': os.path.join(base_dir, f"DDPM_ssp_585_{model_name}")
}

periods = {
    'Near-term\n(2021-2040)': range(2021, 2041),
    'Mid-term\n(2041-2060)': range(2041, 2061),
    'Far-term\n(2081-2100)': range(2081, 2101)
}

# --- 2. CALCULATE HISTORICAL BASELINE ---
print("--- Step 1: Calculating Historical Baseline ---")
ds_hist = xr.open_dataset(hist_file)
ds_base = ds_hist.sel(time=slice("1995-01-01", "2014-12-31"))
da_baseline = ds_base['tas'].mean(dim='time')
if da_baseline.mean() > 200:
    da_baseline -= 273.15
    print("  [Info] Baseline converted to Celsius.")

# --- 3. FAST PROCESSING ENGINE ---
def process_batch(file_list):
    batch_sum = None
    count = 0
    for f in file_list:
        try:
            with Dataset(f, 'r') as nc:
                val = nc.variables['tas'][0, :, :]
                if np.ma.is_masked(val): data = val.filled(np.nan)
                else: data = val
                if batch_sum is None: batch_sum = np.zeros_like(data, dtype=np.float64)
                batch_sum += data
                count += 1
        except: continue
    return batch_sum, count

def get_period_mean_optimized(folder_path, years):
    all_files = []
    for year in years:
        pattern = os.path.join(folder_path, f"{year}-*-*_{model_name}.nc")
        all_files.extend(glob.glob(pattern))
    all_files = sorted(all_files)
    if not all_files: return None, None
    chunks = np.array_split(all_files, N_CORES)
    results = Parallel(n_jobs=N_CORES)(delayed(process_batch)(chunk) for chunk in chunks)
    final_sum = None
    total_count = 0
    for s, c in results:
        if s is not None:
            if final_sum is None: final_sum = s
            else: final_sum += s
            total_count += c
    if total_count == 0: return None, None
    with xr.open_dataset(all_files[0], decode_times=False) as ds:
        coords = ds.coords
    return final_sum / total_count, coords

# --- 4. MASKING HELPER ---
def create_india_mask(coords, shp_path):
    gdf = gpd.read_file(shp_path)
    lons, lats = np.array(coords['lon']), np.array(coords['lat'])
    lon_res = lons[1] - lons[0]
    lat_res = lats[1] - lats[0]
    transform = rasterio.transform.Affine.translation(lons[0]-lon_res/2, lats[0]-lat_res/2) * rasterio.transform.Affine.scale(lon_res, lat_res)
    return rasterio.features.geometry_mask(gdf.geometry, out_shape=(len(lats), len(lons)), transform=transform, invert=True, all_touched=True)

# --- 5. MAIN LOOP ---
data_store = {s: {} for s in scenarios}
india_mask = None 
baseline_regridded = None 

print("--- Step 2: Processing Anomalies ---")
for scenario_name, folder_path in scenarios.items():
    print(f"  Processing {scenario_name}...") 
    for period_name, year_range in periods.items():
        future_data, coords = get_period_mean_optimized(folder_path, year_range)
        if future_data is not None:
            if np.nanmean(future_data) > 200: future_data -= 273.15
            
            if baseline_regridded is None:
                lat_target, lon_target = coords['lat'], coords['lon']
                target_da = xr.DataArray(future_data, coords=[lat_target, lon_target], dims=['lat', 'lon'])
                baseline_regridded = da_baseline.interp_like(target_da, method='linear').values
                
                india_mask = create_india_mask(coords, shp_path)
                if india_mask.shape != future_data.shape: india_mask = india_mask.squeeze()

            anomaly = future_data - baseline_regridded
            anomaly_masked = np.where(india_mask, anomaly, np.nan)
            baseline_masked = np.where(india_mask, baseline_regridded, np.nan)
            
            data_store[scenario_name][period_name] = (anomaly_masked, baseline_masked, coords)
        else:
            data_store[scenario_name][period_name] = (None, None, None)

# --- 6. PLOTTING ---

# A. Colors (Asymmetric)
levels = np.arange(-6.5, 12.5 + 0.1, 0.5)
neg_bins = len(levels[levels < 0])
pos_bins = len(levels[levels >= 0]) - 1 

colors_neg = plt.cm.Blues_r(np.linspace(0.15, 0.95, neg_bins))
colors_pos = plt.cm.Reds(np.linspace(0.05, 0.95, pos_bins))
color_under = plt.cm.Blues_r(0.0)
color_over = plt.cm.Reds(1.0)
all_colors = np.vstack(([color_under], colors_neg, colors_pos, [color_over]))
custom_cmap = mcolors.ListedColormap(all_colors)
norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=len(all_colors), extend='both')

# B. Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), subplot_kw={'projection': ccrs.PlateCarree()})
gdf_boundary = gpd.read_file(shp_path)
scenario_keys = list(scenarios.keys())
period_keys = list(periods.keys())

# Contours: 5°C Steps
contour_levels = [0, 5, 10, 15, 20, 25, 30]

for row_idx, scenario_name in enumerate(scenario_keys):
    for col_idx, period_name in enumerate(period_keys):
        ax = axes[row_idx, col_idx]
        anomaly_data, mean_data, coords = data_store[scenario_name][period_name]
        
        ax.set_extent([66, 100, 6, 38], crs=ccrs.PlateCarree())

        # Large Labels for 100% Zoom Legibility
        if col_idx == 0:
            ax.text(-0.07, 0.55, scenario_name, va='bottom', ha='center',
                    rotation='vertical', transform=ax.transAxes, fontsize=20, fontweight='bold')
        if row_idx == 0:
            ax.set_title(period_name, fontsize=20, fontweight='bold', pad=20)
            
        if anomaly_data is not None:
            # 1. Shading: Anomaly
            im = ax.pcolormesh(coords['lon'], coords['lat'], anomaly_data,
                               transform=ccrs.PlateCarree(),
                               cmap=custom_cmap, 
                               norm=norm)
            
            # 2. Contours: Historical Mean
            cs = ax.contour(coords['lon'], coords['lat'], mean_data, 
                            levels=contour_levels, 
                            colors='0.3', 
                            linewidths=1.0,
                            transform=ccrs.PlateCarree())
            
            ax.clabel(cs, inline=True, fontsize=11, fmt='%1.0f') 
            
            # 3. Boundaries
            ax.add_geometries(gdf_boundary.geometry, ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black', linewidth=1.0)
            ax.coastlines(alpha=0.6)

            # --- STATS & HISTOGRAM (Modified Size) ---
            valid_vals = anomaly_data[~np.isnan(anomaly_data)]
            if len(valid_vals) > 0:
                mu = np.mean(valid_vals)
                sigma = np.std(valid_vals)
                
                # A. Statistics Text (Larger Font)
                stats_text = f"$\mu_{{anom}}={mu:.2f}$\n$\sigma_{{anom}}={sigma:.2f}$"
                ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, 
                        ha='right', va='top', fontsize=14, fontweight='bold',
                        bbox=dict(facecolor='white', alpha=0.9, edgecolor='none', boxstyle='round,pad=0.3'))
                
                # B. Inset Histogram (INCREASED SIZE)
                # width/height increased to 40% / 30%
                ax_ins = inset_axes(ax, width="40%", height="30%", loc='lower right', 
                                    bbox_to_anchor=(0, 0.12, 1, 1), 
                                    bbox_transform=ax.transAxes,
                                    borderpad=1)
                
                # Calculate Histogram Data
                counts, bins = np.histogram(valid_vals, bins=30, density=True)
                bin_centers = 0.5 * (bins[:-1] + bins[1:])
                
                # Plot Colored Bars
                bar_colors = custom_cmap(norm(bin_centers))
                ax_ins.bar(bin_centers, counts, width=bins[1] - bins[0], color=bar_colors, edgecolor='black', linewidth=0.5)

                # Add Zero Line
                ax_ins.axvline(0, color='black', linestyle='--', linewidth=1)

                # Clean but visible aesthetics
                ax_ins.set_facecolor('none')
                ax_ins.spines['top'].set_visible(False)
                ax_ins.spines['right'].set_visible(False)
                ax_ins.spines['left'].set_visible(False)
                ax_ins.set_yticks([]) 
                
                # Larger ticks and label for legibility
                ax_ins.set_xticks([-5, 0, 5, 10])
                ax_ins.tick_params(axis='x', labelsize=10, colors='black', direction='out', length=4)
                ax_ins.set_xlabel("Anomaly (°C)", fontsize=10, labelpad=2)

        else:
            ax.text(0.5, 0.5, "No Data", ha='center', transform=ax.transAxes, fontsize=16)

# Colorbar (Larger Text)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
ticks = [-6, -4, -2, 0, 2, 4, 6, 8, 10, 12]
cbar = fig.colorbar(im, cax=cbar_ax, ticks=ticks)
cbar.set_label('Shading: Temperature Anomaly (°C)\nContours: Hist. Mean (5°C intervals)', fontsize=18)
cbar.ax.tick_params(labelsize=14)

# --- 7. SAVING ---
output_filename = f"Temp_Anomaly_Final_LargeHist_{model_name}.png"
save_path = os.path.join(base_dir, output_filename)
print(f"Saving plot to {save_path} ...")
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print("Saved successfully.")
plt.show()

In [ ]:
import xarray as xr
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset 
import pandas as pd

# --- FAST IMPORTS ---
try:
    import rasterio.features
    from rasterio.transform import from_origin
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# --- 1. CONFIGURATION ---
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"
hist_file = "<SET YOUR PATH HERE>"
model_name = "INM_CM5_0"
N_CORES = 80 

scenarios = {
    'SSP 245': os.path.join(base_dir, f"DDPM_ssp_245_{model_name}"),
    'SSP 370': os.path.join(base_dir, f"DDPM_ssp_370_{model_name}"),
    'SSP 585': os.path.join(base_dir, f"DDPM_ssp_585_{model_name}")
}

periods = {
    'Near-term (2021-2040)': range(2021, 2041),
    'Mid-term (2041-2060)': range(2041, 2061),
    'Far-term (2081-2100)': range(2081, 2101)
}

# --- 2. CALCULATE HISTORICAL BASELINE ---
print("--- Step 1: Calculating Historical Baseline ---")
ds_hist = xr.open_dataset(hist_file)
ds_base = ds_hist.sel(time=slice("1995-01-01", "2014-12-31"))
da_baseline = ds_base['tas'].mean(dim='time')
if da_baseline.mean() > 200:
    da_baseline -= 273.15
    print("  [Info] Baseline converted to Celsius.")

# --- 3. PROCESSING ENGINE ---
def process_batch(file_list):
    batch_sum = None
    count = 0
    for f in file_list:
        try:
            with Dataset(f, 'r') as nc:
                val = nc.variables['tas'][0, :, :]
                if np.ma.is_masked(val): data = val.filled(np.nan)
                else: data = val
                if batch_sum is None: batch_sum = np.zeros_like(data, dtype=np.float64)
                batch_sum += data
                count += 1
        except: continue
    return batch_sum, count

def get_period_mean_optimized(folder_path, years):
    all_files = []
    for year in years:
        pattern = os.path.join(folder_path, f"{year}-*-*_{model_name}.nc")
        all_files.extend(glob.glob(pattern))
    all_files = sorted(all_files)
    if not all_files: return None, None
    chunks = np.array_split(all_files, N_CORES)
    results = Parallel(n_jobs=N_CORES)(delayed(process_batch)(chunk) for chunk in chunks)
    final_sum = None
    total_count = 0
    for s, c in results:
        if s is not None:
            if final_sum is None: final_sum = s
            else: final_sum += s
            total_count += c
    if total_count == 0: return None, None
    with xr.open_dataset(all_files[0], decode_times=False) as ds:
        coords = ds.coords
    return final_sum / total_count, coords

def create_india_mask(coords, shp_path):
    gdf = gpd.read_file(shp_path)
    lons, lats = np.array(coords['lon']), np.array(coords['lat'])
    lon_res = lons[1] - lons[0]
    lat_res = lats[1] - lats[0]
    transform = rasterio.transform.Affine.translation(lons[0]-lon_res/2, lats[0]-lat_res/2) * rasterio.transform.Affine.scale(lon_res, lat_res)
    return rasterio.features.geometry_mask(gdf.geometry, out_shape=(len(lats), len(lons)), transform=transform, invert=True, all_touched=True)

# --- 4. MAIN STATISTICAL LOOP ---
india_mask = None 
baseline_regridded = None 

print("\n" + "="*60)
print(f"   DETAILED CLIMATE STATISTICS REPORT: {model_name}")
print("="*60 + "\n")

for scenario_name, folder_path in scenarios.items():
    print(f"MODEL SCENARIO: {scenario_name}")
    print("-" * 60)
    
    for period_name, year_range in periods.items():
        future_data, coords = get_period_mean_optimized(folder_path, year_range)
        
        if future_data is not None:
            if np.nanmean(future_data) > 200: future_data -= 273.15
            
            # Prepare Grid & Baseline (Run Once)
            if baseline_regridded is None:
                lat_target, lon_target = coords['lat'], coords['lon']
                target_da = xr.DataArray(future_data, coords=[lat_target, lon_target], dims=['lat', 'lon'])
                baseline_regridded = da_baseline.interp_like(target_da, method='linear').values
                india_mask = create_india_mask(coords, shp_path)
                if india_mask.shape != future_data.shape: india_mask = india_mask.squeeze()

            # Calculate Anomaly
            anomaly = future_data - baseline_regridded
            
            # Mask Data (Keep only India pixels)
            valid_anom = anomaly[india_mask]
            valid_base = baseline_regridded[india_mask]
            
            # Remove NaNs (just in case)
            mask_nan = ~np.isnan(valid_anom)
            valid_anom = valid_anom[mask_nan]
            valid_base = valid_base[mask_nan]
            
            # --- A. BASIC STATISTICS ---
            mean_val = np.mean(valid_anom)
            median_val = np.median(valid_anom)
            std_val = np.std(valid_anom)
            min_val = np.min(valid_anom)
            max_val = np.max(valid_anom)
            p1 = np.percentile(valid_anom, 1)
            p99 = np.percentile(valid_anom, 99)
            
            # --- B. DISTRIBUTION ANALYSIS (Cooling vs Warming) ---
            # Threshold for "Neutral" is small (e.g., +/- 0.1 C)
            cooling_pct = (valid_anom < -0.1).mean() * 100
            neutral_pct = ((valid_anom >= -0.1) & (valid_anom <= 0.1)).mean() * 100
            warming_pct = (valid_anom > 0.1).mean() * 100
            
            # --- C. CONTOUR-WISE (ZONAL) ANALYSIS ---
            # We classify pixels based on their HISTORICAL BASELINE temperature
            # Zone 1: Himalayas/High Altitude (< 10°C)
            # Zone 2: North/Transition (10-20°C)
            # Zone 3: Warm Regions (20-25°C)
            # Zone 4: Hot Regions (> 25°C)
            
            mask_z1 = valid_base < 10
            mask_z2 = (valid_base >= 10) & (valid_base < 20)
            mask_z3 = (valid_base >= 20) & (valid_base < 25)
            mask_z4 = valid_base >= 25
            
            z1_anom = valid_anom[mask_z1].mean() if np.any(mask_z1) else np.nan
            z2_anom = valid_anom[mask_z2].mean() if np.any(mask_z2) else np.nan
            z3_anom = valid_anom[mask_z3].mean() if np.any(mask_z3) else np.nan
            z4_anom = valid_anom[mask_z4].mean() if np.any(mask_z4) else np.nan
            
            # --- PRINT REPORT ---
            print(f"  > PERIOD: {period_name.replace(chr(10), ' ')}") # Remove newline
            print(f"    Global Stats:  Mean={mean_val:5.2f}°C | Median={median_val:5.2f}°C | Std={std_val:5.2f} | Range=[{min_val:.1f}, {max_val:.1f}]")
            print(f"    Distribution:  Cooling={cooling_pct:5.1f}% | Neutral={neutral_pct:5.1f}% | Warming={warming_pct:5.1f}%")
            print(f"    Zonal Mean Anomaly (Amplification):")
            print(f"      - Cold Zone (<10°C, Himalayas):  {z1_anom:5.2f}°C")
            print(f"      - Cool Zone (10-20°C):           {z2_anom:5.2f}°C")
            print(f"      - Warm Zone (20-25°C):           {z3_anom:5.2f}°C")
            print(f"      - Hot Zone  (>25°C):             {z4_anom:5.2f}°C")
            print("")
            
    print("-" * 60)
print("Analysis Complete.")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import os
import glob
import numpy as np
import geopandas as gpd

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"

# MSWX BASELINE (LR)
mswx_dir = "<SET YOUR PATH HERE>"

model_name = "INM_CM5_0_RAW_1deg_MSWX_BASELINE_SIMPLE"

raw_files = {
    "SSP 245": "<SET YOUR PATH HERE>",
    "SSP 370": "<SET YOUR PATH HERE>",
    "SSP 585": "<SET YOUR PATH HERE>",
}

periods = {
    "Near-term\n(2021–2040)": slice("2021-01-01", "2040-12-31"),
    "Mid-term\n(2041–2060)": slice("2041-01-01", "2060-12-31"),
    "Far-term\n(2081–2100)": slice("2081-01-01", "2100-12-31"),
}

# ==========================================================
# 2. MSWX HISTORICAL BASELINE (1995–2014)
# ==========================================================
print("--- Step 1: Calculating MSWX Historical Baseline (1995–2014) ---")

def load_mswx_year(year):
    files = sorted(glob.glob(os.path.join(mswx_dir, f"{year}*.nc")))
    if not files:
        return None

    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]

    # enforce south → north
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])

    return da.mean("time")

yearly_means = []
for y in range(1995, 2015):
    da_y = load_mswx_year(y)
    if da_y is not None:
        yearly_means.append(da_y)

da_mswx_baseline = xr.concat(yearly_means, dim="year").mean("year")

print("  MSWX baseline ready:", da_mswx_baseline.shape)

# ==========================================================
# 3. INDIA MASK
# ==========================================================
def create_india_mask(coords, shp_path):
    gdf = gpd.read_file(shp_path)
    lons = np.asarray(coords["lon"])
    lats = np.asarray(coords["lat"])

    lon_res = abs(lons[1] - lons[0])
    lat_res = abs(lats[1] - lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res / 2,
            lats.max() + lat_res / 2
        )
        * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True,
        all_touched=True
    )
    return np.flipud(mask)

# ==========================================================
# 4. PROCESS RAW DATA
# ==========================================================
data_store = {s: {} for s in raw_files}
baseline_regridded = None
india_mask = None
all_anoms = []

for scenario, path in raw_files.items():
    ds = xr.open_dataset(path)

    if ds.lat[0] > ds.lat[-1]:
        ds = ds.reindex(lat=ds.lat[::-1])

    for pname, tslice in periods.items():
        da_future = ds["tas"].sel(time=tslice).mean("time")

        if da_future.mean() > 200:
            da_future -= 273.15

        if baseline_regridded is None:
            baseline_regridded = da_mswx_baseline.interp_like(
                da_future, method="linear"
            ).values
            india_mask = create_india_mask(da_future.coords, shp_path)

        anomaly = da_future.values - baseline_regridded
        anomaly = np.where(india_mask, anomaly, np.nan)
        mean_masked = np.where(india_mask, da_future.values, np.nan)

        data_store[scenario][pname] = (anomaly, mean_masked, da_future.coords)
        all_anoms.append(anomaly[~np.isnan(anomaly)])

# ==========================================================
# 5. COLORMAPS & NORMALIZATION
# ==========================================================
full = np.concatenate(all_anoms)
vmin = np.floor(np.percentile(full, 1) * 2) / 2
vmax = np.ceil(np.percentile(full, 99) * 2) / 2

print(f"Anomaly range used: {vmin} to {vmax} °C")

if vmin < 0 < vmax:
    anom_norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    anom_cmap = plt.cm.RdBu_r
else:
    anom_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    anom_cmap = plt.cm.Reds

mean_cmap = plt.cm.Spectral_r

# ==========================================================
# 5A. DETAILED STATISTICS (PRINT-READY)
# ==========================================================
print("\n" + "="*80)
print("INDIA-WIDE TEMPERATURE STATISTICS (RAW CMIP6 vs MSWX BASELINE)")
print("="*80)

def compute_stats(anom, mean):
    vals = anom[~np.isnan(anom)]
    mean_vals = mean[~np.isnan(mean)]

    stats = {
        "mean_temp": np.mean(mean_vals),
        "mean_anom": np.mean(vals),
        "median_anom": np.median(vals),
        "std_anom": np.std(vals),
        "p05_anom": np.percentile(vals, 5),
        "p95_anom": np.percentile(vals, 95),
        "min_anom": np.min(vals),
        "max_anom": np.max(vals),
        "area_gt_1C": np.sum(vals > 1.0) / len(vals) * 100,
        "area_gt_2C": np.sum(vals > 2.0) / len(vals) * 100,
        "area_gt_3C": np.sum(vals > 3.0) / len(vals) * 100,
    }
    return stats

for scenario in raw_files:
    print(f"\n--- {scenario} ---")
    for pname in periods:
        anom, mean, _ = data_store[scenario][pname]
        s = compute_stats(anom, mean)

        print(f"\n{pname}")
        print(f"  Mean temperature        : {s['mean_temp']:.2f} °C")
        print(f"  Mean anomaly            : {s['mean_anom']:.2f} °C")
        print(f"  Median anomaly          : {s['median_anom']:.2f} °C")
        print(f"  Spatial std (anomaly)   : {s['std_anom']:.2f} °C")
        print(f"  5–95% anomaly range     : [{s['p05_anom']:.2f}, {s['p95_anom']:.2f}] °C")
        print(f"  Min / Max anomaly       : [{s['min_anom']:.2f}, {s['max_anom']:.2f}] °C")
        print(f"  Area > +1°C             : {s['area_gt_1C']:.1f} %")
        print(f"  Area > +2°C             : {s['area_gt_2C']:.1f} %")
        print(f"  Area > +3°C             : {s['area_gt_3C']:.1f} %")

# ==========================================================
# 6. PLOTTING (18 PANELS — CLEAN)
# ==========================================================
fig, axes = plt.subplots(
    3, 6, figsize=(30, 14),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

gdf_boundary = gpd.read_file(shp_path)

for r, scenario in enumerate(raw_files):
    for p, pname in enumerate(periods):

        anom, mean, coords = data_store[scenario][pname]
        axA = axes[r, p * 2]
        axM = axes[r, p * 2 + 1]

        for ax in (axA, axM):
            ax.set_extent([66, 100, 6, 38])
            ax.add_geometries(
                gdf_boundary.geometry,
                ccrs.PlateCarree(),
                facecolor="none",
                edgecolor="black",
                linewidth=1.0
            )

        # --- Anomaly ---
        imA = axA.pcolormesh(
            coords["lon"], coords["lat"], anom,
            cmap=anom_cmap, norm=anom_norm
        )

        # --- Mean ---
        imM = axM.pcolormesh(
            coords["lon"], coords["lat"], mean,
            cmap=mean_cmap
        )

        if r == 0:
            axA.set_title(f"{pname}\nAnomaly", fontsize=16)
            axM.set_title(f"{pname}\nMean", fontsize=16)

        if p == 0:
            axA.text(
                -0.12, 0.5, scenario,
                rotation=90,
                transform=axA.transAxes,
                va="center", ha="center",
                fontsize=18, fontweight="bold"
            )

# ==========================================================
# 7. COLORBARS
# ==========================================================
caxA = fig.add_axes([0.92, 0.18, 0.015, 0.65])
fig.colorbar(imA, cax=caxA, label="Temperature Anomaly (°C)")

caxM = fig.add_axes([0.15, 0.06, 0.65, 0.025])
fig.colorbar(imM, cax=caxM, orientation="horizontal",
             label="Mean Temperature (°C)")

plt.show()

In [ ]:
import os

# ==========================================================
# CRITICAL FIX 1: Disable HDF5 File Locking
# Must be done BEFORE importing xarray/netCDF4 to prevent parallel crashes
# ==========================================================
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import glob
from scipy.stats import skew
from dask.distributed import Client, LocalCluster

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("pip install rasterio")

# ==========================================================
# 0. SETUP CLUSTER
# ==========================================================
if __name__ == "__main__":
    # We use 10 workers with 8 threads each = 80 CPUs total.
    # This prevents the "Worker exceeded memory budget" crash.
    cluster = LocalCluster(
        n_workers=10,           
        threads_per_worker=8,   
        memory_limit='auto',
        processes=True
    )
    client = Client(cluster)
    print(f"🚀 Stable Cluster Started: {client}")
    print(f"   Dashboard: {client.dashboard_link}")

    # ==========================================================
    # 1. CONFIGURATION
    # ==========================================================
    base_dir = "<SET YOUR PATH HERE>"
    shp_path  = "<SET YOUR PATH HERE>"
    mswx_dir = "<SET YOUR PATH HERE>"

    raw_files = {
        "SSP 245": "<SET YOUR PATH HERE>",
        "SSP 370": "<SET YOUR PATH HERE>",
        "SSP 585": "<SET YOUR PATH HERE>",
    }

    periods = {
        "Near-term (2021–2040)": slice("2021-01-01", "2040-12-31"),
        "Mid-term (2041–2060)": slice("2041-01-01", "2060-12-31"),
        "Far-term (2081–2100)": slice("2081-01-01", "2100-12-31"),
    }

    # ==========================================================
    # 2. MSWX BASELINE (FIXED LOAD)
    # ==========================================================
    print("Reading MSWX Baseline...")
    
    # CRITICAL FIX 2: parallel=False here prevents the "Unknown file format" error.
    # The actual computations later will still run on all 80 cores.
    ds_mswx = xr.open_mfdataset(
        os.path.join(mswx_dir, "*.nc"), 
        combine="by_coords", 
        parallel=False,    
        chunks={"time": 365, "lat": "auto", "lon": "auto"}
    )
    
    ds_mswx = ds_mswx.sel(time=slice("1995-01-01", "2014-12-31"))
    da_base = ds_mswx["air_temperature"]
    
    if da_base.lat[0] > da_base.lat[-1]:
        da_base = da_base.reindex(lat=da_base.lat[::-1])

    # Lazy mean calculation (dask)
    baseline_lazy = da_base.mean("time")

    # ==========================================================
    # 3. INDIA MASK FUNCTION
    # ==========================================================
    def create_india_mask(coords_dict):
        gdf = gpd.read_file(shp_path)
        lons = coords_dict["lon"]
        lats = coords_dict["lat"]

        transform = (
            rasterio.transform.Affine.translation(
                lons.min() - (lons[1]-lons[0])/2,
                lats.max() + (lats[1]-lats[0])/2
            )
            * rasterio.transform.Affine.scale(
                abs(lons[1]-lons[0]),
                -abs(lats[1]-lats[0])
            )
        )

        mask = rasterio.features.geometry_mask(
            gdf.geometry,
            out_shape=(len(lats), len(lons)),
            transform=transform,
            invert=True
        )
        return np.flipud(mask)

    # ==========================================================
    # 4. STATS HELPER
    # ==========================================================
    def calculate_stats_numpy(a):
        a = a[~np.isnan(a)]
        if len(a) == 0: return {}
        return {
            "mean": np.mean(a), "median": np.median(a), "std": np.std(a),
            "skew": skew(a), "p05": np.percentile(a, 5), "p95": np.percentile(a, 95),
            "p01": np.percentile(a, 1), "p99": np.percentile(a, 99),
            "area_>1": np.mean(a > 1) * 100, "area_>2": np.mean(a > 2) * 100,
            "area_>3": np.mean(a > 3) * 100, "values": a 
        }

    # ==========================================================
    # 5. PREPARE LAZY COMPUTATIONS (CMIP6)
    # ==========================================================
    print("Building Task Graph for Scenarios...")
    
    india_mask = None
    lazy_results = {} 
    
    # Load first file to generate the mask
    first_scen = list(raw_files.keys())[0]
    ds_temp = xr.open_dataset(raw_files[first_scen])
    if ds_temp.lat[0] > ds_temp.lat[-1]:
        ds_temp = ds_temp.reindex(lat=ds_temp.lat[::-1])
    
    # Create mask using only 2D coordinates
    coords_subset = {"lat": ds_temp.lat.values, "lon": ds_temp.lon.values}
    india_mask = create_india_mask(coords_subset)
    
    mask_da = xr.DataArray(
        india_mask, 
        coords={"lat": ds_temp.lat, "lon": ds_temp.lon}, 
        dims=("lat", "lon")
    )

    for scen, path in raw_files.items():
        # Open with chunks for parallel processing
        ds = xr.open_dataset(path, chunks={"time": 365}) 

        if ds.lat[0] > ds.lat[-1]:
            ds = ds.reindex(lat=ds.lat[::-1])

        da = ds["tas"]
        # Check units (lazy compute)
        if da.mean().compute() > 200: 
            da = da - 273.15

        # 5a. Annual mean (India)
        da_masked = da.where(mask_da)
        annual_lazy = da_masked.mean(dim=["lat", "lon"]).groupby("time.year").mean()
        lazy_results[f"{scen}_annual"] = annual_lazy

        # 5b. Period-wise Anomalies
        for pname, tslice in periods.items():
            mean_p = da.sel(time=tslice).mean("time")
            
            # Interpolate baseline to this scenario's grid
            base_rg = baseline_lazy.interp_like(mean_p)
            
            anom = mean_p - base_rg
            anom = anom.where(mask_da)
            
            lazy_results[f"{scen}_{pname}_anom"] = anom

    # ==========================================================
    # 6. EXECUTE ALL COMPUTATIONS (PARALLEL)
    # ==========================================================
    print("🔥 Executing calculations on 80 cores...")
    
    # Compute everything at once
    computed_data = client.compute(lazy_results, sync=True)
    
    final_results = {}
    for key, data in computed_data.items():
        if key.endswith("_annual"):
            scen = key.replace("_annual", "")
            if scen not in final_results: final_results[scen] = {}
            final_results[scen]["annual"] = data
        
        elif key.endswith("_anom"):
            # Identify scenario from key
            for s_key in raw_files:
                if key.startswith(s_key):
                    scen = s_key
                    pname = key.replace(f"{s_key}_", "").replace("_anom", "")
                    break
            if scen not in final_results: final_results[scen] = {}
            final_results[scen][pname] = calculate_stats_numpy(data.values)

    print("✅ Calculations complete.")

    # ==========================================================
    # 7. PLOTTING
    # ==========================================================
    # Histograms
    for pname in periods:
        plt.figure(figsize=(7,5))
        for scen in raw_files:
            vals = final_results[scen][pname]["values"]
            plt.hist(vals, bins=60, density=True, alpha=0.4, label=scen)
        plt.title(f"Temperature Anomaly Distribution\n{pname}")
        plt.xlabel("Temperature Anomaly (°C)")
        plt.ylabel("Density")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

    # Time Series
    plt.figure(figsize=(10,5))
    for scen in raw_files:
        y = final_results[scen]["annual"]
        t = y.year.values 
        coef = np.polyfit(t, y.values, 1) # simple linear trend
        plt.plot(t, y, label=f"{scen}")
        plt.plot(t, coef[0]*t + coef[1], "--", alpha=0.7)
        print(f"{scen} warming trend: {coef[0]*10:.2f} °C/decade")

    plt.title("India-Mean Temperature")
    plt.ylabel("Temperature (°C)")
    plt.xlabel("Year")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # ==========================================================
    # 8. PRINT STATISTICS
    # ==========================================================
    for scen in raw_files:
        print(f"\n=== {scen} ===")
        for pname in periods:
            if pname in final_results[scen]:
                s = final_results[scen][pname]
                print(f"{pname}")
                for k,v in s.items():
                    if k != "values": 
                        print(f"  {k:10s}: {v:.2f}")

    # Close cluster
    client.close()
    cluster.close()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path  = "<SET YOUR PATH HERE>"

# MSWX-HR BASELINE
mswx_hr_dir = "<SET YOUR PATH HERE>"

model_name = "INM_CM5_0_QDM_MSWX_HR_SIMPLE"
N_CORES = 80

scenarios = {
    "SSP 245": "<SET YOUR PATH HERE>",
    "SSP 370": "<SET YOUR PATH HERE>",
    "SSP 585": "<SET YOUR PATH HERE>",
}

periods = {
    "Near-term\n(2021–2040)": range(2021, 2041),
    "Mid-term\n(2041–2060)": range(2041, 2061),
    "Far-term\n(2081–2100)": range(2081, 2101),
}

# ==========================================================
# 2. MSWX-HR BASELINE (1995–2014)
# ==========================================================
print("--- Step 1: Computing MSWX-HR baseline (1995–2014) ---")

def load_mswx_hr_year(year):
    files = sorted(glob.glob(os.path.join(mswx_hr_dir, f"{year}*.nc")))
    if not files:
        return None

    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]

    # enforce south → north
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])

    return da.mean("time")

baseline_years = []
for y in range(1995, 2015):
    da_y = load_mswx_hr_year(y)
    if da_y is not None:
        baseline_years.append(da_y)

da_mswx_baseline = xr.concat(baseline_years, dim="year").mean("year")

print("  MSWX-HR baseline ready:", da_mswx_baseline.shape)

# ==========================================================
# 3. QDM FAST PROCESSING
# ==========================================================
def process_batch(files):
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]
                if data.ndim == 3:
                    data = data[0]
                data = np.asarray(data, dtype=np.float64)
                if s is None:
                    s = np.zeros_like(data)
                s += data
                c += 1
        except:
            continue
    return s, c

def get_period_mean_qdm(folder, years):
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(folder, f"{y}-*-*_INM_CM5_0.nc")))
    if not files:
        return None, None, None

    chunks = np.array_split(sorted(files), N_CORES)
    results = Parallel(n_jobs=N_CORES)(
        delayed(process_batch)(c) for c in chunks
    )

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0:
        return None, None, None

    with xr.open_dataset(files[0]) as ds:
        lats = ds.lat.values
        lons = ds.lon.values

    mean = total / count

    # enforce south → north
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]

    return mean, lats, lons

# ==========================================================
# 4. INDIA MASK
# ==========================================================
def create_india_mask(coords, shp_path):
    gdf = gpd.read_file(shp_path)
    lons = np.array(coords["lon"])
    lats = np.array(coords["lat"])

    lon_res = abs(lons[1] - lons[0])
    lat_res = abs(lats[1] - lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res / 2,
            lats.max() + lat_res / 2
        )
        * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True,
        all_touched=True
    )
    return np.flipud(mask)

# ==========================================================
# 5. PROCESS DATA (QDM − MSWX-HR)
# ==========================================================
data_store = {s: {} for s in scenarios}
baseline_regridded = None
india_mask = None
all_anoms = []

for scen, path in scenarios.items():
    for pname, yrs in periods.items():
        mean_future, lats, lons = get_period_mean_qdm(path, yrs)
        if mean_future is None:
            continue

        if np.nanmean(mean_future) > 200:
            mean_future -= 273.15

        if baseline_regridded is None:
            target_da = xr.DataArray(
                mean_future,
                coords={"lat": lats, "lon": lons},
                dims=["lat", "lon"]
            )
            baseline_regridded = da_mswx_baseline.interp_like(
                target_da, method="linear"
            ).values
            india_mask = create_india_mask(target_da.coords, shp_path)

        anomaly = mean_future - baseline_regridded
        anomaly = np.where(india_mask, anomaly, np.nan)
        mean_masked = np.where(india_mask, mean_future, np.nan)

        data_store[scen][pname] = (anomaly, mean_masked, target_da.coords)
        all_anoms.append(anomaly[~np.isnan(anomaly)])

# ==========================================================
# 5A. INDIA-WIDE STATISTICS (QDM vs MSWX-HR)
# ==========================================================
print("\n" + "=" * 90)
print("INDIA-WIDE TEMPERATURE STATISTICS (QDM relative to MSWX-HR baseline)")
print("=" * 90)

def compute_stats(anom, mean):
    vals = anom[~np.isnan(anom)]
    mean_vals = mean[~np.isnan(mean)]

    return {
        "mean_temp": np.mean(mean_vals),
        "mean_anom": np.mean(vals),
        "median_anom": np.median(vals),
        "std_anom": np.std(vals),
        "p05": np.percentile(vals, 5),
        "p95": np.percentile(vals, 95),
        "min": np.min(vals),
        "max": np.max(vals),
        "area_gt_1": np.sum(vals > 1.0) / len(vals) * 100,
        "area_gt_2": np.sum(vals > 2.0) / len(vals) * 100,
        "area_gt_3": np.sum(vals > 3.0) / len(vals) * 100,
    }

for scen in scenarios:
    print(f"\n### {scen}")
    for pname in periods:
        anom, mean, _ = data_store[scen][pname]
        s = compute_stats(anom, mean)

        print(f"\n{pname}")
        print(f"  Mean temperature        : {s['mean_temp']:.2f} °C")
        print(f"  Mean anomaly            : {s['mean_anom']:.2f} °C")
        print(f"  Median anomaly          : {s['median_anom']:.2f} °C")
        print(f"  Spatial std (anomaly)   : {s['std_anom']:.2f} °C")
        print(f"  5–95% anomaly range     : [{s['p05']:.2f}, {s['p95']:.2f}] °C")
        print(f"  Min / Max anomaly       : [{s['min']:.2f}, {s['max']:.2f}] °C")
        print(f"  Area > +1°C             : {s['area_gt_1']:.1f} %")
        print(f"  Area > +2°C             : {s['area_gt_2']:.1f} %")
        print(f"  Area > +3°C             : {s['area_gt_3']:.1f} %")

# ==========================================================
# 6. COLORMAPS & NORMS
# ==========================================================
full = np.concatenate(all_anoms)
vmin = np.floor(np.percentile(full, 1) * 2) / 2
vmax = np.ceil(np.percentile(full, 99) * 2) / 2
print(f"Anomaly range used: {vmin} to {vmax} °C")

if vmin < 0 < vmax:
    anom_norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    anom_cmap = plt.cm.RdBu_r
else:
    anom_norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    anom_cmap = plt.cm.Reds

mean_cmap = plt.cm.Spectral_r

# ==========================================================
# 7. PLOTTING (18 PANELS — SIMPLE)
# ==========================================================
fig, axes = plt.subplots(
    3, 6, figsize=(30, 14),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

gdf_boundary = gpd.read_file(shp_path)

for r, scen in enumerate(scenarios):
    for p, pname in enumerate(periods):

        anom, mean, coords = data_store[scen][pname]
        axA = axes[r, p * 2]
        axM = axes[r, p * 2 + 1]

        for ax in (axA, axM):
            ax.set_extent([66, 100, 6, 38])
            ax.add_geometries(
                gdf_boundary.geometry,
                ccrs.PlateCarree(),
                facecolor="none",
                edgecolor="black",
                linewidth=1.0
            )
            # ax.coastlines(alpha=0.6)

        # --- Anomaly ---
        imA = axA.pcolormesh(
            coords["lon"], coords["lat"], anom,
            cmap=anom_cmap, norm=anom_norm
        )

        # --- Mean ---
        imM = axM.pcolormesh(
            coords["lon"], coords["lat"], mean,
            cmap=mean_cmap
        )

        if r == 0:
            axA.set_title(f"{pname}\nAnomaly", fontsize=16)
            axM.set_title(f"{pname}\nMean", fontsize=16)

        if p == 0:
            axA.text(
                -0.12, 0.5, scen,
                rotation=90,
                transform=axA.transAxes,
                va="center", ha="center",
                fontsize=18, fontweight="bold"
            )

# ==========================================================
# 8. COLORBARS
# ==========================================================
caxA = fig.add_axes([0.92, 0.18, 0.015, 0.65])
fig.colorbar(imA, cax=caxA, label="QDM Temperature Anomaly (°C)")

caxM = fig.add_axes([0.15, 0.06, 0.65, 0.025])
fig.colorbar(imM, cax=caxM, orientation="horizontal",
             label="Mean Temperature (°C)")

plt.show()

In [ ]:
import os, glob
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import skew
from netCDF4 import Dataset
from joblib import Parallel, delayed

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path  = "<SET YOUR PATH HERE>"

mswx_hr_dir = "<SET YOUR PATH HERE>"

scenarios = {
    "SSP 245": "<SET YOUR PATH HERE>",
    "SSP 370": "<SET YOUR PATH HERE>",
    "SSP 585": "<SET YOUR PATH HERE>",
}

periods = {
    "Near-term (2021–2040)": range(2021, 2041),
    "Mid-term (2041–2060)": range(2041, 2061),
    "Far-term (2081–2100)": range(2081, 2101),
}

N_JOBS = 64  # safe on your node

# ==========================================================
# 2. MSWX-HR BASELINE (SERIAL, SAFE)
# ==========================================================
print("--- Step 1: Computing MSWX-HR baseline ---")

def load_mswx_year(year):
    files = sorted(glob.glob(os.path.join(mswx_hr_dir, f"{year}*.nc")))
    if not files: return None
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])
    return da.mean("time")

base_list = []
for y in range(1995, 2015):
    da_y = load_mswx_year(y)
    if da_y is not None:
        base_list.append(da_y)

da_baseline = xr.concat(base_list, dim="year").mean("year")
print("Baseline shape:", da_baseline.shape)

# ==========================================================
# 3. INDIA MASK (ONCE)
# ==========================================================
def create_india_mask(lats, lons):
    gdf = gpd.read_file(shp_path)
    import rasterio.transform, rasterio.features

    lon_res = abs(lons[1]-lons[0])
    lat_res = abs(lats[1]-lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res/2,
            lats.max() + lat_res/2
        ) * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True
    )
    return np.flipud(mask)

# ==========================================================
# 4. WORKER: PERIOD MEAN + ANOMALY
# ==========================================================
def process_one_period(scen, folder, pname, years):
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(folder, f"{y}-*-*_INM_CM5_0.nc")))
    if not files:
        return None

    total, count = None, 0
    lats = lons = None

    for f in sorted(files):
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]
                if data.ndim == 3:
                    data = data[0]
                if np.nanmean(data) > 200:
                    data -= 273.15

                if total is None:
                    total = np.zeros_like(data)
                total += data
                count += 1

                if lats is None:
                    lats = nc.variables["lat"][:]
                    lons = nc.variables["lon"][:]

        except:
            continue

    if count == 0:
        return None

    mean = total / count
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]

    return scen, pname, mean, lats, lons

# ==========================================================
# 5. RUN JOBLIB
# ==========================================================
print(f"--- Step 2: Running Joblib ({N_JOBS} cores) ---")

tasks = []
for scen, folder in scenarios.items():
    for pname, yrs in periods.items():
        tasks.append((scen, folder, pname, yrs))

results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(process_one_period)(scen, folder, pname, yrs)
    for scen, folder, pname, yrs in tasks
)

# ==========================================================
# 6. POST-PROCESS
# ==========================================================
final = {}
mask = None

for r in results:
    if r is None:
        continue

    scen, pname, mean, lats, lons = r

    if mask is None:
        base_rg = da_baseline.interp(lat=lats, lon=lons)
        mask = create_india_mask(lats, lons)
    else:
        base_rg = da_baseline.interp(lat=lats, lon=lons)

    anom = mean - base_rg.values
    anom = np.where(mask, anom, np.nan)

    a = anom[~np.isnan(anom)]

    stats = {
        "mean": np.mean(a),
        "median": np.median(a),
        "std": np.std(a),
        "skew": skew(a),
        "p05": np.percentile(a,5),
        "p95": np.percentile(a,95),
        "p01": np.percentile(a,1),
        "p99": np.percentile(a,99),
        "area>1": np.mean(a>1)*100,
        "area>2": np.mean(a>2)*100,
        "area>3": np.mean(a>3)*100,
        "values": a
    }

    final.setdefault(scen, {})[pname] = stats

# ==========================================================
# 7. HISTOGRAMS
# ==========================================================
for pname in periods:
    plt.figure(figsize=(8,6))
    for scen in scenarios:
        plt.hist(final[scen][pname]["values"], bins=80, alpha=0.4, density=True, label=scen)
    plt.title(f"QDM Anomaly Distribution — {pname}")
    plt.xlabel("Anomaly (°C)")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# ==========================================================
# 8. ANNUAL TRENDS
# ==========================================================
def get_annual_mean_qdm(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.nc")))
    yearly = {}

    for f in files:
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]
                if data.ndim == 3:
                    data = data[0]
                if np.nanmean(data) > 200:
                    data -= 273.15

                year = int(os.path.basename(f).split("-")[0])
                yearly.setdefault(year, []).append(data)
        except:
            continue

    years = sorted(yearly)
    series = [np.nanmean(np.stack(yearly[y])) for y in years]
    return np.array(years), np.array(series)

plt.figure(figsize=(12,6))
for scen, folder in scenarios.items():
    y, s = get_annual_mean_qdm(folder)
    coef = np.polyfit(y, s, 1)
    plt.plot(y, s, label=scen)
    plt.plot(y, coef[0]*y + coef[1], "--")
    print(f"{scen} trend: {coef[0]*10:.2f} °C/decade")

plt.title("India-Mean Temperature (QDM)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ==========================================================
# 9. TABLES
# ==========================================================
for scen in final:
    print(f"\n=== {scen} ===")
    for pname, s in final[scen].items():
        print(f"\n{pname}")
        for k,v in s.items():
            if k!="values":
                print(f"  {k:10s}: {v:.2f}")


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"

# --- MSWX-HR baseline ---
mswx_hr_dir = "<SET YOUR PATH HERE>"

model_name = "INM_CM5_0_DDPM_MSWX_HR_SIMPLE"
N_CORES = 80

# --- DDPM OUTPUT DIRECTORIES ---
scenarios = {
    "SSP 245": os.path.join(base_dir, "DDPM_ssp_245_INM_CM5_0"),
    "SSP 370": os.path.join(base_dir, "DDPM_ssp_370_INM_CM5_0"),
    "SSP 585": os.path.join(base_dir, "DDPM_ssp_585_INM_CM5_0"),
}

periods = {
    "Near-term\n(2021–2040)": range(2021, 2041),
    "Mid-term\n(2041–2060)": range(2041, 2061),
    "Far-term\n(2081–2100)": range(2081, 2101),
}

# ==========================================================
# 2. MSWX-HR BASELINE (1995–2014)
# ==========================================================
print("--- Step 1: Computing MSWX-HR baseline (1995–2014) ---")

def load_mswx_year(year):
    files = sorted(glob.glob(os.path.join(mswx_hr_dir, f"{year}*.nc")))
    if not files:
        return None

    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]

    # enforce south → north
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])

    return da.mean("time")

baseline_years = []
for y in range(1995, 2015):
    da_y = load_mswx_year(y)
    if da_y is not None:
        baseline_years.append(da_y)

da_mswx_baseline = xr.concat(baseline_years, dim="year").mean("year")

if da_mswx_baseline.lat[0] > da_mswx_baseline.lat[-1]:
    da_mswx_baseline = da_mswx_baseline.reindex(
        lat=da_mswx_baseline.lat[::-1]
    )

print("  MSWX-HR baseline ready:", da_mswx_baseline.shape)

# ==========================================================
# 3. DDPM FAST PROCESSING
# ==========================================================
def process_batch(files):
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]
                if data.ndim == 3:
                    data = data[0]
                data = np.asarray(data, dtype=np.float64)
                if s is None:
                    s = np.zeros_like(data)
                s += data
                c += 1
        except:
            continue
    return s, c

def get_period_mean_ddpm(folder, years):
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(folder, f"{y}-*-*.nc")))
    if not files:
        return None, None, None

    chunks = np.array_split(sorted(files), N_CORES)
    results = Parallel(n_jobs=N_CORES)(
        delayed(process_batch)(c) for c in chunks
    )

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0:
        return None, None, None

    with xr.open_dataset(files[0]) as ds:
        lats = ds.lat.values
        lons = ds.lon.values

    mean = total / count

    # enforce south → north
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]

    return mean, lats, lons

# ==========================================================
# 4. INDIA MASK
# ==========================================================
def create_india_mask(lats, lons, shp_path):
    gdf = gpd.read_file(shp_path)

    if lats[0] > lats[-1]:
        lats = lats[::-1]

    lon_res = abs(lons[1] - lons[0])
    lat_res = abs(lats[1] - lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res / 2,
            lats.max() + lat_res / 2
        )
        * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True,
        all_touched=True,
    )
    return np.flipud(mask)

# ==========================================================
# 5. PROCESS DATA (DDPM − MSWX-HR)
# ==========================================================
data_store = {s: {} for s in scenarios}
baseline_interp = None
india_mask = None
all_anoms = []

for scen, path in scenarios.items():
    for pname, yrs in periods.items():
        fut, lats, lons = get_period_mean_ddpm(path, yrs)
        if fut is None:
            continue

        if np.nanmean(fut) > 200:
            fut -= 273.15

        if baseline_interp is None:
            baseline_interp = da_mswx_baseline.interp(
                lat=lats, lon=lons, method="linear"
            ).values
            india_mask = create_india_mask(lats, lons, shp_path)

        anom = fut - baseline_interp
        anom = np.where(india_mask, anom, np.nan)
        mean = np.where(india_mask, fut, np.nan)

        data_store[scen][pname] = (anom, mean, {"lat": lats, "lon": lons})
        all_anoms.append(anom[~np.isnan(anom)])

# ==========================================================
# 6. CORRECT & ROBUST ANOMALY RANGE (FIXED)
# ==========================================================
full = np.concatenate(all_anoms)

# Robust absolute anomaly (India-wide, all SSPs)
abs_p99 = np.percentile(np.abs(full), 99)

# Round to nearest 0.5°C
amax = np.ceil(abs_p99 * 2) / 2
vmin, vmax = -amax, amax

print(f"Final anomaly range (symmetric): {vmin} → {vmax} °C")

# Divergent, zero-centered — ALWAYS correct for DDPM
anom_norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
anom_cmap = plt.cm.RdBu_r

# Mean cmap unchanged
mean_cmap = plt.cm.Spectral_r

# ==========================================================
# 6A. INDIA-WIDE DDPM STATISTICS (vs MSWX-HR)
# ==========================================================
print("\n" + "=" * 100)
print("INDIA-WIDE TEMPERATURE STATISTICS (DDPM relative to MSWX-HR baseline)")
print("=" * 100)

def compute_ddpm_stats(anom, mean):
    vals = anom[~np.isnan(anom)]
    mean_vals = mean[~np.isnan(mean)]

    return {
        "mean_temp": np.mean(mean_vals),
        "mean_anom": np.mean(vals),
        "median_anom": np.median(vals),
        "std_anom": np.std(vals),
        "p01": np.percentile(vals, 1),
        "p05": np.percentile(vals, 5),
        "p95": np.percentile(vals, 95),
        "p99": np.percentile(vals, 99),
        "min": np.min(vals),
        "max": np.max(vals),
        "area_gt_1": np.sum(vals > 1.0) / len(vals) * 100,
        "area_gt_2": np.sum(vals > 2.0) / len(vals) * 100,
        "area_gt_3": np.sum(vals > 3.0) / len(vals) * 100,
        "area_lt_m1": np.sum(vals < -1.0) / len(vals) * 100,
    }

for scen in scenarios:
    print(f"\n### {scen}")
    for pname in periods:
        anom, mean, _ = data_store[scen][pname]
        s = compute_ddpm_stats(anom, mean)

        print(f"\n{pname}")
        print(f"  Mean temperature        : {s['mean_temp']:.2f} °C")
        print(f"  Mean anomaly            : {s['mean_anom']:.2f} °C")
        print(f"  Median anomaly          : {s['median_anom']:.2f} °C")
        print(f"  Spatial std (anomaly)   : {s['std_anom']:.2f} °C")
        print(f"  1–99 % anomaly range    : [{s['p01']:.2f}, {s['p99']:.2f}] °C")
        print(f"  5–95 % anomaly range    : [{s['p05']:.2f}, {s['p95']:.2f}] °C")
        print(f"  Min / Max anomaly       : [{s['min']:.2f}, {s['max']:.2f}] °C")
        print(f"  Area > +1°C             : {s['area_gt_1']:.1f} %")
        print(f"  Area > +2°C             : {s['area_gt_2']:.1f} %")
        print(f"  Area > +3°C             : {s['area_gt_3']:.1f} %")
        print(f"  Area < −1°C             : {s['area_lt_m1']:.1f} %")


# ==========================================================
# 7. PLOTTING (18 PANELS — CLEAN)
# ==========================================================
fig, axes = plt.subplots(
    3, 6, figsize=(30, 14),
    subplot_kw={"projection": ccrs.PlateCarree()}
)
gdf = gpd.read_file(shp_path)

for r, scen in enumerate(scenarios):
    for p, pname in enumerate(periods):

        anom, mean, coords = data_store[scen][pname]
        axA = axes[r, p * 2]
        axM = axes[r, p * 2 + 1]

        for ax in (axA, axM):
            ax.set_extent([66, 100, 6, 38])
            ax.add_geometries(
                gdf.geometry, ccrs.PlateCarree(),
                facecolor="none", edgecolor="black"
            )
            ax.coastlines(alpha=0.6)

        imA = axA.pcolormesh(
            coords["lon"], coords["lat"], anom,
            cmap=anom_cmap, norm=anom_norm
        )

        imM = axM.pcolormesh(
            coords["lon"], coords["lat"], mean,
            cmap=mean_cmap
        )

        if r == 0:
            axA.set_title(f"{pname}\nAnomaly", fontsize=16)
            axM.set_title(f"{pname}\nMean", fontsize=16)

        if p == 0:
            axA.text(
                -0.12, 0.5, scen,
                rotation=90, transform=axA.transAxes,
                fontsize=18, fontweight="bold",
                va="center", ha="center"
            )

# ==========================================================
# 8. COLORBARS
# ==========================================================
caxA = fig.add_axes([0.92, 0.18, 0.015, 0.65])
fig.colorbar(imA, cax=caxA, label="DDPM Temperature Anomaly (°C)")

caxM = fig.add_axes([0.15, 0.06, 0.65, 0.025])
fig.colorbar(imM, cax=caxM, orientation="horizontal",
             label="Mean Temperature (°C)")

plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker 
from matplotlib.offsetbox import AnchoredText
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"
mswx_hr_dir = "<SET YOUR PATH HERE>"

N_CORES = 96

scenarios = {
    "SSP 245": os.path.join(base_dir, "DDPM_ssp_245_INM_CM5_0"),
    "SSP 370": os.path.join(base_dir, "DDPM_ssp_370_INM_CM5_0"),
    "SSP 585": os.path.join(base_dir, "DDPM_ssp_585_INM_CM5_0"),
}

periods = {
    "Near-term\n(2021–2040)": range(2021, 2041),
    "Mid-term\n(2041–2060)": range(2041, 2061),
    "Far-term\n(2081–2100)": range(2081, 2101),
}

# ==========================================================
# 2. HELPER FUNCTIONS
# ==========================================================
def process_batch_sum(files, var_name="tas"):
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f, 'r') as nc:
                if var_name in nc.variables: v = var_name
                elif 'air_temperature' in nc.variables: v = 'air_temperature'
                else: continue
                
                data = nc.variables[v][:]
                if data.ndim == 3: data = data[0]
                data = np.asarray(data, dtype=np.float32)
                
                if s is None: s = np.zeros_like(data)
                s += data
                c += 1
        except Exception: continue
    return s, c

def get_mean_field_parallel(folder, years, var_name="tas"):
    files = []
    for y in years:
        found = sorted(glob.glob(os.path.join(folder, f"*{y}*.nc")))
        files.extend(found)
    if not files: return None, None, None

    chunks = np.array_split(sorted(files), N_CORES)
    results = Parallel(n_jobs=N_CORES)(delayed(process_batch_sum)(c, var_name) for c in chunks)

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0: return None, None, None

    with xr.open_dataset(files[0]) as ds:
        lats, lons = ds.lat.values, ds.lon.values

    mean = total / count
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]
    if np.nanmean(mean) > 200: mean -= 273.15
        
    return mean, lats, lons

def create_india_mask(lats, lons, shp_path):
    gdf = gpd.read_file(shp_path)
    lon_res, lat_res = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])
    transform = rasterio.transform.Affine.translation(
        lons.min() - lon_res/2, lats.max() + lat_res/2
    ) * rasterio.transform.Affine.scale(lon_res, -lat_res)
    mask = rasterio.features.geometry_mask(
        gdf.geometry, out_shape=(len(lats), len(lons)),
        transform=transform, invert=True, all_touched=True
    )
    return np.flipud(mask)

def add_histogram_inset(ax, data, color, bins=30):
    # Position: [x, y, width, height]
    ax_ins = ax.inset_axes([0.60, 0.05, 0.38, 0.25]) 
    ax_ins.patch.set_alpha(0.6)
    
    data_clean = data[~np.isnan(data)]
    ax_ins.hist(data_clean, bins=bins, color=color, alpha=0.9, density=True, 
                histtype='stepfilled', edgecolor='black', linewidth=0.8)
    
    ax_ins.tick_params(axis='x', labelsize=11, labelcolor='black', bottom=True, pad=1)
    ax_ins.tick_params(axis='y', left=False, labelleft=False) 
    ax_ins.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    
    ax_ins.spines['top'].set_visible(False)
    ax_ins.spines['right'].set_visible(False)
    ax_ins.spines['left'].set_visible(False)

# ==========================================================
# 3. DATA PROCESSING
# ==========================================================
print("--- Computing MSWX Baseline ---")
mswx_mean, m_lats, m_lons = get_mean_field_parallel(
    mswx_hr_dir, range(1995, 2015), var_name="air_temperature"
)
india_mask = create_india_mask(m_lats, m_lons, shp_path)
data_store = {s: {} for s in scenarios}

all_anoms = []
all_means = [] 

print("--- Processing DDPM Scenarios ---")
for scen, path in scenarios.items():
    for pname, yrs in periods.items():
        fut, f_lats, f_lons = get_mean_field_parallel(path, yrs, var_name="tas")
        if fut is None: continue

        anom_full = fut - mswx_mean
        anom_masked = np.where(india_mask, anom_full, np.nan)
        mean_masked = np.where(india_mask, fut, np.nan)
        
        data_store[scen][pname] = {
            "anom": anom_masked, "mean": mean_masked,
            "lats": f_lats, "lons": f_lons
        }
        all_anoms.append(anom_masked[~np.isnan(anom_masked)])
        all_means.append(mean_masked[~np.isnan(mean_masked)])

# ==========================================================
# 4. COLORMAP LOGIC (STEP = 0.5)
# ==========================================================
# Anomaly Range
full_anom_array = np.concatenate(all_anoms)
vmin_anom = int(np.floor(np.percentile(full_anom_array, 1)))
vmax_anom = int(np.ceil(np.percentile(full_anom_array, 99)))

if vmin_anom > 0: vmin_anom = 0
if vmax_anom < 0: vmax_anom = 0

anom_step = 0.5
anom_bounds = np.arange(vmin_anom, vmax_anom + anom_step, anom_step)
print(f"Discrete Anomaly Bounds (Step 0.5): {anom_bounds}")

# Sampling Logic
max_abs = max(abs(vmin_anom), abs(vmax_anom))
bin_centers = anom_bounds[:-1] + (anom_step / 2)
normalized_centers = bin_centers / max_abs
cmap_indices = (normalized_centers / 2) + 0.5
base_cmap = plt.cm.PuOr_r
sampled_colors = base_cmap(cmap_indices)

anom_cmap_discrete = mcolors.ListedColormap(sampled_colors)
anom_norm_discrete = mcolors.BoundaryNorm(anom_bounds, len(sampled_colors))

# Mean Range
full_mean_array = np.concatenate(all_means)
vmin_mean = np.floor(np.percentile(full_mean_array, 1)) 
vmax_mean = np.ceil(np.percentile(full_mean_array, 99)) 
mean_cmap = plt.cm.RdBu_r

print(f"Mean Range: {vmin_mean} to {vmax_mean} °C")

# ==========================================================
# 5. PLOTTING (6 ROWS x 3 COLS)
# ==========================================================
print("--- Generating Plot ---")
fig, axes = plt.subplots(6, 3, figsize=(22, 36), subplot_kw={"projection": ccrs.PlateCarree()})
gdf = gpd.read_file(shp_path)

for r, scen in enumerate(scenarios):
    row_anom = r * 2
    row_mean = r * 2 + 1
    
    for p, pname in enumerate(periods):
        if pname not in data_store[scen]: continue
        d = data_store[scen][pname]
        lons, lats = d['lons'], d['lats']
        
        axA = axes[row_anom, p]
        axM = axes[row_mean, p]
        
        for ax in (axA, axM):
            ax.set_extent([66, 100, 6, 38])
            ax.add_geometries(gdf.geometry, ccrs.PlateCarree(), facecolor="none", edgecolor="black", linewidth=1.0)
            
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                              linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (p == 0)
            gl.bottom_labels = (row_mean == 5) and (ax == axM)
            gl.xformatter = LONGITUDE_FORMATTER
            gl.yformatter = LATITUDE_FORMATTER
            gl.xlabel_style = {'size': 14, 'color': 'black'}
            gl.ylabel_style = {'size': 14, 'color': 'black'}

        # --- PLOT ANOMALY ---
        imA = axA.pcolormesh(lons, lats, d['anom'], cmap=anom_cmap_discrete, norm=anom_norm_discrete)
        
        # Stats
        mu_anom = np.nanmean(d['anom'])
        p95_anom = np.nanpercentile(d['anom'], 95)
        stats_text_A = fr"$\mu = {mu_anom:+.1f}$" + "\n" + fr"$P_{{95}} = {p95_anom:+.1f}$"
        at_A = AnchoredText(stats_text_A, prop=dict(size=14, fontweight='bold'), frameon=True, loc='upper right')
        at_A.patch.set_boxstyle("round,pad=0.3,rounding_size=0.2")
        axA.add_artist(at_A)
        
        add_histogram_inset(axA, d['anom'], color='gray')

        # --- FIGURE LABEL (a, b, c...) ---
        # Logic: Anomaly Row Index is (row_anom * 3) + p
        label_index_A = (row_anom * 3) + p
        label_char_A = chr(97 + label_index_A) # 97 is ASCII for 'a'
        axA.text(0.02, 0.92, f"({label_char_A})", transform=axA.transAxes, 
                 fontsize=24, fontweight='bold', va='top', ha='left')

        # --- PLOT MEAN ---
        imM = axM.pcolormesh(lons, lats, d['mean'], cmap=mean_cmap, vmin=vmin_mean, vmax=vmax_mean)
        
        # Stats
        mu_tas = np.nanmean(d['mean'])
        p95_tas = np.nanpercentile(d['mean'], 95)
        stats_text_M = fr"$\mu = {mu_tas:.1f}$" + "\n" + fr"$P_{{95}} = {p95_tas:.1f}$"
        at_M = AnchoredText(stats_text_M, prop=dict(size=14, fontweight='bold'), frameon=True, loc='upper right')
        at_M.patch.set_boxstyle("round,pad=0.3,rounding_size=0.2")
        axM.add_artist(at_M)
        
        add_histogram_inset(axM, d['mean'], color='darkred')

        # --- FIGURE LABEL (d, e, f...) ---
        # Logic: Mean Row Index is (row_mean * 3) + p
        label_index_M = (row_mean * 3) + p
        label_char_M = chr(97 + label_index_M)
        axM.text(0.02, 0.92, f"({label_char_M})", transform=axM.transAxes, 
                 fontsize=24, fontweight='bold', va='top', ha='left')

        # Titles & Labels
        if row_anom == 0:
            axA.set_title(pname, fontsize=24, fontweight='bold', pad=20)
            
        if p == 0:
            axA.text(-0.25, 0.5, f"{scen}\nAnomaly", rotation=90, transform=axA.transAxes, 
                     fontsize=20, fontweight="bold", va="center", ha="center")
            axM.text(-0.25, 0.5, f"{scen}\nMean", rotation=90, transform=axM.transAxes, 
                     fontsize=20, fontweight="bold", va="center", ha="center")

# Colorbars
caxA = fig.add_axes([0.15, 0.08, 0.7, 0.015])
cbA = fig.colorbar(imA, cax=caxA, orientation="horizontal", label="DDPM Temperature Anomaly (°C)", extend='both', ticks=mticker.MultipleLocator(1))
cbA.ax.tick_params(labelsize=16)
cbA.set_label("DDPM Temperature Anomaly (°C)", fontsize=18, fontweight='bold')

caxM = fig.add_axes([0.15, 0.04, 0.7, 0.015])
cbM = fig.colorbar(imM, cax=caxM, orientation="horizontal", label="Mean Temperature (°C)", extend='both')
cbM.ax.tick_params(labelsize=16)
cbM.set_label("Mean Temperature (°C)", fontsize=18, fontweight='bold')

plt.subplots_adjust(wspace=0.1, hspace=0.15, bottom=0.12)
plt.savefig("DDPM_Final_Labeled.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker 
from matplotlib.offsetbox import AnchoredText
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset
import warnings

warnings.filterwarnings("ignore")

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"
mswx_hr_dir = "<SET YOUR PATH HERE>"

N_CORES = 96

scenarios = {
    "SSP 245": os.path.join(base_dir, "DDPM_ssp_245_INM_CM5_0"),
    "SSP 370": os.path.join(base_dir, "DDPM_ssp_370_INM_CM5_0"),
    "SSP 585": os.path.join(base_dir, "DDPM_ssp_585_INM_CM5_0"),
}

periods = {
    "Near-term\n(2021–2040)": range(2021, 2041),
    "Mid-term\n(2041–2060)": range(2041, 2061),
    "Far-term\n(2081–2100)": range(2081, 2101),
}

# ==========================================================
# 2. HELPER FUNCTIONS
# ==========================================================
def process_batch_sum(files, var_name="tas"):
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f, 'r') as nc:
                if var_name in nc.variables: v = var_name
                elif 'air_temperature' in nc.variables: v = 'air_temperature'
                else: continue
                data = nc.variables[v][:]
                if data.ndim == 3: data = data[0]
                data = np.asarray(data, dtype=np.float32)
                if s is None: s = np.zeros_like(data)
                s += data
                c += 1
        except Exception:
            continue
    return s, c

def get_mean_field_parallel(folder, years, var_name="tas"):
    files = []
    for y in years:
        files.extend(sorted(glob.glob(os.path.join(folder, f"*{y}*.nc"))))
    if not files: return None, None, None

    chunks = np.array_split(sorted(files), N_CORES)
    results = Parallel(n_jobs=N_CORES)(
        delayed(process_batch_sum)(c, var_name) for c in chunks
    )

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0: return None, None, None

    with xr.open_dataset(files[0]) as ds:
        lats, lons = ds.lat.values, ds.lon.values

    mean = total / count
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]
    if np.nanmean(mean) > 200:
        mean -= 273.15
    return mean, lats, lons

def create_india_mask(lats, lons, shp_path):
    gdf = gpd.read_file(shp_path)
    lon_res, lat_res = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])
    transform = rasterio.transform.Affine.translation(
        lons.min() - lon_res/2, lats.max() + lat_res/2
    ) * rasterio.transform.Affine.scale(lon_res, -lat_res)
    mask = rasterio.features.geometry_mask(
        gdf.geometry, out_shape=(len(lats), len(lons)),
        transform=transform, invert=True, all_touched=True
    )
    return np.flipud(mask)

def add_histogram_inset(ax, data, color, bins=30):
    ax_ins = ax.inset_axes([0.60, 0.05, 0.38, 0.25])
    ax_ins.patch.set_alpha(0.6)
    data_clean = data[~np.isnan(data)]
    ax_ins.hist(data_clean, bins=bins, color=color, alpha=0.9, density=True,
                histtype='stepfilled', edgecolor='black', linewidth=0.8)
    ax_ins.tick_params(axis='x', labelsize=11, labelcolor='black', bottom=True, pad=1)
    ax_ins.tick_params(axis='y', left=False, labelleft=False)
    ax_ins.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    ax_ins.spines['top'].set_visible(False)
    ax_ins.spines['right'].set_visible(False)
    ax_ins.spines['left'].set_visible(False)

# ==========================================================
# 3. DATA PROCESSING
# ==========================================================
print("--- Computing MSWX Baseline ---")
mswx_mean, m_lats, m_lons = get_mean_field_parallel(
    mswx_hr_dir, range(1995, 2015), var_name="air_temperature"
)
india_mask = create_india_mask(m_lats, m_lons, shp_path)

data_store = {s: {} for s in scenarios}
all_anoms, all_means = [], []

print("--- Processing DDPM Scenarios ---")
for scen, path in scenarios.items():
    for pname, yrs in periods.items():
        fut, f_lats, f_lons = get_mean_field_parallel(path, yrs, var_name="tas")
        if fut is None: continue
        anom_full = fut - mswx_mean
        anom_masked = np.where(india_mask, anom_full, np.nan)
        mean_masked = np.where(india_mask, fut, np.nan)
        data_store[scen][pname] = {
            "anom": anom_masked, "mean": mean_masked,
            "lats": f_lats, "lons": f_lons
        }
        all_anoms.append(anom_masked[~np.isnan(anom_masked)])
        all_means.append(mean_masked[~np.isnan(mean_masked)])

# ==========================================================
# 4. COLORMAPS
# ==========================================================
full_anom_array = np.concatenate(all_anoms)
vmin_anom = int(np.floor(np.percentile(full_anom_array, 1)))
vmax_anom = int(np.ceil(np.percentile(full_anom_array, 99)))
if vmin_anom > 0: vmin_anom = 0
if vmax_anom < 0: vmax_anom = 0

anom_step = 0.5
anom_bounds = np.arange(vmin_anom, vmax_anom + anom_step, anom_step)

max_abs = max(abs(vmin_anom), abs(vmax_anom))
bin_centers = anom_bounds[:-1] + (anom_step / 2)
normalized_centers = bin_centers / max_abs
cmap_indices = (normalized_centers / 2) + 0.5
base_cmap = plt.cm.PuOr_r
sampled_colors = base_cmap(cmap_indices)

anom_cmap_discrete = mcolors.ListedColormap(sampled_colors)
anom_norm_discrete = mcolors.BoundaryNorm(anom_bounds, len(sampled_colors))

full_mean_array = np.concatenate(all_means)
vmin_mean = np.floor(np.percentile(full_mean_array, 1))
vmax_mean = np.ceil(np.percentile(full_mean_array, 99))
mean_cmap = plt.cm.RdBu_r

# ==========================================================
# 5. PLOTTING
# ==========================================================
fig, axes = plt.subplots(6, 3, figsize=(22, 36),
                         subplot_kw={"projection": ccrs.PlateCarree()})
gdf = gpd.read_file(shp_path)

for r, scen in enumerate(scenarios):
    row_anom = r * 2
    row_mean = r * 2 + 1

    for p, pname in enumerate(periods):
        if pname not in data_store[scen]: continue
        d = data_store[scen][pname]
        lons, lats = d['lons'], d['lats']

        axA = axes[row_anom, p]
        axM = axes[row_mean, p]

        for ax in (axA, axM):
            ax.set_extent([66, 100, 6, 38])
            ax.add_geometries(gdf.geometry, ccrs.PlateCarree(),
                              facecolor="none", edgecolor="black", linewidth=1.0)

            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                              linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.left_labels = (p == 0)
            gl.bottom_labels = (row_mean == 5) and (ax == axM)
            gl.xformatter = LONGITUDE_FORMATTER
            gl.yformatter = LATITUDE_FORMATTER
            gl.xlabel_style = {'size': 14}
            gl.ylabel_style = {'size': 14}

        # ---------- ANOM ----------
        imA = axA.pcolormesh(lons, lats, d['anom'],
                             cmap=anom_cmap_discrete, norm=anom_norm_discrete)

        mu_anom = np.nanmean(d['anom'])
        p95_anom = np.nanpercentile(d['anom'], 95)

        stats_text_A = (
            fr"$\mu_{{\mathrm{{anom}}}} = {mu_anom:+.1f}$" + "\n" +
            fr"$P_{{95,\mathrm{{anom}}}} = {p95_anom:+.1f}$"
        )

        at_A = AnchoredText(stats_text_A,
                            prop=dict(size=16, fontweight='bold'),
                            frameon=True, loc='upper right')
        at_A.patch.set_boxstyle("round,pad=0.3,rounding_size=0.2")
        axA.add_artist(at_A)

        add_histogram_inset(axA, d['anom'], color='gray')

        # ---------- MEAN ----------
        imM = axM.pcolormesh(lons, lats, d['mean'],
                             cmap=mean_cmap, vmin=vmin_mean, vmax=vmax_mean)

        mu_tas = np.nanmean(d['mean'])
        p95_tas = np.nanpercentile(d['mean'], 95)

        stats_text_M = (
            fr"$\mu_{{\mathrm{{TAS}}}} = {mu_tas:.1f}$" + "\n" +
            fr"$P_{{95,\mathrm{{TAS}}}} = {p95_tas:.1f}$"
        )

        at_M = AnchoredText(stats_text_M,
                            prop=dict(size=16, fontweight='bold'),
                            frameon=True, loc='upper right')
        at_M.patch.set_boxstyle("round,pad=0.3,rounding_size=0.2")
        axM.add_artist(at_M)

        add_histogram_inset(axM, d['mean'], color='darkred')

        # ----- COLUMN TITLES -----
        if row_anom == 0:
            axA.set_title(pname, fontsize=24, fontweight='bold', pad=20)

        # ----- ROW LABELS -----
        if p == 0:
            axA.text(-0.25, 0.5, f"{scen}\nAnomaly",
                     rotation=90, transform=axA.transAxes,
                     fontsize=20, fontweight="bold",
                     va="center", ha="center")

            axM.text(-0.25, 0.5, f"{scen}\nMean",
                     rotation=90, transform=axM.transAxes,
                     fontsize=20, fontweight="bold",
                     va="center", ha="center")

# ---------- COLORBARS ----------
caxA = fig.add_axes([0.15, 0.08, 0.7, 0.015])
cbA = fig.colorbar(imA, cax=caxA, orientation="horizontal",
                   label="DDPM Temperature Anomaly (°C)",
                   extend='both', ticks=mticker.MultipleLocator(1))
cbA.ax.tick_params(labelsize=16)
cbA.set_label("DDPM Temperature Anomaly (°C)", fontsize=18, fontweight='bold')

caxM = fig.add_axes([0.15, 0.04, 0.7, 0.015])
cbM = fig.colorbar(imM, cax=caxM, orientation="horizontal",
                   label="Mean Temperature (°C)", extend='both')
cbM.ax.tick_params(labelsize=16)
cbM.set_label("Mean Temperature (°C)", fontsize=18, fontweight='bold')

plt.subplots_adjust(wspace=0.1, hspace=0.15, bottom=0.12)
plt.savefig("DDPM_Final_Labeled_2.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.stats import skew
from joblib import Parallel, delayed
from netCDF4 import Dataset

# ==========================================================
# 0. SAFETY
# ==========================================================
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

try:
    import rasterio.features
    import rasterio.transform
except:
    raise ImportError("Install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"

mswx_hr_dir = "<SET YOUR PATH HERE>"

scenarios = {
    "SSP 245": os.path.join(base_dir, "DDPM_ssp_245_INM_CM5_0"),
    "SSP 370": os.path.join(base_dir, "DDPM_ssp_370_INM_CM5_0"),
    "SSP 585": os.path.join(base_dir, "DDPM_ssp_585_INM_CM5_0"),
}

periods = {
    "Near-term (2021–2040)": range(2021, 2041),
    "Mid-term (2041–2060)": range(2041, 2061),
    "Far-term (2081–2100)": range(2081, 2101),
}

N_JOBS = 100
VAR_NAME = "tas"

# ==========================================================
# 2. MSWX-HR BASELINE (1995–2014)
# ==========================================================
print("\n--- Step 1: Computing MSWX-HR baseline (1995–2014) ---")

def load_mswx_year(year):
    files = sorted(glob.glob(os.path.join(mswx_hr_dir, f"{year}*.nc")))
    if not files:
        return None
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])
    return da.mean("time")

baseline_years = [load_mswx_year(y) for y in range(1995, 2015)]
baseline_years = [b for b in baseline_years if b is not None]
da_baseline = xr.concat(baseline_years, dim="year").mean("year")
print("Baseline shape:", da_baseline.shape)

# ==========================================================
# 3. INDIA MASK
# ==========================================================
def create_india_mask(lats, lons):
    gdf = gpd.read_file(shp_path)

    lon_res = abs(lons[1] - lons[0])
    lat_res = abs(lats[1] - lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res / 2,
            lats.max() + lat_res / 2
        )
        * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True,
        all_touched=True
    )
    return np.flipud(mask)

# ==========================================================
# 4. NUMPY STATS
# ==========================================================
def compute_stats_numpy(a):
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return None
    return {
        "mean": np.mean(a),
        "median": np.median(a),
        "std": np.std(a),
        "skew": skew(a),
        "p05": np.percentile(a, 5),
        "p95": np.percentile(a, 95),
        "p01": np.percentile(a, 1),
        "p99": np.percentile(a, 99),
        "area_>1": np.mean(a > 1) * 100,
        "area_>2": np.mean(a > 2) * 100,
        "area_>3": np.mean(a > 3) * 100,
        "values": a
    }

# ==========================================================
# 5. FAST DDPM PERIOD MEAN (JOBLIB)
# ==========================================================
def process_batch(files):
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f) as nc:
                arr = nc.variables[VAR_NAME][:]
                if arr.ndim == 3:
                    arr = arr[0]
                arr = arr.astype(np.float64)
                s = arr if s is None else s + arr
                c += 1
        except:
            continue
    return s, c

def get_period_mean_ddpm(folder, years):
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(folder, f"{y}-*-*.nc")))
    if not files:
        return None, None, None

    chunks = np.array_split(sorted(files), N_JOBS)
    results = Parallel(n_jobs=N_JOBS)(
        delayed(process_batch)(c) for c in chunks
    )

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0:
        return None, None, None

    with xr.open_dataset(files[0]) as ds:
        lats = ds.lat.values
        lons = ds.lon.values

    mean = total / count
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]

    return mean, lats, lons

# ==========================================================
# 6. MAIN LOOP
# ==========================================================
print("\n--- Step 2: Computing DDPM anomalies & statistics ---")

final_stats = {}

for scen, folder in scenarios.items():
    print(f"\n>>> {scen}")
    final_stats[scen] = {}

    for pname, yrs in periods.items():
        fut, lats, lons = get_period_mean_ddpm(folder, yrs)
        if fut is None:
            continue

        if fut.mean() > 200:
            fut -= 273.15

        baseline_rg = da_baseline.interp(
            lat=lats, lon=lons, method="linear"
        ).values

        india_mask = create_india_mask(lats, lons)

        anom = np.where(india_mask, fut - baseline_rg, np.nan)

        final_stats[scen][pname] = compute_stats_numpy(anom)

# ==========================================================
# 7. HISTOGRAMS
# ==========================================================
for pname in periods:
    plt.figure(figsize=(8,6))
    for scen in scenarios:
        vals = final_stats[scen][pname]["values"]
        plt.hist(vals, bins=80, density=True, alpha=0.4, label=scen)

    plt.title(f"DDPM Temperature Anomaly Distribution\n{pname}")
    plt.xlabel("Anomaly (°C)")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# ==========================================================
# 8. PRINT TABLES
# ==========================================================
print("\n================= INDIA-WIDE DDPM STATISTICS =================")
for scen in final_stats:
    print(f"\n--- {scen} ---")
    for pname, s in final_stats[scen].items():
        print(f"\n{pname}")
        for k, v in s.items():
            if k != "values":
                print(f"  {k:10s}: {v:.2f}")

print("\n✅ DDPM HR statistical analysis complete.")
# ==========================================================
# 9. INDIA-MEAN TIME SERIES & TRENDS (DDPM HR)
# ==========================================================
print("\n--- Step 3: Computing DDPM India-mean time series & trends ---")

from netCDF4 import Dataset

def get_annual_mean_ddpm(folder):
    """Robust India-mean annual series from DDPM files (skips corrupted files)"""

    files = sorted(glob.glob(os.path.join(folder, "*.nc")))
    if not files:
        return None, None

    yearly = {}

    for f in files:
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]

                if data.ndim == 3:
                    data = data[0]

                if np.nanmean(data) > 200:
                    data = data - 273.15

                year = int(os.path.basename(f).split("-")[0])

                if year not in yearly:
                    yearly[year] = []
                yearly[year].append(data)

        except Exception as e:
            print(f"⚠️ Skipping corrupted file: {os.path.basename(f)}")
            continue

    years = sorted(yearly.keys())
    means = []

    for y in years:
        stack = np.stack(yearly[y])
        means.append(np.nanmean(stack))

    return np.array(years), np.array(means)


plt.figure(figsize=(12, 6))

for scen, folder in scenarios.items():
    years, series = get_annual_mean_ddpm(folder)

    # Linear trend
    coef = np.polyfit(years, series, 1)
    trend = coef[0] * 10  # °C / decade

    plt.plot(years, series, label=scen)
    plt.plot(years, coef[0]*years + coef[1], "--")

    print(f"{scen} trend: {trend:.2f} °C / decade")

plt.title("India-Mean Temperature (DDPM HR)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import xarray as xr
ds_qdm_245 = xr.open_dataset("<SET YOUR PATH HERE>") # example individual daily file
ds_qdm_370 = xr.open_dataset("<SET YOUR PATH HERE>") # example individual daily file
ds_qdm_585 = xr.open_dataset("<SET YOUR PATH HERE>") # example individual daily file

print("QDM SSP2-4.5", ds_qdm_245)
print("QDM SSP3-7.0", ds_qdm_370)
print("QDM SSP5-8.5", ds_qdm_585)

In [ ]:
import xarray as xr
ds = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds)

In [ ]:
%%writefile mswx_std.py
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import glob
import dask
import rioxarray
import warnings

# Optimize Dask for your 128-core machine
dask.config.set(scheduler='processes') 

# Suppress warnings
warnings.filterwarnings("ignore")

# ================= CONFIGURATION =================
# 1. PATHS
SHP_PATH = "<SET YOUR PATH HERE>"
MSWX_DIR = "<SET YOUR PATH HERE>"

# 2. SETTINGS
VAR_NAME = 'air_temperature'  # As seen in your print(ds)
START_YEAR = 1985
END_YEAR = 2014

# ================= CALCULATION FUNCTION =================
def calculate_mswx_std(folder_path, start_year, end_year, gdf):
    print(f"Starting MSWX Baseline Analysis ({start_year}-{end_year})...")
    
    # 1. Gather Files (Pattern: YYYYDDD.nc)
    files = []
    for y in range(start_year, end_year + 1):
        # Matches "1985*.nc" (catches 1985001.nc, 1985002.nc, etc.)
        found = sorted(glob.glob(os.path.join(folder_path, f"{y}*.nc")))
        files.extend(found)
        
    if not files:
        print(f"  [ERROR] No files found in {folder_path} for years {start_year}-{end_year}")
        return np.nan

    print(f"  -> Found {len(files)} daily files.")

    try:
        # 2. Open with Dask Chunks (Lazy Loading)
        # chunks={'time': 100} allows 128 CPUs to process ~3 months at a time per core
        with xr.open_mfdataset(files, combine='nested', concat_dim='time', parallel=True, chunks={'time': 100}) as ds:
            
            # --- Fix Dimensions (lon/lat -> x/y) ---
            # Using specific names from your file structure
            ds = ds.rename({'lon': 'x', 'lat': 'y'})
            ds = ds.rio.write_crs("EPSG:4326")
            
            # --- Spatial Slicing ---
            minx, miny, maxx, maxy = gdf.total_bounds
            
            # Handle Descending Latitude (49.95 -> 0.05)
            # You confirmed lat is descending in your print(ds)
            ds_box = ds.sel(x=slice(minx, maxx), y=slice(maxy, miny))

            # --- Clip to India Shapefile ---
            clipped = ds_box[VAR_NAME].rio.clip(gdf.geometry.values, gdf.crs, drop=True)

            # --- THE "ONE-PASS" ALGORITHM ---
            # We compute Count, Sum, and Sum_of_Squares simultaneously.
            # This avoids loading the massive dataset twice.
            print(f"  -> Distributing tasks to 128 CPUs...")
            
            count_lazy = clipped.count()
            sum_lazy = clipped.sum()
            sum_sq_lazy = (clipped**2).sum()
            
            # Trigger computation
            count_val, sum_val, sum_sq_val = dask.compute(count_lazy, sum_lazy, sum_sq_lazy)
            
            # --- Final Statistics ---
            N = count_val.item()
            Sum = sum_val.item()
            SumSq = sum_sq_val.item()
            
            if N == 0: 
                print("  [ERROR] No valid data points found after clipping (check shapefile overlap).")
                return np.nan
            
            mean = Sum / N
            mean_sq = SumSq / N
            
            # Variance = E[X^2] - (E[X])^2
            variance = mean_sq - (mean**2)
            
            # Handle precision edge cases
            if variance < 0: variance = 0
                
            std_val = np.sqrt(variance)
            
            return std_val

    except Exception as e:
        print(f"  [CRASH] Error: {e}")
        import traceback
        traceback.print_exc()
        return np.nan

# ================= MAIN =================
def main():
    # Load Shapefile
    print("Loading Shapefile...")
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs != "EPSG:4326": 
        gdf = gdf.to_crs("EPSG:4326")
    
    # Calculate
    std_val = calculate_mswx_std(MSWX_DIR, START_YEAR, END_YEAR, gdf)

    # Output
    print("\n" + "="*50)
    if not np.isnan(std_val):
        print(f"MSWX baseline std: {std_val:.2f}°C")
    else:
        print(f"MSWX baseline std: [CALCULATION FAILED]")
    print("="*50)

if __name__ == "__main__":
    main()

In [ ]:
!python mswx_std.py

In [ ]:
import xarray as xr
ds = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds)

In [ ]:
%%writefile ddpm_std.py
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import glob
import dask
import rioxarray
import warnings

# Optimize Dask for 128 cores
dask.config.set(scheduler='processes') 

# Suppress warnings
warnings.filterwarnings("ignore")

# ================= CONFIGURATION =================
# 1. PATHS
SHP_PATH = "<SET YOUR PATH HERE>"
DDPM_DIR = "<SET YOUR PATH HERE>"

# 2. SETTINGS
VAR_NAME = 'tas'  # As seen in your print(ds)
START_YEAR = 2081
END_YEAR = 2100

# ================= CALCULATION FUNCTION =================
def calculate_ddpm_std(folder_path, start_year, end_year, gdf):
    print(f"Starting DDPM Far-term Analysis ({start_year}-{end_year})...")
    
    # 1. Gather Files (Pattern: YYYY-MM-DD_INM_CM5_0.nc)
    files = []
    for y in range(start_year, end_year + 1):
        # Matches "2081-01-01_INM_CM5_0.nc"
        found = sorted(glob.glob(os.path.join(folder_path, f"{y}-*-*_INM_CM5_0.nc")))
        files.extend(found)
        
    if not files:
        print(f"  [ERROR] No files found in {folder_path} for years {start_year}-{end_year}")
        return np.nan

    print(f"  -> Found {len(files)} daily files.")

    try:
        # 2. Open with Dask Chunks (Lazy Loading)
        with xr.open_mfdataset(files, combine='nested', concat_dim='time', parallel=True, chunks={'time': 100}) as ds:
            
            # --- Fix Dimensions (lon/lat -> x/y) ---
            # Your print(ds) showed 'lon' and 'lat'
            rename_dict = {}
            if 'lon' in ds.dims: rename_dict['lon'] = 'x'
            if 'lat' in ds.dims: rename_dict['lat'] = 'y'
            ds = ds.rename(rename_dict)
            
            ds = ds.rio.write_crs("EPSG:4326")
            
            # --- Spatial Slicing ---
            minx, miny, maxx, maxy = gdf.total_bounds
            
            # Handle Descending Latitude (49.95 -> 0.05)
            # Your print(ds) confirmed lat is descending
            if ds.y[0] > ds.y[-1]:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(maxy, miny))
            else:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(miny, maxy))

            # --- Clip to India Shapefile ---
            clipped = ds_box[VAR_NAME].rio.clip(gdf.geometry.values, gdf.crs, drop=True)

            # --- THE "ONE-PASS" ALGORITHM ---
            print(f"  -> Distributing tasks to 128 CPUs...")
            
            # Define lazy calculations
            count_lazy = clipped.count()
            sum_lazy = clipped.sum()
            sum_sq_lazy = (clipped**2).sum()
            
            # Compute all 3 stats at once
            count_val, sum_val, sum_sq_val = dask.compute(count_lazy, sum_lazy, sum_sq_lazy)
            
            # --- Final Statistics ---
            N = count_val.item()
            Sum = sum_val.item()
            SumSq = sum_sq_val.item()
            
            if N == 0: 
                print("  [ERROR] No valid data points found after clipping.")
                return np.nan
            
            mean = Sum / N
            mean_sq = SumSq / N
            
            # Variance = E[X^2] - (E[X])^2
            variance = mean_sq - (mean**2)
            
            if variance < 0: variance = 0
                
            std_val = np.sqrt(variance)
            
            return std_val

    except Exception as e:
        print(f"  [CRASH] Error: {e}")
        return np.nan

# ================= MAIN =================
def main():
    # Load Shapefile
    print("Loading Shapefile...")
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs != "EPSG:4326": 
        gdf = gdf.to_crs("EPSG:4326")
    
    # Calculate
    std_val = calculate_ddpm_std(DDPM_DIR, START_YEAR, END_YEAR, gdf)

    # Output
    print("\n" + "="*50)
    if not np.isnan(std_val):
        print(f"DDPM absolute std: {std_val:.2f}°C")
    else:
        print(f"DDPM absolute std: [CALCULATION FAILED]")
    print("="*50)

if __name__ == "__main__":
    main()

In [ ]:
!python ddpm_std.py

In [ ]:
import xarray as xr
ds = xr.open_dataset("<SET YOUR PATH HERE>")
print(ds)

In [ ]:
%%writefile qdm_std.py
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import glob
import dask
import rioxarray
import warnings

# Optimize Dask for 128 cores
dask.config.set(scheduler='processes') 

# Suppress warnings
warnings.filterwarnings("ignore")

# ================= CONFIGURATION =================
# 1. PATHS
SHP_PATH = "<SET YOUR PATH HERE>"
QDM_DIR = "<SET YOUR PATH HERE>"

# 2. SETTINGS
VAR_NAME = 'tas'  # As seen in your print(ds)
START_YEAR = 2081
END_YEAR = 2100

# ================= CALCULATION FUNCTION =================
def calculate_qdm_std(folder_path, start_year, end_year, gdf):
    print(f"Starting QDM Far-term Analysis ({start_year}-{end_year})...")
    
    # 1. Gather Files (Pattern: YYYY-MM-DD_INM_CM5_0.nc)
    files = []
    for y in range(start_year, end_year + 1):
        # Matches "2081-01-01_INM_CM5_0.nc"
        found = sorted(glob.glob(os.path.join(folder_path, f"{y}-*-*_INM_CM5_0.nc")))
        files.extend(found)
        
    if not files:
        print(f"  [ERROR] No files found in {folder_path} for years {start_year}-{end_year}")
        return np.nan

    print(f"  -> Found {len(files)} daily files.")

    try:
        # 2. Open with Dask Chunks (Lazy Loading)
        # Using combine='nested' and concat_dim='time' because individual files lack time dim on variable
        with xr.open_mfdataset(files, combine='nested', concat_dim='time', parallel=True, chunks={'time': 100}) as ds:
            
            # --- Fix Dimensions (lon/lat -> x/y) ---
            # Your print(ds) showed 'lon' and 'lat' dimensions
            rename_dict = {}
            if 'lon' in ds.dims: rename_dict['lon'] = 'x'
            if 'lat' in ds.dims: rename_dict['lat'] = 'y'
            ds = ds.rename(rename_dict)
            
            ds = ds.rio.write_crs("EPSG:4326")
            
            # --- Spatial Slicing ---
            minx, miny, maxx, maxy = gdf.total_bounds
            
            # Handle Descending Latitude (49.95 -> 0.05)
            # Your print(ds) confirmed lat is descending
            if ds.y[0] > ds.y[-1]:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(maxy, miny))
            else:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(miny, maxy))

            # --- Clip to India Shapefile ---
            clipped = ds_box[VAR_NAME].rio.clip(gdf.geometry.values, gdf.crs, drop=True)

            # --- THE "ONE-PASS" ALGORITHM ---
            print(f"  -> Distributing tasks to 128 CPUs...")
            
            # Define lazy calculations
            count_lazy = clipped.count()
            sum_lazy = clipped.sum()
            sum_sq_lazy = (clipped**2).sum()
            
            # Compute all 3 stats at once
            count_val, sum_val, sum_sq_val = dask.compute(count_lazy, sum_lazy, sum_sq_lazy)
            
            # --- Final Statistics ---
            N = count_val.item()
            Sum = sum_val.item()
            SumSq = sum_sq_val.item()
            
            if N == 0: 
                print("  [ERROR] No valid data points found after clipping.")
                return np.nan
            
            mean = Sum / N
            mean_sq = SumSq / N
            
            # Variance = E[X^2] - (E[X])^2
            variance = mean_sq - (mean**2)
            
            if variance < 0: variance = 0
                
            std_val = np.sqrt(variance)
            
            return std_val

    except Exception as e:
        print(f"  [CRASH] Error: {e}")
        import traceback
        traceback.print_exc()
        return np.nan

# ================= MAIN =================
def main():
    # Load Shapefile
    print("Loading Shapefile...")
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs != "EPSG:4326": 
        gdf = gdf.to_crs("EPSG:4326")
    
    # Calculate
    std_val = calculate_qdm_std(QDM_DIR, START_YEAR, END_YEAR, gdf)

    # Output
    print("\n" + "="*50)
    if not np.isnan(std_val):
        print(f"QDM absolute std: {std_val:.2f}°C")
    else:
        print(f"QDM absolute std: [CALCULATION FAILED]")
    print("="*50)

if __name__ == "__main__":
    main()

In [ ]:
!python qdm_std.py

In [ ]:
%%writefile spatial_corr_final.py
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import glob
import dask
import rioxarray
import warnings

# Optimize for your 128 cores
dask.config.set(scheduler='processes')
warnings.filterwarnings("ignore")

# ================= CONFIGURATION =================
# 1. PATHS
SHP_PATH = "<SET YOUR PATH HERE>"
MSWX_DIR = "<SET YOUR PATH HERE>"
DDPM_DIR = "<SET YOUR PATH HERE>"
QDM_DIR  = "<SET YOUR PATH HERE>"

# 2. SETTINGS
# Baseline (MSWX)
MSWX_START = 1985
MSWX_END   = 2014

# Future (Models)
FUTURE_START = 2081
FUTURE_END   = 2100

# ================= FUNCTION: COMPUTE CLIMATOLOGY (MEAN MAP) =================
def get_climatology(folder_path, start_year, end_year, gdf, label):
    print(f"Computing Climatology for {label} ({start_year}-{end_year})...")
    
    # 1. Find Files
    files = []
    for y in range(start_year, end_year + 1):
        # Flexible pattern to catch both YYYYDDD and YYYY-MM-DD
        found = sorted(glob.glob(os.path.join(folder_path, f"*{y}*.nc")))
        files.extend(found)
        
    if not files:
        print(f"  [ERROR] No files found for {label}")
        return None

    try:
        # 2. Open Lazy
        # We use chunks to prevent loading everything into RAM
        with xr.open_mfdataset(files, combine='nested', concat_dim='time', parallel=True, chunks={'time': 100}) as ds:
            
            # --- Auto-Detect Variable ---
            if 'tas' in ds: var = 'tas'
            elif 'air_temperature' in ds: var = 'air_temperature'
            else: 
                print(f"  [ERROR] Variable not found (checked 'tas', 'air_temperature')")
                return None

            # --- Standardize Dimensions ---
            rename = {}
            if 'lon' in ds.dims: rename['lon'] = 'x'
            if 'lat' in ds.dims: rename['lat'] = 'y'
            ds = ds.rename(rename).rio.write_crs("EPSG:4326")
            
            # --- Spatial Slicing ---
            minx, miny, maxx, maxy = gdf.total_bounds
            if ds.y[0] > ds.y[-1]:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(maxy, miny))
            else:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(miny, maxy))

            # --- Clip to India ---
            clipped = ds_box[var].rio.clip(gdf.geometry.values, gdf.crs, drop=True)

            # --- COMPUTE MEAN (COLLAPSE TIME) ---
            # This is the step: .mean(dim='time')
            # It returns a 2D array (Lat x Lon) of the average temperature
            print(f"  -> Averaging {len(files)} daily files (collapsing time)...")
            mean_map = clipped.mean(dim='time').compute()
            
            print("  -> Done.")
            return mean_map

    except Exception as e:
        print(f"  [CRASH] {label}: {e}")
        return None

# ================= MAIN =================
def main():
    print("Loading Shapefile...")
    gdf = gpd.read_file(SHP_PATH).to_crs("EPSG:4326")
    
    print("="*60)
    
    # 1. Get 2D Mean Maps
    # shape: (Lat, Lon) e.g., (500, 500)
    map_mswx = get_climatology(MSWX_DIR, MSWX_START, MSWX_END, gdf, "MSWX Baseline")
    map_ddpm = get_climatology(DDPM_DIR, FUTURE_START, FUTURE_END, gdf, "DDPM Far-term")
    map_qdm  = get_climatology(QDM_DIR,  FUTURE_START, FUTURE_END, gdf, "QDM Far-term")
    
    print("="*60)

    if map_mswx is None or map_ddpm is None or map_qdm is None:
        print("Comparison failed due to missing data.")
        return

    # 2. Align Grids (Crucial for Correlation)
    # Even if dimensions look same, slight coordinate shifts cause NaNs in correlation.
    # We force DDPM/QDM to match MSWX's exact pixel grid.
    print("\nAligning grids...")
    map_ddpm_aligned = map_ddpm.rio.reproject_match(map_mswx)
    map_qdm_aligned  = map_qdm.rio.reproject_match(map_mswx)

    # 3. Flatten and Correlate
    # We flatten the 2D maps into 1D arrays
    v_mswx = map_mswx.values.flatten()
    v_ddpm = map_ddpm_aligned.values.flatten()
    v_qdm  = map_qdm_aligned.values.flatten()

    # 4. Handle NaNs (Ocean pixels)
    # We only want to correlate pixels where ALL datasets have data (Land)
    valid_mask = ~np.isnan(v_mswx) & ~np.isnan(v_ddpm) & ~np.isnan(v_qdm)
    
    clean_mswx = v_mswx[valid_mask]
    clean_ddpm = v_ddpm[valid_mask]
    clean_qdm  = v_qdm[valid_mask]

    print(f"Calculating correlation over {len(clean_mswx)} land pixels...")

    # 5. Compute Spatial Pattern Correlation
    corr_QDM_MSWX  = np.corrcoef(clean_qdm,  clean_mswx)[0, 1]
    corr_DDPM_MSWX = np.corrcoef(clean_ddpm, clean_mswx)[0, 1]

    # 6. Output
    print("\n" + "="*60)
    print("SPATIAL PATTERN CORRELATION (Baseline vs Future)")
    print("Does the future map look like the real world map?")
    print("="*60)
    print(f"QDM vs MSWX Correlation:  {corr_QDM_MSWX:.4f}")
    print(f"DDPM vs MSWX Correlation: {corr_DDPM_MSWX:.4f}")
    
    print("-" * 60)
    if corr_DDPM_MSWX > corr_QDM_MSWX:
        print("WINNER: DDPM preserves the spatial structure better.")
    else:
        print("WINNER: QDM preserves the spatial structure better.")
    print("="*60)

if __name__ == "__main__":
    main()

In [ ]:
!python spatial_corr_final.py

In [ ]:
%%writefile warming_anomaly.py
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import glob
import dask
import rioxarray
import warnings

# Optimize for 128 cores
dask.config.set(scheduler='processes')
warnings.filterwarnings("ignore")

# ================= CONFIGURATION =================
# 1. PATHS
SHP_PATH = "<SET YOUR PATH HERE>"
MSWX_DIR = "<SET YOUR PATH HERE>"
DDPM_DIR = "<SET YOUR PATH HERE>"

# 2. SETTINGS
# Baseline (MSWX)
MSWX_START = 1985
MSWX_END   = 2014

# Future (DDPM)
FUTURE_START = 2081
FUTURE_END   = 2100

# ================= FUNCTION: COMPUTE CLIMATOLOGY =================
def get_climatology(folder_path, start_year, end_year, gdf, label):
    print(f"Computing Climatology for {label} ({start_year}-{end_year})...")
    
    files = []
    for y in range(start_year, end_year + 1):
        found = sorted(glob.glob(os.path.join(folder_path, f"*{y}*.nc")))
        files.extend(found)
        
    if not files:
        print(f"  [ERROR] No files found for {label}")
        return None

    try:
        with xr.open_mfdataset(files, combine='nested', concat_dim='time', parallel=True, chunks={'time': 100}) as ds:
            
            # Auto-Detect Variable
            if 'tas' in ds: var = 'tas'
            elif 'air_temperature' in ds: var = 'air_temperature'
            else: return None

            # Standardize Dimensions
            rename = {}
            if 'lon' in ds.dims: rename['lon'] = 'x'
            if 'lat' in ds.dims: rename['lat'] = 'y'
            ds = ds.rename(rename).rio.write_crs("EPSG:4326")
            
            # Spatial Slicing
            minx, miny, maxx, maxy = gdf.total_bounds
            if ds.y[0] > ds.y[-1]:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(maxy, miny))
            else:
                ds_box = ds.sel(x=slice(minx, maxx), y=slice(miny, maxy))

            # Clip
            clipped = ds_box[var].rio.clip(gdf.geometry.values, gdf.crs, drop=True)

            # Compute Time-Mean (Climatology)
            mean_map = clipped.mean(dim='time').compute()
            
            return mean_map

    except Exception as e:
        print(f"  [CRASH] {label}: {e}")
        return None

# ================= MAIN =================
def main():
    print("Loading Shapefile...")
    gdf = gpd.read_file(SHP_PATH).to_crs("EPSG:4326")
    
    # 1. Compute Maps
    print("="*60)
    map_mswx = get_climatology(MSWX_DIR, MSWX_START, MSWX_END, gdf, "MSWX Baseline")
    map_ddpm = get_climatology(DDPM_DIR, FUTURE_START, FUTURE_END, gdf, "DDPM Future")
    print("="*60)

    if map_mswx is None or map_ddpm is None:
        print("Calculation failed.")
        return

    # 2. Align Grids (Essential for subtraction)
    print("\nAligning DDPM grid to MSWX grid...")
    map_ddpm_aligned = map_ddpm.rio.reproject_match(map_mswx)

    # 3. Calculate RAW Warming Map (Future - Baseline)
    # This is the map of "+3°C here, +5°C there"
    raw_warming_map = map_ddpm_aligned - map_mswx
    
    # 4. Calculate Domain Mean Warming
    # This is the single scalar number (e.g., +4.2°C average over India)
    domain_mean_warming = raw_warming_map.mean().item()
    
    # 5. Calculate Warming ANOMALY
    # (Where does it warm MORE or LESS than the average?)
    # Positive values = Hotspots (e.g., Mountains)
    # Negative values = Relative Cool spots (e.g., Coasts)
    warming_anomaly = raw_warming_map - domain_mean_warming

    # 6. Statistics
    min_anom = warming_anomaly.min().item()
    max_anom = warming_anomaly.max().item()
    std_anom = warming_anomaly.std().item()

    print("\n" + "="*60)
    print("WARMING ANOMALY ANALYSIS")
    print("============================================================")
    print(f"Average Warming across India:  +{domain_mean_warming:.2f}°C")
    print("-" * 60)
    print(f"Region warming MOST above avg: +{max_anom:.2f}°C (likely Himalayas)")
    print(f"Region warming LEAST above avg: {min_anom:.2f}°C (likely Coasts/South)")
    print(f"Spatial Variability (Std Dev):  {std_anom:.2f}°C")
    print("="*60)
    
    # Optional: Save this map to check in QGIS/Panoply
    # warming_anomaly.to_netcdf("warming_anomaly_ddpm.nc")
    # print("Saved 'warming_anomaly_ddpm.nc' for visual inspection.")

if __name__ == "__main__":
    main()

In [ ]:
!python warming_anomaly.py

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import os
import glob
import numpy as np
import geopandas as gpd
from joblib import Parallel, delayed
from netCDF4 import Dataset
from matplotlib.gridspec import GridSpec

try:
    import rasterio.features
    import rasterio.transform
except ImportError:
    raise ImportError("Please install rasterio: pip install rasterio")

# ==========================================================
# 1. CONFIGURATION
# ==========================================================
base_dir = "<SET YOUR PATH HERE>"
shp_path = "<SET YOUR PATH HERE>"
mswx_hr_dir = "<SET YOUR PATH HERE>"

# QDM and DDPM directories
qdm_dir = "<SET YOUR PATH HERE>"
ddpm_dir = os.path.join(base_dir, "DDPM_ssp_585_INM_CM5_0")

N_CORES = 80
FAR_TERM_YEARS = range(2081, 2101)
BASELINE_YEARS = range(1985, 2015)  # 30-year baseline

# ==========================================================
# 2. HELPER FUNCTIONS
# ==========================================================
def load_mswx_year(year):
    """Load single year of MSWX data."""
    files = sorted(glob.glob(os.path.join(mswx_hr_dir, f"{year}*.nc")))
    if not files:
        return None
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    da = ds["air_temperature"]
    if da.lat[0] > da.lat[-1]:
        da = da.reindex(lat=da.lat[::-1])
    return da.mean("time")


def process_batch(files):
    """Process batch of NetCDF files for parallel loading."""
    s, c = None, 0
    for f in files:
        try:
            with Dataset(f) as nc:
                data = nc.variables["tas"][:]
                if data.ndim == 3:
                    data = data[0]
                data = np.asarray(data, dtype=np.float64)
                if s is None:
                    s = np.zeros_like(data)
                s += data
                c += 1
        except:
            continue
    return s, c


def get_period_mean(folder, years, n_cores=N_CORES):
    """Compute climatological mean for a period."""
    files = []
    for y in years:
        files.extend(glob.glob(os.path.join(folder, f"{y}-*-*.nc")))
    if not files:
        print(f"  WARNING: No files found in {folder} for years {years.start}-{years.stop-1}")
        return None, None, None

    print(f"  Found {len(files)} files")
    
    chunks = np.array_split(sorted(files), n_cores)
    results = Parallel(n_jobs=n_cores)(
        delayed(process_batch)(c) for c in chunks
    )

    total, count = None, 0
    for s, c in results:
        if s is not None:
            total = s if total is None else total + s
            count += c
    if count == 0:
        return None, None, None

    print(f"  Processed {count} files successfully")

    with xr.open_dataset(files[0]) as ds:
        lats = ds.lat.values
        lons = ds.lon.values

    mean = total / count

    # Enforce south → north
    if lats[0] > lats[-1]:
        mean = np.flipud(mean)
        lats = lats[::-1]

    # Convert K to °C if needed
    if np.nanmean(mean) > 200:
        mean -= 273.15

    return mean, lats, lons


def create_india_mask(lats, lons, shp_path):
    """Create boolean mask for India land area."""
    gdf = gpd.read_file(shp_path)
    if lats[0] > lats[-1]:
        lats = lats[::-1]

    lon_res = abs(lons[1] - lons[0])
    lat_res = abs(lats[1] - lats[0])

    transform = (
        rasterio.transform.Affine.translation(
            lons.min() - lon_res / 2,
            lats.max() + lat_res / 2
        )
        * rasterio.transform.Affine.scale(lon_res, -lat_res)
    )

    mask = rasterio.features.geometry_mask(
        gdf.geometry,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        invert=True,
        all_touched=True,
    )
    return np.flipud(mask)


# ==========================================================
# 3. LOAD ALL DATA
# ==========================================================
print("=" * 70)
print("QDM vs DDPM COMPARISON - SSP5-8.5 FAR-TERM (2081-2100)")
print("=" * 70)

# --- MSWX Baseline ---
print(f"\n[1/4] Loading MSWX baseline ({BASELINE_YEARS.start}-{BASELINE_YEARS.stop-1})...")
baseline_years_data = []
for y in BASELINE_YEARS:
    da_y = load_mswx_year(y)
    if da_y is not None:
        baseline_years_data.append(da_y)
        
da_mswx_baseline = xr.concat(baseline_years_data, dim="year").mean("year")
if da_mswx_baseline.lat[0] > da_mswx_baseline.lat[-1]:
    da_mswx_baseline = da_mswx_baseline.reindex(lat=da_mswx_baseline.lat[::-1])
print(f"  MSWX baseline shape: {da_mswx_baseline.shape}")

# --- DDPM Far-term ---
print(f"\n[2/4] Loading DDPM far-term ({FAR_TERM_YEARS.start}-{FAR_TERM_YEARS.stop-1})...")
ddpm_clim, lats, lons = get_period_mean(ddpm_dir, FAR_TERM_YEARS)
print(f"  DDPM climatology shape: {ddpm_clim.shape}")

# --- QDM Far-term ---
print(f"\n[3/4] Loading QDM far-term ({FAR_TERM_YEARS.start}-{FAR_TERM_YEARS.stop-1})...")
qdm_clim, qdm_lats, qdm_lons = get_period_mean(qdm_dir, FAR_TERM_YEARS)
print(f"  QDM climatology shape: {qdm_clim.shape}")

# --- Create mask and interpolate baseline ---
print("\n[4/4] Creating India mask and interpolating baseline...")
india_mask = create_india_mask(lats, lons, shp_path)
mswx_interp = da_mswx_baseline.interp(lat=lats, lon=lons, method="linear").values

# Apply mask to all data
mswx_masked = np.where(india_mask, mswx_interp, np.nan)
ddpm_masked = np.where(india_mask, ddpm_clim, np.nan)
qdm_masked = np.where(india_mask, qdm_clim, np.nan)

print("  Data loading complete!")

# ==========================================================
# 4. COMPUTE STATISTICS
# ==========================================================
print("\n" + "=" * 70)
print("COMPUTING COMPARISON STATISTICS")
print("=" * 70)

# Flatten for correlation (land pixels only)
mswx_flat = mswx_masked[~np.isnan(mswx_masked)]
ddpm_flat = ddpm_masked[~np.isnan(ddpm_masked)]
qdm_flat = qdm_masked[~np.isnan(qdm_masked)]

n_pixels = len(mswx_flat)
print(f"\nAnalyzing {n_pixels:,} land pixels...")

# Pattern correlations
corr_qdm = np.corrcoef(qdm_flat, mswx_flat)[0, 1]
corr_ddpm = np.corrcoef(ddpm_flat, mswx_flat)[0, 1]

# Spatial standard deviations (absolute temperature)
std_mswx = np.std(mswx_flat)
std_qdm = np.std(qdm_flat)
std_ddpm = np.std(ddpm_flat)

# Variance retention
var_ret_qdm = (std_qdm / std_mswx) * 100
var_ret_ddpm = (std_ddpm / std_mswx) * 100

# Anomalies (future - baseline)
ddpm_anom = ddpm_masked - mswx_masked
qdm_anom = qdm_masked - mswx_masked

# Anomaly standard deviations
std_anom_qdm = np.nanstd(qdm_anom)
std_anom_ddpm = np.nanstd(ddpm_anom)

# Domain-mean warming
mean_warming_ddpm = np.nanmean(ddpm_anom)
mean_warming_qdm = np.nanmean(qdm_anom)

# Warming heterogeneity (deviation from domain mean)
warming_deviation = ddpm_anom - mean_warming_ddpm

# Print results
print("\n" + "-" * 70)
print("SPATIAL PATTERN CORRELATION (Future Climatology vs MSWX Baseline)")
print("-" * 70)
print(f"  QDM vs MSWX:   r = {corr_qdm:.4f}")
print(f"  DDPM vs MSWX:  r = {corr_ddpm:.4f}")
print(f"  → DDPM improvement: +{(corr_ddpm - corr_qdm):.4f}")

print("\n" + "-" * 70)
print("SPATIAL VARIANCE (Absolute Temperature Climatology)")
print("-" * 70)
print(f"  MSWX baseline std:  {std_mswx:.2f}°C (reference)")
print(f"  QDM absolute std:   {std_qdm:.2f}°C  ({var_ret_qdm:.1f}% retention)")
print(f"  DDPM absolute std:  {std_ddpm:.2f}°C  ({var_ret_ddpm:.1f}% retention)")

print("\n" + "-" * 70)
print("ANOMALY VARIANCE (Reflects pattern mismatch with baseline)")
print("-" * 70)
print(f"  QDM anomaly std:    {std_anom_qdm:.2f}°C")
print(f"  DDPM anomaly std:   {std_anom_ddpm:.2f}°C")
print(f"  → Lower DDPM anomaly std = better pattern preservation")

print("\n" + "-" * 70)
print("MEAN WARMING (Domain Average)")
print("-" * 70)
print(f"  QDM mean warming:   +{mean_warming_qdm:.2f}°C")
print(f"  DDPM mean warming:  +{mean_warming_ddpm:.2f}°C")

print("\n" + "-" * 70)
print("WARMING HETEROGENEITY (DDPM)")
print("-" * 70)
print(f"  Max warming above domain mean: +{np.nanmax(warming_deviation):.2f}°C")
print(f"  Min warming below domain mean: {np.nanmin(warming_deviation):.2f}°C")
print(f"  Spatial variability (std):     {np.nanstd(warming_deviation):.2f}°C")

# ==========================================================
# 5. CREATE COMPARISON FIGURE
# ==========================================================
print("\n" + "=" * 70)
print("GENERATING COMPARISON FIGURE")
print("=" * 70)

gdf = gpd.read_file(shp_path)

fig = plt.figure(figsize=(22, 14))
gs = GridSpec(2, 4, figure=fig, height_ratios=[1, 1],
              hspace=0.18, wspace=0.05,
              left=0.04, right=0.96, top=0.93, bottom=0.08)

# --- Color scales ---
# Temperature (absolute)
vmin_temp, vmax_temp = -10, 32
temp_cmap = plt.cm.Spectral_r
temp_norm = mcolors.Normalize(vmin=vmin_temp, vmax=vmax_temp)

# Difference (DDPM - QDM)
diff_data = ddpm_masked - qdm_masked
diff_max = np.ceil(np.nanpercentile(np.abs(diff_data), 99) * 2) / 2
diff_norm = mcolors.TwoSlopeNorm(vmin=-diff_max, vcenter=0, vmax=diff_max)
diff_cmap = plt.cm.RdBu_r

# Anomaly (symmetric around 0)
anom_max = np.ceil(max(np.nanpercentile(np.abs(qdm_anom), 99),
                       np.nanpercentile(np.abs(ddpm_anom), 99)))
anom_norm = mcolors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=anom_max)
anom_cmap = plt.cm.RdBu_r

# Warming heterogeneity
warm_max = np.ceil(max(abs(np.nanmin(warming_deviation)),
                       abs(np.nanmax(warming_deviation))) * 2) / 2
warm_norm = mcolors.TwoSlopeNorm(vmin=-warm_max, vcenter=0, vmax=warm_max)

# ==========================================================
# ROW 1: Climatological Temperature Fields
# ==========================================================
row1_data = [mswx_masked, qdm_masked, ddpm_masked, diff_data]
row1_titles = [
    f"(a) MSWX Baseline\n(1985–2014)",
    f"(b) QDM Climatology\n(2081–2100)",
    f"(c) DDPM Climatology\n(2081–2100)",
    f"(d) DDPM − QDM\nDifference"
]
row1_cmaps = [temp_cmap, temp_cmap, temp_cmap, diff_cmap]
row1_norms = [temp_norm, temp_norm, temp_norm, diff_norm]

row1_ims = []
for i in range(4):
    ax = fig.add_subplot(gs[0, i], projection=ccrs.PlateCarree())
    ax.set_extent([66, 100, 6, 38])
    ax.add_geometries(gdf.geometry, ccrs.PlateCarree(),
                      facecolor="none", edgecolor="black", linewidth=0.8)
    ax.coastlines(alpha=0.5, linewidth=0.5)

    im = ax.pcolormesh(lons, lats, row1_data[i],
                       cmap=row1_cmaps[i], norm=row1_norms[i],
                       transform=ccrs.PlateCarree())
    row1_ims.append(im)

    ax.set_title(row1_titles[i], fontsize=14, fontweight='bold', pad=10)

    # Add statistics annotations
    if i == 1:  # QDM
        ax.text(0.03, 0.03, f"r = {corr_qdm:.4f}\nσ = {std_qdm:.2f}°C",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
    elif i == 2:  # DDPM
        ax.text(0.03, 0.03, f"r = {corr_ddpm:.4f}\nσ = {std_ddpm:.2f}°C",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
    elif i == 0:  # MSWX
        ax.text(0.03, 0.03, f"σ = {std_mswx:.2f}°C\n(reference)",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# ==========================================================
# ROW 2: Anomalies and Warming Heterogeneity
# ==========================================================
row2_data = [qdm_anom, ddpm_anom, warming_deviation]
row2_titles = [
    f"(e) QDM Anomaly\n(vs MSWX Baseline)",
    f"(f) DDPM Anomaly\n(vs MSWX Baseline)",
    f"(g) Warming Heterogeneity\n(Deviation from +{mean_warming_ddpm:.2f}°C)"
]
row2_cmaps = [anom_cmap, anom_cmap, plt.cm.BrBG_r]
row2_norms = [anom_norm, anom_norm, warm_norm]

row2_ims = []
for i in range(3):
    ax = fig.add_subplot(gs[1, i], projection=ccrs.PlateCarree())
    ax.set_extent([66, 100, 6, 38])
    ax.add_geometries(gdf.geometry, ccrs.PlateCarree(),
                      facecolor="none", edgecolor="black", linewidth=0.8)
    ax.coastlines(alpha=0.5, linewidth=0.5)

    im = ax.pcolormesh(lons, lats, row2_data[i],
                       cmap=row2_cmaps[i], norm=row2_norms[i],
                       transform=ccrs.PlateCarree())
    row2_ims.append(im)

    ax.set_title(row2_titles[i], fontsize=14, fontweight='bold', pad=10)

    # Add statistics annotations
    if i == 0:  # QDM anomaly
        ax.text(0.03, 0.03, f"Mean: +{mean_warming_qdm:.2f}°C\nStd: {std_anom_qdm:.2f}°C",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
    elif i == 1:  # DDPM anomaly
        ax.text(0.03, 0.03, f"Mean: +{mean_warming_ddpm:.2f}°C\nStd: {std_anom_ddpm:.2f}°C",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
    elif i == 2:  # Warming heterogeneity
        ax.text(0.03, 0.03, f"Max: +{np.nanmax(warming_deviation):.2f}°C\nMin: {np.nanmin(warming_deviation):.2f}°C",
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                verticalalignment='bottom',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# ==========================================================
# PANEL (h): Summary Statistics Table
# ==========================================================
ax_table = fig.add_subplot(gs[1, 3])
ax_table.axis('off')
ax_table.set_xlim(0, 1)
ax_table.set_ylim(0, 1)

# Title
ax_table.text(0.5, 0.98, '(h) Quantitative Comparison',
              fontsize=14, fontweight='bold',
              ha='center', va='top')

# Table data
table_data = [
    ['Metric', 'QDM', 'DDPM', 'Winner'],
    ['Pattern Corr (r)', f'{corr_qdm:.4f}', f'{corr_ddpm:.4f}', 'DDPM ✓'],
    ['Abs. Temp σ (°C)', f'{std_qdm:.2f}', f'{std_ddpm:.2f}', 'DDPM ✓' if std_ddpm > std_qdm else 'QDM'],
    ['Variance Ret. (%)', f'{var_ret_qdm:.1f}', f'{var_ret_ddpm:.1f}', 'DDPM ✓' if var_ret_ddpm > var_ret_qdm else 'QDM'],
    ['Anomaly σ (°C)', f'{std_anom_qdm:.2f}', f'{std_anom_ddpm:.2f}', 'DDPM ✓'],
    ['Mean Warming (°C)', f'+{mean_warming_qdm:.2f}', f'+{mean_warming_ddpm:.2f}', '—'],
]

# Draw table manually
y_positions = [0.88, 0.76, 0.64, 0.52, 0.40, 0.28]
x_positions = [0.02, 0.42, 0.62, 0.85]

for row_idx, row in enumerate(table_data):
    y = y_positions[row_idx]
    
    if row_idx == 0:  # Header row
        for col_idx, (text, x) in enumerate(zip(row, x_positions)):
            ax_table.text(x, y, text, fontsize=11, fontweight='bold',
                         ha='left' if col_idx == 0 else 'center', va='center',
                         family='monospace')
        # Draw separator line below header
        ax_table.plot([0.02, 0.98], [y - 0.06, y - 0.06], 
                     color='gray', linewidth=1, clip_on=False)
    else:  # Data rows
        for col_idx, (text, x) in enumerate(zip(row, x_positions)):
            color = 'darkgreen' if 'DDPM' in text and '✓' in text else 'black'
            weight = 'bold' if 'DDPM' in text and '✓' in text else 'normal'
            ax_table.text(x, y, text, fontsize=10,
                         ha='left' if col_idx == 0 else 'center', va='center',
                         family='monospace', color=color, fontweight=weight)


# ==========================================================
# COLORBARS
# ==========================================================
# Temperature colorbar (Row 1, panels a-c)
cax1 = fig.add_axes([0.04, 0.52, 0.52, 0.02])
cb1 = fig.colorbar(row1_ims[0], cax=cax1, orientation='horizontal', extend='both')
cb1.set_label('Mean Temperature (°C)', fontsize=12)

# Difference colorbar (Row 1, panel d)
cax2 = fig.add_axes([0.60, 0.52, 0.15, 0.02])
cb2 = fig.colorbar(row1_ims[3], cax=cax2, orientation='horizontal', extend='both')
cb2.set_label('DDPM − QDM (°C)', fontsize=12)

# Anomaly colorbar (Row 2, panels e-f)
cax3 = fig.add_axes([0.04, 0.04, 0.40, 0.02])
cb3 = fig.colorbar(row2_ims[0], cax=cax3, orientation='horizontal', extend='both')
cb3.set_label('Temperature Anomaly vs Baseline (°C)', fontsize=12)

# Warming heterogeneity colorbar (Row 2, panel g)
cax4 = fig.add_axes([0.48, 0.04, 0.20, 0.02])
cb4 = fig.colorbar(row2_ims[2], cax=cax4, orientation='horizontal', extend='both')
cb4.set_label('Warming Deviation (°C)', fontsize=12)

# ==========================================================
# SAVE AND SHOW
# ==========================================================
# plt.suptitle('Spatial Pattern Fidelity: QDM vs DDPM under SSP5-8.5 (2081–2100)',
#              fontsize=18, fontweight='bold', y=0.98)

# output_path = "QDM_vs_DDPM_comparison_SSP585_farterm.png"
# plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
# print(f"\nFigure saved: {output_path}")

plt.show()

print("\n" + "=" * 70)
print("FIGURE GENERATION COMPLETE")
print("=" * 70)

In [ ]:
import xarray as xr
import numpy as np
import datetime

def main():
    # --- 1. CONFIGURATION ---
    input_path = "<SET YOUR PATH HERE>"
    ref_path   = "<SET YOUR PATH HERE>"
    output_path = "<SET YOUR PATH HERE>"

    # --- 2. LOAD DATA ---
    print("Loading datasets...")
    ds_imd = xr.open_dataset(input_path)
    ds_ref = xr.open_dataset(ref_path)

    # --- 3. DEFINE "TRUTH" MASK (Land Definition) ---
    print("Creating high-resolution land mask...")
    # Convert Boolean Mask -> Float (1.0 = Land, 0.0 = Ocean) for interpolation
    mask_imd_numeric = ds_imd['tmean'].isel(time=0).notnull().astype(float)
    
    # Interpolate 'Nearest' to project the exact 1.0 degree land shape onto 0.1 grid
    expected_land_mask = mask_imd_numeric.interp_like(ds_ref, method="nearest")

    # --- 4. HYBRID INTERPOLATION (The Q1 Method) ---
    print("Executing Hybrid Interpolation...")
    
    # A. Linear: High accuracy, smooth gradients (Primary)
    da_linear = ds_imd['tmean'].interp_like(ds_ref, method="linear")
    
    # B. Nearest: Blocky, but preserves edges (Secondary)
    da_nearest = ds_imd['tmean'].interp_like(ds_ref, method="nearest")
    
    # C. Hybrid Fill: Use Linear, fill gaps (coastlines) with Nearest
    da_hybrid = da_linear.fillna(da_nearest)
    
    # D. Masking: Enforce the 'Truth' mask to remove ocean overflow
    da_final = da_hybrid.where(expected_land_mask > 0)

    # --- 5. METADATA INJECTION (CRITICAL FOR JOURNALS) ---
    print("Attaching metadata provenance...")

    # Create a new Dataset
    ds_final = da_final.to_dataset(name='tmean')

    # A. Global Attributes (Dataset level)
    ds_final.attrs = ds_imd.attrs.copy()  # Start with original attributes
    ds_final.attrs.update({
        'title': 'Spatially Downscaled IMD Temperature (0.1 deg)',
        'source_original': 'India Meteorological Department (IMD) 1.0 deg Gridded Data',
        'source_reference': 'MSWX High-Resolution Reference Grid (0.1 deg)',
        'interpolation_method': 'Hybrid Scheme: Bilinear Interpolation (Interior) + Nearest Neighbor (Coastal Boundaries)',
        'justification': 'Hybrid method used to preserve spatial gradients while preventing data loss at complex coastlines (e.g., Hofstra et al., 2008).',
        'creation_date': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'created_by': 'User/Researcher Name', # Update this
        'software': f'Python {np.__version__}, xarray {xr.__version__}',
        'spatial_resolution': '0.1 degree',
        'masking_logic': 'Restricted to valid land pixels derived from nearest-neighbor projection of original IMD mask.'
    })

    # B. Variable Attributes (Data level)
    ds_final['tmean'].attrs = ds_imd['tmean'].attrs.copy()
    ds_final['tmean'].attrs.update({
        'long_name': 'Mean Temperature',
        'standard_name': 'air_temperature',
        'units': 'degC', # Ensure this matches your data
        'valid_range': [ds_final['tmean'].min().values, ds_final['tmean'].max().values],
        'comment': 'interpolated from 1.0 deg to 0.1 deg'
    })

    # --- 6. SAVE AND COMPRESS ---
    print(f"Saving file with metadata to: {output_path}")
    
    # Compression saves disk space and improves I/O speed
    encoding = {
        'tmean': {
            'zlib': True, 
            'complevel': 5, 
            'dtype': 'float32', # float32 is sufficient for temp, saves 50% space
            '_FillValue': -9999.0
        }
    }
    
    ds_final.to_netcdf(output_path, encoding=encoding)
    print("Success! File saved.")

    # --- 7. VERIFY METADATA ---
    print("\n--- METADATA VERIFICATION ---")
    print(ds_final)

if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# 1. Load Data with Chunks (Parallel processing enabled)
# 'chunks' is critical here so it doesn't load 16,000 days into RAM at once.
print("Loading data and computing mean (this may take a moment)...")
ds_orig = xr.open_dataset("<SET YOUR PATH HERE>", chunks={'time': -1})
ds_interp = xr.open_dataset("<SET YOUR PATH HERE>", chunks={'time': -1})

# 2. Compute the Time-Mean (Climatology)
# .compute() triggers the actual calculation
mean_orig = ds_orig['tmean'].mean(dim='time').compute()
mean_interp = ds_interp['tmean'].mean(dim='time').compute()

# 3. Setup Figure
fig, axes = plt.subplots(1, 2, figsize=(18, 9), subplot_kw={'projection': ccrs.PlateCarree()})

# Calculate min/max from the MEAN data (Climatology has narrower range than daily data)
vmin = min(mean_orig.min(), mean_interp.min())
vmax = max(mean_orig.max(), mean_interp.max())

# Geographic Window
india_extent = [66, 98, 6, 38] 

datasets = [mean_orig, mean_interp]
titles = ["Original IMD Mean (1.0°)", "Interpolated IMD Mean (0.1°)"]

for i, ax in enumerate(axes):
    ax.set_extent(india_extent, crs=ccrs.PlateCarree())
    
    # Q1 Journal Features
    ax.add_feature(cfeature.COASTLINE, linewidth=1.2)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=1.0)
    ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
    
    # Plotting
    im = datasets[i].plot(
        ax=ax, 
        transform=ccrs.PlateCarree(),
        cmap='RdYlBu_r',
        add_colorbar=False,
        vmin=vmin,
        vmax=vmax
    )
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(titles[i], fontsize=15, fontweight='bold', pad=10)

# Shared Colorbar
cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Time-Averaged Mean Temperature (°C)', fontsize=12, fontweight='bold')

plt.suptitle(f"Long-Term Climatological Mean Comparison (1979-2023)", fontsize=18, fontweight='bold', y=0.98)
plt.show()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

# Path to the folder containing the ENSEMBLE OUTPUT NetCDF files
# (This matches the OUTPUT_FOLDER from your inference script)
ENSEMBLE_OUTPUT_FOLDER = "<SET YOUR PATH HERE>"

# Path to the HIGH-RESOLUTION GROUND TRUTH folder
HR_FOLDER = "<SET YOUR PATH HERE>"

# Variable name used in the NetCDF files
VAR_NAME = "air_temperature"

# Output filename for the plot
OUTPUT_PLOT_FILE = "uncertainty_vs_anomaly_scatter.png"

# Classification Thresholds (Percentiles)
# Top 10% = Heatwave, Bottom 10% = Cold Spell
PERCENTILE_LOW = 10
PERCENTILE_HIGH = 90

# ============================================================================
# DATA PROCESSING
# ============================================================================

def load_and_process_data(ensemble_folder, hr_folder, var_name):
    """
    Loads ensemble outputs and matching ground truth to calculate
    daily spatial means for anomaly and uncertainty.
    """
    ensemble_path = Path(ensemble_folder)
    hr_path = Path(hr_folder)
    
    # Find all ensemble output files
    ensemble_files = sorted(list(ensemble_path.glob("*_ddpm_ensemble_*.nc")))
    
    if not ensemble_files:
        print(f"No ensemble files found in {ensemble_path}")
        return []

    processed_data = []

    print(f"Found {len(ensemble_files)} files. Processing metrics...")

    for ens_file in tqdm(ensemble_files, desc="Reading Data"):
        try:
            # Extract date from filename (assumes format YYYYDDD_...)
            file_name = ens_file.name
            date_str = file_name.split('_')[0]
            
            # 1. Load Ensemble Data
            with xr.open_dataset(ens_file) as ds:
                # Get the spatial mean of the ensemble uncertainty (std dev)
                # We average over lat/lon to get one point per day
                ens_std_map = ds[f'{var_name}_ensemble_std'].values
                daily_uncertainty = np.nanmean(ens_std_map)
                
                # Get the ensemble mean prediction map
                ens_mean_map = ds[f'{var_name}_ensemble_mean'].values
            
            # 2. Load Ground Truth (HR) Data
            # HR files are typically named YYYYDDD.nc
            hr_file = hr_path / f"{date_str}.nc"
            
            # Fallback search if exact name doesn't match
            if not hr_file.exists():
                potential_matches = list(hr_path.glob(f"{date_str}*.nc"))
                if potential_matches:
                    hr_file = potential_matches[0]
                else:
                    # Skip this day if no ground truth is available
                    continue

            with xr.open_dataset(hr_file) as hr_ds:
                if var_name in hr_ds:
                    gt_map = hr_ds[var_name].values
                    # Handle if data has time dim (1, H, W) -> (H, W)
                    if gt_map.ndim == 3:
                        gt_map = gt_map[0]
                    
                    # 3. Calculate Anomaly (Error)
                    # Anomaly = Predicted - Actual
                    # Positive = Model is warmer than reality
                    # Negative = Model is colder than reality
                    anomaly_map = ens_mean_map - gt_map
                    daily_anomaly = np.nanmean(anomaly_map)
                    
                    # Store absolute temperature for classification
                    daily_temp = np.nanmean(gt_map)
                    
                    processed_data.append({
                        'date': date_str,
                        'uncertainty': daily_uncertainty,
                        'anomaly': daily_anomaly,
                        'temperature': daily_temp
                    })
                    
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue
            
    return processed_data

def classify_events(data, p_low, p_high):
    """Classifies days into Heatwave, Cold Spell, or Normal."""
    if not data:
        return []
        
    temps = [d['temperature'] for d in data]
    
    # Calculate thresholds based on the loaded data distribution
    thresh_low = np.percentile(temps, p_low)
    thresh_high = np.percentile(temps, p_high)
    
    print(f"\nClassification Thresholds:")
    print(f"  Cold Spell < {thresh_low:.2f} (Bottom {p_low}%)")
    print(f"  Heatwave   > {thresh_high:.2f} (Top {100-p_high}%)")
    
    for entry in data:
        t = entry['temperature']
        if t >= thresh_high:
            entry['type'] = 'Heatwave'
        elif t <= thresh_low:
            entry['type'] = 'Cold Spell'
        else:
            entry['type'] = 'Normal'
            
    return data

# ============================================================================
# PLOTTING
# ============================================================================

def plot_uncertainty_scatter(data, output_file):
    """Generates the scatterplot."""
    if not data:
        print("No data to plot.")
        return

    # Extract arrays for plotting
    anomalies = [d['anomaly'] for d in data]
    uncertainties = [d['uncertainty'] for d in data]
    types = [d['type'] for d in data]
    
    # Setup Figure
    plt.figure(figsize=(11, 8))
    sns.set_style("whitegrid")
    
    # Define Colors
    colors = {
        'Heatwave': '#d62728',   # Red
        'Cold Spell': '#1f77b4', # Blue
        'Normal': '#7f7f7f'      # Gray
    }
    
    markers = {
        'Heatwave': 'o',
        'Cold Spell': 'o',
        'Normal': 'o'
    }
    
    # Convert to numpy for easier masking
    anomalies = np.array(anomalies)
    uncertainties = np.array(uncertainties)
    types = np.array(types)
    
    # Plot 'Normal' points first so they are in the background
    mask = types == 'Normal'
    if np.any(mask):
        plt.scatter(anomalies[mask], uncertainties[mask], 
                   c=colors['Normal'], label='Normal Days', alpha=0.4, s=40, edgecolors='w')

    # Plot 'Cold Spell'
    mask = types == 'Cold Spell'
    if np.any(mask):
        plt.scatter(anomalies[mask], uncertainties[mask], 
                   c=colors['Cold Spell'], label='Cold Spell', alpha=0.8, s=80, edgecolors='w')
        
    # Plot 'Heatwave'
    mask = types == 'Heatwave'
    if np.any(mask):
        plt.scatter(anomalies[mask], uncertainties[mask], 
                   c=colors['Heatwave'], label='Heatwave', alpha=0.8, s=80, edgecolors='w')

    # Add reference lines
    plt.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    
    # Labels and Title
    plt.title('Uncertainty vs. Temperature Anomaly\n(DDPM Ensemble)', fontsize=16, fontweight='bold')
    plt.xlabel('Temperature Anomaly (°C)\n(Model Prediction - Ground Truth)', fontsize=12)
    plt.ylabel('Model Uncertainty (°C)\n(Ensemble Standard Deviation)', fontsize=12)
    
    # Annotations explaining quadrants (optional but helpful)
    plt.text(min(anomalies), max(uncertainties), 'High Uncertainty\nUnder-prediction', 
             ha='left', va='top', fontsize=9, alpha=0.6, bbox=dict(facecolor='white', alpha=0.7))
    plt.text(max(anomalies), max(uncertainties), 'High Uncertainty\nOver-prediction', 
             ha='right', va='top', fontsize=9, alpha=0.6, bbox=dict(facecolor='white', alpha=0.7))

    plt.legend(title='Event Type', loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
    plt.tight_layout()
    
    # Save
    # plt.savefig(output_file, dpi=300, bbox_inches='tight')
    # print(f"\nPlot saved to: {output_file}")
    plt.show()

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # 1. Load Data
    data = load_and_process_data(ENSEMBLE_OUTPUT_FOLDER, HR_FOLDER, VAR_NAME)
    
    # 2. Classify Events
    data = classify_events(data, PERCENTILE_LOW, PERCENTILE_HIGH)
    
    # 3. Generate Plot
    plot_uncertainty_scatter(data, OUTPUT_PLOT_FILE)

In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

LR_FOLDER = "<SET YOUR PATH HERE>"
MODEL_PATH = "<SET YOUR PATH HERE>"
CLIMATOLOGY_PATH = "<SET YOUR PATH HERE>"

DAYS_TO_PLOT = [
    {'date': 2013150, 'label': 'Normal Day'},
    {'date': 2014178, 'label': 'Heatwave Peak'}
]

# Reduced to practical sizes for visualization (N=50 is fine with batching)
ENSEMBLE_SIZES = [2, 10, 50]
MAX_N = max(ENSEMBLE_SIZES)

# PERFORMANCE CONFIG
BATCH_SIZE = 64  # Process 64 patches at once. Lower to 32 if you get CUDA OOM.

# Model Config
IN_CHANNELS = 2
OUT_CHANNELS = 1
TIME_DIM = 256
BASE_DIM = 64
PATCH_SIZE = 128
STRIDE = 64
SIGMA_FACTOR = 0.3
DDIM_STEPS = 50
VAR_NAME = "air_temperature"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# 2. MODEL DEFINITIONS (Exact match to checkpoint)
# ============================================================================

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv.unbind(1)
        scale = 1.0 / math.sqrt(C)
        attn = torch.einsum('bci,bcj->bij', q, k) * scale
        attn = attn.softmax(dim=-1)
        h = torch.einsum('bij,bcj->bci', attn, v).reshape(B, C, H, W)
        return x + self.proj(h)

class ResNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t):
        h = self.activation(self.norm1(self.conv1(x)))
        h = h + self.activation(self.time_mlp(t))[..., None, None]
        h = self.activation(self.norm2(self.conv2(self.dropout(h))))
        return h + self.skip(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_layers=2, downsample=True, attention=False):
        super().__init__()
        self.downsample = downsample
        self.resnets = nn.ModuleList([ResNetBlock(in_ch if i == 0 else out_ch, out_ch, time_emb_dim) for i in range(num_layers)])
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in range(num_layers)]) if attention else [None]*num_layers
        self.downsample_conv = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1) if downsample else None
    def forward(self, x, t):
        for resnet, attn in zip(self.resnets, self.attentions):
            x = resnet(x, t)
            if attn: x = attn(x)
        if self.downsample_conv: x = self.downsample_conv(x)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_layers=2, upsample=True, attention=False):
        super().__init__()
        self.upsample_conv = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1) if upsample else None
        self.resnets = nn.ModuleList()
        for i in range(num_layers):
            self.resnets.append(ResNetBlock(in_ch + out_ch if i == 0 else out_ch, out_ch, time_emb_dim))
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in range(num_layers)]) if attention else [None]*num_layers
    def forward(self, x, skip, t):
        if self.upsample_conv: x = self.upsample_conv(x)
        x = torch.cat([x, skip], dim=1)
        for resnet, attn in zip(self.resnets, self.attentions):
            x = resnet(x, t)
            if attn: x = attn(x)
        return x

class EnhancedUNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, time_dim=256, base_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim),
        )
        self.conv_in = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        down_dims = [base_dim, base_dim * 2, base_dim * 4, base_dim * 8]
        self.downs = nn.ModuleList([
            DownBlock(down_dims[0], down_dims[1], time_dim, attention=False),
            DownBlock(down_dims[1], down_dims[2], time_dim, attention=False),
            DownBlock(down_dims[2], down_dims[3], time_dim, attention=True),
        ])
        self.bottleneck = ResNetBlock(down_dims[3], down_dims[3], time_dim)
        self.bottleneck_attn = AttentionBlock(down_dims[3])
        self.ups = nn.ModuleList([
            UpBlock(down_dims[3], down_dims[2], time_dim, attention=True),
            UpBlock(down_dims[2], down_dims[1], time_dim, attention=False),
            UpBlock(down_dims[1], down_dims[0], time_dim, attention=False),
        ])
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, base_dim), nn.SiLU(), nn.Conv2d(base_dim, out_channels, 3, padding=1),
        )
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x = self.conv_in(x)
        skips = [x]
        x = self.downs[0](x, t_emb); skips.append(x)
        x = self.downs[1](x, t_emb); skips.append(x)
        x = self.downs[2](x, t_emb)
        x = self.bottleneck_attn(self.bottleneck(x, t_emb))
        x = self.ups[0](x, skips[2], t_emb)
        x = self.ups[1](x, skips[1], t_emb)
        x = self.ups[2](x, skips[0], t_emb)
        return self.conv_out(x)

# ============================================================================
# 3. OPTIMIZED BATCHED INFERENCE
# ============================================================================

def create_gaussian_weights(patch_size, sigma_factor=0.3):
    center = patch_size // 2
    y, x = np.ogrid[:patch_size, :patch_size]
    sigma = sigma_factor * patch_size
    dist_sq = (x - center)**2 + (y - center)**2
    weights = np.exp(-dist_sq / (2 * sigma**2))
    return (weights / weights.max()).astype(np.float32)

@torch.no_grad()
def generate_ensemble_for_day_fast(date_int, model, normalizer, device, n_samples=50, batch_size=BATCH_SIZE):
    """
    Optimized generation using Batch Processing.
    Instead of 1 patch at a time, we process 64 patches at once.
    """
    print(f"\nProcessing {date_int} (Ensemble N={n_samples})...")
    
    # 1. Load File
    lr_folder = Path(LR_FOLDER)
    matches = list(lr_folder.glob(f"{date_int}*1deg_to_hr.nc"))
    if not matches:
        # Fallback search
        matches = list(lr_folder.glob(f"{date_int}*.nc"))
        matches = [f for f in matches if "1deg" in f.name or "LR" in f.name]
    
    if not matches:
        print(f"  Error: File not found for {date_int}")
        return None
    
    lr_path = matches[0]
    with xr.open_dataset(lr_path) as ds:
        lr_img = ds[VAR_NAME].values
        if lr_img.ndim == 3: lr_img = lr_img[0]
        # Quick NaN fill
        mask = np.isnan(lr_img)
        if mask.any():
            from scipy.interpolate import NearestNDInterpolator
            coords = np.array(np.nonzero(~mask)).T
            values = lr_img[~mask]
            if len(values) > 0:
                interp = NearestNDInterpolator(coords, values)
                lr_img[mask] = interp(np.array(np.nonzero(mask)).T)
            else:
                lr_img[:] = 0

    # 2. Extract and Pre-Normalize ALL Patches
    h, w = lr_img.shape
    patches_list = []
    positions = []
    
    # Define patch grid
    for top in range(0, h - PATCH_SIZE + 1, STRIDE):
        for left in range(0, w - PATCH_SIZE + 1, STRIDE):
            patches_list.append(lr_img[top:top+PATCH_SIZE, left:left+PATCH_SIZE])
            positions.append((top, left))
            
    # Add edges
    if (w - PATCH_SIZE) % STRIDE != 0:
        left = w - PATCH_SIZE
        for top in range(0, h - PATCH_SIZE + 1, STRIDE):
            patches_list.append(lr_img[top:top+PATCH_SIZE, left:left+PATCH_SIZE])
            positions.append((top, left))
    if (h - PATCH_SIZE) % STRIDE != 0:
        top = h - PATCH_SIZE
        for left in range(0, w - PATCH_SIZE + 1, STRIDE):
            patches_list.append(lr_img[top:top+PATCH_SIZE, left:left+PATCH_SIZE])
            positions.append((top, left))
    if (h - PATCH_SIZE) % STRIDE != 0 and (w - PATCH_SIZE) % STRIDE != 0:
        patches_list.append(lr_img[h-PATCH_SIZE:h, w-PATCH_SIZE:w])
        positions.append((h-PATCH_SIZE, w-PATCH_SIZE))

    # Normalize patches and stack into one big tensor
    tensor_patches = []
    for patch, (top, left) in zip(patches_list, positions):
        mean_p = normalizer['mean'][top:top+PATCH_SIZE, left:left+PATCH_SIZE]
        std_p = normalizer['std'][top:top+PATCH_SIZE, left:left+PATCH_SIZE]
        std_p_safe = np.where(std_p < 1e-6, 1e-6, std_p)
        norm_p = (patch - mean_p) / std_p_safe
        norm_p = np.clip(norm_p, -5, 5)
        tensor_patches.append(norm_p)
        
    # Shape: (Num_Patches, 1, H, W) -> (Num_Patches, 1, 128, 128)
    all_patches_tensor = torch.from_numpy(np.stack(tensor_patches)).float().unsqueeze(1).to(device)
    num_total_patches = len(all_patches_tensor)
    
    print(f"  Total patches: {num_total_patches} | Processing in batches of {batch_size}")
    
    # Pre-calculate gaussian weights for stitching
    gaussian_w = create_gaussian_weights(PATCH_SIZE, SIGMA_FACTOR)
    
    # Setup Scheduler
    betas = torch.linspace(0.0001, 0.02, 1000, device=device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    seq = range(0, 1000, 1000//DDIM_STEPS)
    
    # 3. Ensemble Generation Loop
    ensemble_results = []
    
    # Outer loop: Ensemble members (Stochasticity happens here)
    for member_idx in tqdm(range(n_samples), desc="Generating Members"):
        
        # Holder for stitched image
        full_pred = np.zeros((h, w), dtype=np.float32)
        weight_sum = np.zeros((h, w), dtype=np.float32)
        
        # Inner Loop: Batches of patches
        # We process 'batch_size' patches simultaneously
        for b_start in range(0, num_total_patches, batch_size):
            b_end = min(b_start + batch_size, num_total_patches)
            
            # Get batch of condition (LR) patches
            # Shape: (Batch_Size, 1, 128, 128)
            cond_batch = all_patches_tensor[b_start:b_end]
            current_bs = cond_batch.shape[0]
            
            # Start from random noise (Stochastic!)
            x_t = torch.randn_like(cond_batch)
            
            # Run DDIM Steps on the Batch
            with torch.amp.autocast('cuda'):
                for t_idx in reversed(range(len(seq))):
                    t = seq[t_idx]
                    prev_t = seq[t_idx-1] if t_idx > 0 else 0
                    
                    # Create time tensor for the whole batch
                    t_tensor = torch.full((current_bs,), t, device=device, dtype=torch.long)
                    
                    # Predict noise
                    model_input = torch.cat([x_t, cond_batch], dim=1)
                    noise_pred = model(model_input, t_tensor)
                    
                    # Update Step
                    alpha_t = alphas_cumprod[t]
                    alpha_prev = alphas_cumprod[prev_t]
                    sigma = 0 
                    
                    pred_x0 = (x_t - (1 - alpha_t).sqrt() * noise_pred) / alpha_t.sqrt()
                    dir_xt = (1 - alpha_prev - sigma**2).sqrt() * noise_pred
                    x_t = alpha_prev.sqrt() * pred_x0 + dir_xt
            
            # Stitch batch results back to image
            decoded_batch = x_t.cpu().float().numpy().squeeze(1) # (BS, 128, 128)
            
            for i, idx in enumerate(range(b_start, b_end)):
                patch_out = decoded_batch[i]
                top, left = positions[idx]
                
                full_pred[top:top+PATCH_SIZE, left:left+PATCH_SIZE] += patch_out * gaussian_w
                weight_sum[top:top+PATCH_SIZE, left:left+PATCH_SIZE] += gaussian_w
        
        # Denormalize full image
        norm_res = full_pred / np.where(weight_sum < 1e-8, 1.0, weight_sum)
        final_res = norm_res * normalizer['std'] + normalizer['mean']
        ensemble_results.append(final_res)
        
    return np.stack(ensemble_results, axis=0)

# ============================================================================
# 4. MAIN
# ============================================================================

def main():
    print("Loading model...")
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    
    model = EnhancedUNet(IN_CHANNELS, OUT_CHANNELS, TIME_DIM, BASE_DIM)
    
    # Load Weights (handling EMA mismatch gracefully)
    if 'ema_state_dict' in checkpoint:
        try:
            model.load_state_dict(checkpoint['ema_state_dict'])
        except RuntimeError:
            print("  EMA keys mismatch, using standard weights.")
            model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        
    model.to(DEVICE)
    model.eval()
    
    print("Loading climatology...")
    with xr.open_dataset(CLIMATOLOGY_PATH) as ds:
        normalizer = {
            'mean': ds['mean_map'].values,
            'std': ds['std_map'].values
        }

    # Setup Plot
    fig, axes = plt.subplots(len(DAYS_TO_PLOT), len(ENSEMBLE_SIZES), figsize=(15, 9))
    
    for row_idx, day_info in enumerate(DAYS_TO_PLOT):
        date = day_info['date']
        label = day_info['label']
        
        # Run FAST inference
        ensemble_stack = generate_ensemble_for_day_fast(date, model, normalizer, DEVICE, n_samples=MAX_N)
        
        if ensemble_stack is None: continue
        
        for col_idx, n in enumerate(ENSEMBLE_SIZES):
            ax = axes[row_idx, col_idx]
            
            # Slice first N members
            subset = ensemble_stack[:n, :, :]
            
            # Compute Uncertainty
            uncertainty_map = np.std(subset, axis=0)
            
            # Plot
            im = ax.imshow(uncertainty_map, cmap='magma', vmin=0, vmax=2.5)
            
            if row_idx == 0:
                ax.set_title(f"Ensemble Size: N={n}", fontsize=14, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(f"{label}\n({date})", fontsize=14, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label('Spatial Uncertainty Std Dev (°C)', fontsize=12)
    
    # plt.suptitle("Convergence of Spatial Uncertainty (Optimized Batched Inference)", fontsize=16, y=0.96)
    # plt.savefig("spatial_uncertainty_fast.png", dpi=300, bbox_inches='tight')
    # print("\nDone! Plot saved to spatial_uncertainty_fast.png")
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
import rioxarray
from pathlib import Path
from tqdm import tqdm
import traceback
from shapely.geometry import mapping

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

LR_FOLDER = "<SET YOUR PATH HERE>"
HR_FOLDER = "<SET YOUR PATH HERE>"
MODEL_PATH = "<SET YOUR PATH HERE>"
CLIMATOLOGY_PATH = "<SET YOUR PATH HERE>"
SHAPEFILE_PATH = "<SET YOUR PATH HERE>"

DAYS_TO_PLOT = [
    {'date': 2013150, 'label': 'Normal Day'},
    {'date': 2014178, 'label': 'Heatwave Peak'}
]

ENSEMBLE_SIZES_MAP = [2, 10, 50]
STATS_STEPS = list(range(2, 51, 2))
MAX_N = 50
BATCH_SIZE = 64

IN_CHANNELS = 2
OUT_CHANNELS = 1
TIME_DIM = 256
BASE_DIM = 64
PATCH_SIZE = 128
STRIDE = 64
SIGMA_FACTOR = 0.3
DDIM_STEPS = 50
VAR_NAME = "air_temperature"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================================
# 2. ROBUST MASKING
# ============================================================================

def create_static_mask(reference_nc_path, shapefile_path):
    print("Creating static India mask...")
    try:
        ds = xr.open_dataset(reference_nc_path)
        if VAR_NAME in ds: grid = ds[VAR_NAME]
        else: grid = ds[list(ds.data_vars)[0]]
        
        if 'time' in grid.dims: grid = grid.isel(time=0)
        
        lon_name, lat_name = None, None
        for c in grid.coords:
            if any(x in c.lower() for x in ['lon', 'long', 'x']): lon_name = c
            if any(x in c.lower() for x in ['lat', 'y']): lat_name = c
        if not lon_name: lon_name = 'lon' if 'lon' in grid.dims else 'longitude'
        if not lat_name: lat_name = 'lat' if 'lat' in grid.dims else 'latitude'

        # Fix 0-360 -> -180-180
        if grid[lon_name].max() > 180:
            print(f"  Detected 0-360 longitude. Converting to -180 to 180...")
            grid.coords[lon_name] = (grid.coords[lon_name] + 180) % 360 - 180
            grid = grid.sortby(lon_name)
            
        # Handle Descending Latitude
        orig_lats = grid[lat_name].values
        is_descending = orig_lats[0] > orig_lats[-1]
        if is_descending:
            grid = grid.sortby(lat_name) # Force ascending for clipping

        # Clip
        grid.rio.set_spatial_dims(x_dim=lon_name, y_dim=lat_name, inplace=True)
        grid.rio.write_crs("EPSG:4326", inplace=True)
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
        
        print("  Clipping...")
        masked = grid.rio.clip(gdf.geometry.values, gdf.crs, drop=False, invert=False, all_touched=True)
        mask_array = ~np.isnan(masked.values)
        
        # Revert orientation if original was descending
        if is_descending:
            print("  Reverting mask orientation to match model output...")
            mask_array = np.flipud(mask_array)
            
        print(f"✓ Mask created. Valid pixels: {np.sum(mask_array)}")
        return mask_array
        
    except Exception as e:
        print(f"Error creating mask: {e}")
        traceback.print_exc()
        return None

# ============================================================================
# 3. METRICS
# ============================================================================

def compute_crps_ensemble(ensemble_preds, ground_truth):
    n = ensemble_preds.shape[0]
    mae = np.mean(np.abs(ensemble_preds - ground_truth), axis=0)
    preds_sorted = np.sort(ensemble_preds, axis=0)
    diff = preds_sorted[1:] - preds_sorted[:-1]
    weights = np.arange(1, n) * np.arange(n - 1, 0, -1)
    spread = np.sum(diff * weights[:, None], axis=0) / (n**2 * (n - 1) / 2) if n > 1 else 0
    return np.mean(mae - 0.5 * spread)

def compute_masked_statistics(ensemble_stack, hr_image, n_subset, full_std_map, mask):
    ens_valid = ensemble_stack[:n_subset][:, mask]
    hr_valid = hr_image[mask]
    
    ens_mean = np.mean(ens_valid, axis=0)
    ens_std = np.std(ens_valid, axis=0)
    
    rmse = np.sqrt(np.mean((ens_mean - hr_valid)**2))
    spread = np.sqrt(np.mean(ens_std**2))
    ratio = spread / rmse if rmse > 0 else 0
    crps = compute_crps_ensemble(ens_valid, hr_valid)
    unc_corr = np.corrcoef(ens_std, full_std_map[mask])[0, 1]
    
    return {'N': n_subset, 'RMSE': rmse, 'Spread': spread, 'Spread_Error_Ratio': ratio, 'CRPS': crps, 'Uncertainty_Corr': unc_corr}

# ============================================================================
# 4. MODEL (Corrected Init Calls)
# ============================================================================

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv.unbind(1)
        scale = 1.0 / math.sqrt(C)
        attn = torch.einsum('bci,bcj->bij', q, k) * scale
        attn = attn.softmax(dim=-1)
        h = torch.einsum('bij,bcj->bci', attn, v).reshape(B, C, H, W)
        return x + self.proj(h)

class ResNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.activation = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t):
        h = self.activation(self.norm1(self.conv1(x)))
        h = h + self.activation(self.time_mlp(t))[..., None, None]
        h = self.activation(self.norm2(self.conv2(h))) # Corrected input to conv2
        return h + self.skip(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_layers=2, downsample=True, attention=False):
        super().__init__()
        self.downsample = downsample
        self.resnets = nn.ModuleList([ResNetBlock(in_ch if i == 0 else out_ch, out_ch, time_emb_dim) for i in range(num_layers)])
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in range(num_layers)]) if attention else [None]*num_layers
        self.downsample_conv = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1) if downsample else None
    def forward(self, x, t):
        for resnet, attn in zip(self.resnets, self.attentions):
            x = resnet(x, t)
            if attn: x = attn(x)
        if self.downsample_conv: x = self.downsample_conv(x)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_layers=2, upsample=True, attention=False):
        super().__init__()
        self.upsample = upsample
        self.upsample_conv = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1) if upsample else None
        self.resnets = nn.ModuleList()
        for i in range(num_layers):
            self.resnets.append(ResNetBlock(in_ch + out_ch if i == 0 else out_ch, out_ch, time_emb_dim))
        self.attentions = nn.ModuleList([AttentionBlock(out_ch) for _ in range(num_layers)]) if attention else [None]*num_layers
    def forward(self, x, skip, t):
        if self.upsample_conv: x = self.upsample_conv(x)
        x = torch.cat([x, skip], dim=1)
        for resnet, attn in zip(self.resnets, self.attentions):
            x = resnet(x, t)
            if attn: x = attn(x)
        return x

class EnhancedUNet(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, base_dim):
        super().__init__()
        self.time_mlp = nn.Sequential(SinusoidalPositionEmbeddings(time_dim), nn.Linear(time_dim, time_dim*4), nn.SiLU(), nn.Linear(time_dim*4, time_dim))
        self.conv_in = nn.Conv2d(in_channels, base_dim, 3, padding=1)
        d = [base_dim, base_dim*2, base_dim*4, base_dim*8]
        
        # --- FIX: USE KEYWORD ARGS TO PREVENT num_layers=False BUG ---
        self.downs = nn.ModuleList([
            DownBlock(d[0], d[1], time_dim, attention=False),
            DownBlock(d[1], d[2], time_dim, attention=False),
            DownBlock(d[2], d[3], time_dim, attention=True)
        ])
        
        self.bottleneck = ResNetBlock(d[3], d[3], time_dim)
        self.bottleneck_attn = AttentionBlock(d[3])
        
        self.ups = nn.ModuleList([
            UpBlock(d[3], d[2], time_dim, attention=True),
            UpBlock(d[2], d[1], time_dim, attention=False),
            UpBlock(d[1], d[0], time_dim, attention=False)
        ])
        self.conv_out = nn.Sequential(nn.GroupNorm(8, base_dim), nn.SiLU(), nn.Conv2d(base_dim, out_channels, 3, padding=1))

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        x = self.conv_in(x)
        skips = [x]
        x = self.downs[0](x, t_emb); skips.append(x)
        x = self.downs[1](x, t_emb); skips.append(x)
        x = self.downs[2](x, t_emb)
        x = self.bottleneck_attn(self.bottleneck(x, t_emb))
        x = self.ups[0](x, skips[2], t_emb)
        x = self.ups[1](x, skips[1], t_emb)
        x = self.ups[2](x, skips[0], t_emb)
        return self.conv_out(x)

# ============================================================================
# 5. INFERENCE
# ============================================================================

def create_gaussian_weights(patch_size, sigma_factor=0.3):
    center = patch_size // 2
    y, x = np.ogrid[:patch_size, :patch_size]
    weights = np.exp(-((x - center)**2 + (y - center)**2) / (2 * (sigma_factor * patch_size)**2))
    return (weights / weights.max()).astype(np.float32)

@torch.no_grad()
def generate_ensemble_for_day_fast(date_int, model, normalizer, device, n_samples=50):
    print(f"\nProcessing {date_int} (Ensemble N={n_samples})...")
    lr_folder = Path(LR_FOLDER)
    matches = list(lr_folder.glob(f"{date_int}*1deg_to_hr.nc"))
    if not matches: matches = [f for f in list(lr_folder.glob(f"{date_int}*.nc")) if "1deg" in f.name or "LR" in f.name]
    if not matches: return None, None
    
    with xr.open_dataset(matches[0]) as ds:
        lr_img = ds[VAR_NAME].values
        if lr_img.ndim == 3: lr_img = lr_img[0]
        if np.isnan(lr_img).any():
            from scipy.interpolate import NearestNDInterpolator
            mask = np.isnan(lr_img)
            coords = np.array(np.nonzero(~mask)).T
            lr_img[mask] = NearestNDInterpolator(coords, lr_img[~mask])(np.array(np.nonzero(mask)).T)

    hr_folder = Path(HR_FOLDER)
    hr_matches = list(hr_folder.glob(f"{date_int}*.nc"))
    hr_img = None
    if hr_matches:
        with xr.open_dataset(hr_matches[0]) as ds:
            hr_img = ds[VAR_NAME].values
            if hr_img.ndim == 3: hr_img = hr_img[0]

    h, w = lr_img.shape
    pad_h = (STRIDE - (h - PATCH_SIZE) % STRIDE) % STRIDE
    pad_w = (STRIDE - (w - PATCH_SIZE) % STRIDE) % STRIDE
    lr_padded = np.pad(lr_img, ((0, pad_h), (0, pad_w)), mode='reflect')
    ph, pw = lr_padded.shape
    
    patches_list, positions = [], []
    for top in range(0, ph - PATCH_SIZE + 1, STRIDE):
        for left in range(0, pw - PATCH_SIZE + 1, STRIDE):
            patches_list.append(lr_padded[top:top+PATCH_SIZE, left:left+PATCH_SIZE])
            positions.append((top, left))
            
    tensor_patches = []
    for patch, (top, left) in zip(patches_list, positions):
        m_top, m_left = min(top, h-PATCH_SIZE), min(left, w-PATCH_SIZE)
        mean_p = normalizer['mean'][m_top:m_top+PATCH_SIZE, m_left:m_left+PATCH_SIZE]
        std_p = normalizer['std'][m_top:m_top+PATCH_SIZE, m_left:m_left+PATCH_SIZE]
        norm_p = np.clip((patch - mean_p) / np.where(std_p < 1e-6, 1e-6, std_p), -5, 5)
        tensor_patches.append(norm_p)
    all_patches_tensor = torch.from_numpy(np.stack(tensor_patches)).float().unsqueeze(1).to(device)
    
    gaussian_w = create_gaussian_weights(PATCH_SIZE, SIGMA_FACTOR)
    betas = torch.linspace(0.0001, 0.02, 1000, device=device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    seq = range(0, 1000, 1000//DDIM_STEPS)
    ensemble_results = []
    
    for _ in tqdm(range(n_samples), desc="Generating"):
        full_pred = np.zeros((ph, pw), dtype=np.float32)
        weight_sum = np.zeros((ph, pw), dtype=np.float32)
        
        for b_start in range(0, len(all_patches_tensor), BATCH_SIZE):
            cond_batch = all_patches_tensor[b_start:min(b_start + BATCH_SIZE, len(all_patches_tensor))]
            bs = cond_batch.shape[0]
            x_t = torch.randn_like(cond_batch)
            
            with torch.amp.autocast('cuda'):
                for t_idx in reversed(range(len(seq))):
                    t_val, t_prev = seq[t_idx], seq[t_idx-1] if t_idx > 0 else 0
                    t_tensor = torch.full((bs,), t_val, device=device, dtype=torch.long)
                    noise_pred = model(torch.cat([x_t, cond_batch], dim=1), t_tensor)
                    at, atp = alphas_cumprod[t_val], alphas_cumprod[t_prev]
                    x_t = atp.sqrt() * (x_t - (1-at).sqrt()*noise_pred)/at.sqrt() + (1-atp).sqrt()*noise_pred
            
            decoded = x_t.cpu().float().numpy().squeeze(1)
            for i, p_idx in enumerate(range(b_start, b_start + bs)):
                top, left = positions[p_idx]
                full_pred[top:top+PATCH_SIZE, left:left+PATCH_SIZE] += decoded[i] * gaussian_w
                weight_sum[top:top+PATCH_SIZE, left:left+PATCH_SIZE] += gaussian_w
                
        norm_res = full_pred / np.where(weight_sum < 1e-8, 1.0, weight_sum)
        ensemble_results.append(norm_res[:h, :w] * normalizer['std'][:h, :w] + normalizer['mean'][:h, :w])
        
    return np.stack(ensemble_results, axis=0), hr_img

# ============================================================================
# 6. MAIN
# ============================================================================

def load_checkpoint_robust(model, path, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    state_dict = checkpoint.get('ema_state_dict', checkpoint['model_state_dict'])
    new_sd = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    try:
        model.load_state_dict(new_sd)
        print("Weights loaded.")
    except Exception as e:
        print(f"Standard load failed, trying strict=False: {e}")
        model.load_state_dict(new_sd, strict=False)

def main():
    print("Loading model...")
    model = EnhancedUNet(IN_CHANNELS, OUT_CHANNELS, TIME_DIM, BASE_DIM).to(DEVICE)
    load_checkpoint_robust(model, MODEL_PATH, DEVICE)
    model.eval()
    
    print("Loading Climatology...")
    with xr.open_dataset(CLIMATOLOGY_PATH) as ds:
        normalizer = {'mean': ds['mean_map'].values, 'std': ds['std_map'].values}
        
    india_mask = create_static_mask(CLIMATOLOGY_PATH, SHAPEFILE_PATH)
    if india_mask is None: return

    all_stats = []
    fig_maps, axes_maps = plt.subplots(len(DAYS_TO_PLOT), len(ENSEMBLE_SIZES_MAP), figsize=(15, 9))
    
    for row_idx, day_info in enumerate(DAYS_TO_PLOT):
        date, label = day_info['date'], day_info['label']
        ensemble_stack, hr_img = generate_ensemble_for_day_fast(date, model, normalizer, DEVICE, n_samples=MAX_N)
        if ensemble_stack is None: continue
        
        print(f"  Calculating MASKED stats for {label}...")
        full_std = np.std(ensemble_stack, axis=0)
        for n in tqdm(STATS_STEPS, desc="Stats"):
            if hr_img is not None:
                stats = compute_masked_statistics(ensemble_stack, hr_img, n, full_std, india_mask)
                stats.update({'Date': date, 'Label': label})
                all_stats.append(stats)
        
        for col_idx, n in enumerate(ENSEMBLE_SIZES_MAP):
            ax = axes_maps[row_idx, col_idx]
            unc_map = np.where(india_mask, np.std(ensemble_stack[:n], axis=0), np.nan)
            im = ax.imshow(unc_map, cmap='magma', vmin=0, vmax=2.5)
            if row_idx == 0: ax.set_title(f"N={n}", fontsize=12, fontweight='bold')
            if col_idx == 0: ax.set_ylabel(f"{label}\n({date})", fontsize=12)
            ax.set_xticks([]); ax.set_yticks([])

    plt.figure(fig_maps.number)
    cbar_ax = fig_maps.add_axes([0.92, 0.15, 0.02, 0.7])
    fig_maps.colorbar(im, cax=cbar_ax, label='Uncertainty Std Dev (°C)')
    plt.suptitle("Spatial Uncertainty (Masked)", fontsize=16, y=0.95)
    plt.savefig("spatial_uncertainty_maps_masked.png", dpi=300, bbox_inches='tight')
    
    if all_stats:
        df = pd.DataFrame(all_stats)
        fig_stats, axes_stats = plt.subplots(1, 3, figsize=(18, 5))
        sns.lineplot(data=df, x='N', y='CRPS', hue='Label', ax=axes_stats[0], marker='o')
        axes_stats[0].set_title("Probabilistic Skill (CRPS)")
        sns.lineplot(data=df, x='N', y='Spread_Error_Ratio', hue='Label', ax=axes_stats[1], marker='o')
        axes_stats[1].axhline(1.0, color='k', linestyle='--')
        axes_stats[1].set_title("Reliability (Spread/RMSE)")
        sns.lineplot(data=df, x='N', y='Uncertainty_Corr', hue='Label', ax=axes_stats[2], marker='o')
        axes_stats[2].set_title("Spatial Convergence")
        plt.tight_layout()
        plt.savefig("ensemble_convergence_stats_masked.png", dpi=300, bbox_inches='tight')
        
        print("\n" + "="*80)
        print("FINAL STATISTICS (N=50)")
        print(df[df['N']==50][['Label', 'RMSE', 'CRPS', 'Spread_Error_Ratio']].round(4).to_string(index=False))
        print("="*80)

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Publication-quality spatial diagnostics figure: DDPM vs MSWX
=============================================================
Upgraded for 2x2 grid, maximum distance legibility, uncluttered geography, 
and accurate min-max symmetric colorbars. Scaled specifically for Word embedding.
"""

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
from scipy.stats import norm as scipy_norm
import glob, os, warnings
import geopandas as gpd
import rioxarray

warnings.filterwarnings("ignore")

# ── optional imports ──────────────────────────────────────────────────────────
try:
    import cmcrameri.cm as cmc
    CMAP_TEMP = cmc.roma          
    CMAP_RMSE = cmc.devon_r       
    CMAP_BIAS = cmc.vik           
    print("cmcrameri loaded ✓")
except ImportError:
    print("cmcrameri not found — falling back to matplotlib defaults")
    CMAP_TEMP = "RdYlBu_r"
    CMAP_RMSE = "YlOrRd"
    CMAP_BIAS = "RdBu_r"

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = True
    print("cartopy loaded ✓")
except ImportError:
    HAS_CARTOPY = False
    print("cartopy not found — plotting without true projection")

# =============================================================================
# PATHS
# =============================================================================
HR_MSWX    = "<SET YOUR PATH HERE>"
HR_DDPM    = "<SET YOUR PATH HERE>"
INDIA_SHP  = "<SET YOUR PATH HERE>"
ELEV_NC    = "<SET YOUR PATH HERE>"
SAVE_PATH  = "<SET YOUR PATH HERE>"

# =============================================================================
# HELPERS
# =============================================================================
def get_date(fname):
    base = os.path.basename(fname).replace("_ddpm", "").replace(".nc", "")
    return int(base)

def doy_to_dt(yydoy):
    yr  = int(str(yydoy)[:4])
    doy = int(str(yydoy)[4:])
    return pd.to_datetime(f"{yr}-{doy:03d}", format="%Y-%j")

def filter_period(files, start=2009219, end=2014365):
    return [f for f in files if start <= get_date(f) <= end]

def clip_to_india(da, gdf, rename=True):
    if rename:
        da = da.rename({k: v for k, v in [("lat","y"),("lon","x")] if k in da.dims})
    da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
    da.rio.write_crs("EPSG:4326", inplace=True)
    return da.rio.clip(gdf.geometry, gdf.crs)

# =============================================================================
# LOAD DATA
# =============================================================================
print("Loading files ...")
f_mswx = filter_period(sorted(glob.glob(os.path.join(HR_MSWX, "*.nc"))))
f_ddpm = filter_period(sorted(glob.glob(os.path.join(HR_DDPM, "*_ddpm.nc"))))
print(f"  {len(f_mswx)} MSWX  |  {len(f_ddpm)} DDPM")

ds_m = xr.open_mfdataset(f_mswx, combine="nested", concat_dim="time")
ds_d = xr.open_mfdataset(f_ddpm, combine="nested", concat_dim="time")

if "time" not in ds_m.coords:
    times = [doy_to_dt(get_date(f)) for f in f_mswx]
    ds_m  = ds_m.assign_coords(time=("time", times))
    ds_d  = ds_d.assign_coords(time=("time", times))

da_m = ds_m["air_temperature"]
da_d = ds_d["air_temperature"]

gdf = gpd.read_file(INDIA_SHP)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# ── compute maps ──────────────────────────────────────────────────────────────
print("Computing maps ...")
bias_ts   = da_d - da_m
rmse_map  = np.sqrt((bias_ts**2).mean("time", skipna=True))
clim_ddpm = da_d.mean("time", skipna=True)
bias_map  = bias_ts.mean("time", skipna=True)

rmse_c = clip_to_india(rmse_map,  gdf)
mean_c = clip_to_india(clim_ddpm, gdf)
bias_c = clip_to_india(bias_map,  gdf)

rmse_vals = rmse_c.values.flatten(); rmse_vals = rmse_vals[np.isfinite(rmse_vals)]
mean_vals = mean_c.values.flatten(); mean_vals = mean_vals[np.isfinite(mean_vals)]
bias_vals = bias_c.values.flatten(); bias_vals = bias_vals[np.isfinite(bias_vals)]

elev_contour = None
if os.path.exists(ELEV_NC):
    try:
        ds_e = xr.open_dataset(ELEV_NC)
        elev_var = [v for v in ds_e.data_vars][0]
        elev_contour = ds_e[elev_var].interp(
            lat=rmse_map.y if "y" in rmse_map.dims else rmse_map.lat,
            lon=rmse_map.x if "x" in rmse_map.dims else rmse_map.lon,
            method="nearest")
        print("Elevation data loaded ✓")
    except Exception as e:
        print(f"Elevation load failed ({e}) — contours skipped")

# =============================================================================
# FIGURE LAYOUT - 2x2 Grid
# =============================================================================
print("Plotting ...")

FONT = "DejaVu Sans"
plt.rcParams.update({
    "font.family":       FONT,
    "axes.linewidth":    1.2,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "figure.dpi":        300, 
})

fig = plt.figure(figsize=(14, 12))

# 2x2 Grid setup
gs = GridSpec(2, 2, figure=fig, wspace=0.15, hspace=0.25, 
              left=0.06, right=0.96, top=0.92, bottom=0.08)

if HAS_CARTOPY:
    proj = ccrs.PlateCarree()
    ax1 = fig.add_subplot(gs[0, 0], projection=proj)
    ax2 = fig.add_subplot(gs[0, 1], projection=proj)
    ax3 = fig.add_subplot(gs[1, 0], projection=proj)
else:
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])

ax4 = fig.add_subplot(gs[1, 1]) 

# ── colour limits ─────────────────────────────────────────────────────────────
rmse_vmax = float(np.percentile(rmse_vals, 98))  
mean_vmin = float(np.percentile(mean_vals,  2))
mean_vmax = float(np.percentile(mean_vals, 98))
bias_abs  = max(abs(float(np.nanmin(bias_vals))), abs(float(np.nanmax(bias_vals))))
bias_norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-bias_abs, vmax=bias_abs)

def add_geography(ax, show_left_lat=False, show_bottom_lon=True):
    gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2,
                      transform=ccrs.PlateCarree() if HAS_CARTOPY else None,
                      zorder=5)
    if HAS_CARTOPY:
        ax.set_extent([67, 98, 6, 38], crs=ccrs.PlateCarree())
        gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), color='none')
        gl.top_labels = False
        gl.right_labels = False
        gl.left_labels = show_left_lat
        gl.bottom_labels = show_bottom_lon
        gl.xlocator = mticker.FixedLocator([70, 80, 90])
        gl.ylocator = mticker.FixedLocator([10, 20, 30])
        gl.xlabel_style = {'size': 16, 'color': '#222222'}
        gl.ylabel_style = {'size': 16, 'color': '#222222'}
    else:
        if not show_left_lat: ax.set_yticks([])
        if not show_bottom_lon: ax.set_xticks([])
        ax.tick_params(labelsize=16)
    ax.set_xlabel(""); ax.set_ylabel("")

def label_panel(ax, letter, subtitle, stats_str):
    ax.set_title(subtitle, fontsize=20, fontweight="bold", pad=12, color="#222222")
    ax.text(0.02, 0.98, f"({letter})", transform=ax.transAxes,
            fontsize=24, fontweight="bold", va="top", ha="left",
            path_effects=[pe.withStroke(linewidth=4, foreground="white")], zorder=10)
    
    # Text alignments update here: multi-line block
    ax.text(0.98, 0.03, stats_str, transform=ax.transAxes,
            fontsize=13, fontweight="semibold", va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="#888888", alpha=0.9, linewidth=1),
            zorder=10)

def add_regional_labels(ax):
    labels = [(77.5, 32.5, "Himalayas"), (79.0, 26.0, "IGP"),
              (75.5, 21.0, "Deccan\nPlateau"), (76.5, 11.5, "Western\nGhats"),
              (84.0, 21.0, "Eastern\nGhats")]
    for lon_, lat_, txt in labels:
        ax.text(lon_, lat_, txt, fontsize=14, color="#111111", ha="center", va="center", 
                style="italic", fontweight="bold",
                path_effects=[pe.withStroke(linewidth=3, foreground="white")],
                transform=ccrs.PlateCarree() if HAS_CARTOPY else ax.transData, zorder=6)

# =============================================================================
# PANEL (a) — Temporal RMSE
# =============================================================================
im1 = rmse_c.plot(ax=ax1, cmap=CMAP_RMSE, vmin=0, vmax=rmse_vmax, add_colorbar=False,
                  transform=ccrs.PlateCarree() if HAS_CARTOPY else None)
add_geography(ax1, show_left_lat=True, show_bottom_lon=False) 
add_regional_labels(ax1)

# Changed to 3 lines
label_panel(ax1, "a", "Temporal RMSE",
            f"Mean: {np.mean(rmse_vals):.2f} °C\n"
            f"P75: {np.percentile(rmse_vals,75):.2f} °C\n"
            f"P95: {np.percentile(rmse_vals,95):.2f} °C")

cbar1 = fig.colorbar(im1, ax=ax1, orientation="horizontal", fraction=0.045, pad=0.04, extend="max", shrink=0.9)
cbar1.set_label("RMSE (°C)", fontsize=16, labelpad=8, fontweight="bold")
cbar1.ax.tick_params(labelsize=14)

# =============================================================================
# PANEL (b) — DDPM Climatological Mean
# =============================================================================
im2 = mean_c.plot(ax=ax2, cmap=CMAP_TEMP, vmin=mean_vmin, vmax=mean_vmax, add_colorbar=False,
                  transform=ccrs.PlateCarree() if HAS_CARTOPY else None)
add_geography(ax2, show_left_lat=False, show_bottom_lon=False)

# Changed to 3 lines
label_panel(ax2, "b", "DDPM Climatology",
            f"Mean: {np.mean(mean_vals):.2f} °C\n"
            f"Min: {np.min(mean_vals):.2f} °C\n"
            f"Max: {np.max(mean_vals):.2f} °C")

cbar2 = fig.colorbar(im2, ax=ax2, orientation="horizontal", fraction=0.045, pad=0.04, extend="both", shrink=0.9)
cbar2.set_label("Temperature (°C)", fontsize=16, labelpad=8, fontweight="bold")
cbar2.ax.tick_params(labelsize=14)

# =============================================================================
# PANEL (c) — Mean Bias + elevation contours
# =============================================================================
im3 = bias_c.plot(ax=ax3, cmap=CMAP_BIAS, norm=bias_norm, add_colorbar=False,
                  transform=ccrs.PlateCarree() if HAS_CARTOPY else None)

if elev_contour is not None:
    try:
        elev_lons = elev_contour.lon.values if "lon" in elev_contour.dims else elev_contour.x.values
        elev_lats = elev_contour.lat.values if "lat" in elev_contour.dims else elev_contour.y.values
        EL, ELA = np.meshgrid(elev_lons, elev_lats)
        elev_data = elev_contour.values
        if elev_data.ndim == 3: elev_data = elev_data[0]
        ax3.contour(EL, ELA, elev_data, levels=[500, 1500, 3000],
                    colors=["#555555", "#333333", "#111111"], linewidths=[0.8, 1.2, 1.6],
                    linestyles=["--", "-.", "-"], alpha=0.7,
                    transform=ccrs.PlateCarree() if HAS_CARTOPY else None, zorder=5)
        from matplotlib.lines import Line2D
        handles = [Line2D([0],[0], color=c, lw=lw, ls=ls, label=f"{h} m")
                   for h, c, lw, ls in zip([500, 1500, 3000], ["#555555","#333333","#111111"], [0.8, 1.2, 1.6], ["--", "-.", "-"])]
        ax3.legend(handles=handles, loc="lower left", fontsize=12, framealpha=0.9, edgecolor="#aaaaaa", handlelength=2.5)
    except Exception as e:
        print(f"Elevation contours failed: {e}")

add_geography(ax3, show_left_lat=True, show_bottom_lon=True)

# Changed to 4 lines
label_panel(ax3, "c", "Mean Bias (DDPM − MSWX)",
            f"MB: {np.nanmean(bias_vals):.3f} °C\n"
            f"Std: {np.nanstd(bias_vals):.3f} °C\n"
            f"P05: {np.percentile(bias_vals,5):.2f} °C\n"
            f"P95: {np.percentile(bias_vals,95):.2f} °C")

cbar3 = fig.colorbar(im3, ax=ax3, orientation="horizontal", fraction=0.045, pad=0.07, shrink=0.9)
cbar3.set_label("Bias (°C)", fontsize=16, labelpad=8, fontweight="bold")
cbar3.ax.tick_params(labelsize=14)
cbar3.ax.axvline(0, color="k", linewidth=1.5)

# =============================================================================
# PANEL (d) — Bias PDF
# =============================================================================
ax4.set_facecolor("#f9f9f9")
for spine in ax4.spines.values():
    spine.set_linewidth(1.0); spine.set_color("#888888")

n_bins = 60
counts, edges, patches = ax4.hist(bias_vals, bins=n_bins, orientation="horizontal",
                                  color="#4a90c4", alpha=0.75, edgecolor="white", linewidth=0.8, density=True, zorder=3)

mu, sigma = scipy_norm.fit(bias_vals)
y_range = np.linspace(bias_vals.min(), bias_vals.max(), 300)
ax4.plot(scipy_norm.pdf(y_range, mu, sigma), y_range, color="#c0392b", linewidth=3.0, label=f"N({mu:.2f}, {sigma:.2f}²)", zorder=4)

ax4.axhline(0, color="#111111", linewidth=1.5, linestyle="--", alpha=0.8, zorder=5)

for pct, ls_ in [(5,"dotted"), (95,"dotted")]:
    val = np.percentile(bias_vals, pct)
    ax4.axhline(val, color="#666666", linewidth=1.5, linestyle=ls_, zorder=4)
    ax4.text(ax4.get_xlim()[1]*0.95, val, f"P{pct}", ha="right", va="bottom", fontsize=14, color="#444444", fontweight="bold")

ax4.set_xlabel("Density", fontsize=16, labelpad=8, fontweight="bold")
ax4.set_ylabel("Bias (°C)", fontsize=16, labelpad=8, fontweight="bold")
ax4.set_title("(d) Bias Distribution", fontsize=20, fontweight="bold", color="#222222", pad=12)
ax4.tick_params(labelsize=14)
ax4.legend(fontsize=14, loc="upper right", framealpha=0.9, edgecolor="#cccccc", borderpad=0.6)
ax4.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

# =============================================================================
# SAVE
# =============================================================================
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
fig.savefig(SAVE_PATH, dpi=300, bbox_inches="tight", facecolor="white", edgecolor="none")
print(f"\nFigure saved → {SAVE_PATH}")
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
Simple U-Net Evaluation Script — India Shapefile Masked
=========================================================
Corrected version: metrics computed on India land pixels only
(31,174 pixels defined by IMD boundary shapefile), matching
the masking applied to ResU-Net and DDPM evaluations.

Key difference from original script:
  - India shapefile mask applied before ALL metric computations
  - Ocean and out-of-boundary pixels are excluded
  - Results are directly comparable to ResU-Net (0.818°C) and DDPM (0.315°C)

Dependencies:
    pip install geopandas rasterio shapely --break-system-packages
    pip install scikit-image scipy scikit-learn tqdm xarray --break-system-packages
"""

import os
import json
import warnings
from datetime import datetime
from typing import Tuple, List, Optional, Dict

import numpy as np
import xarray as xr
from tqdm import tqdm
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance, skew, kurtosis
from scipy.spatial import cKDTree
from sklearn.metrics import mean_squared_error, mean_absolute_error
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import geometry_mask

warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION — update paths if needed
# =============================================================================

class Config:
    MODEL_PATH  = "<SET YOUR PATH HERE>"
    HR_DIR      = "<SET YOUR PATH HERE>"
    LR_DIR      = "<SET YOUR PATH HERE>"
    SHAPEFILE   = "<SET YOUR PATH HERE>"
    RESULTS_DIR = "<SET YOUR PATH HERE>"

    # Must match training configuration
    TRAIN_FRACTION = 0.70
    VAL_FRACTION   = 0.15
    VAR_NAME       = "air_temperature"

    # Patch / blending — must match original evaluation
    SIGMA_FACTOR = 0.3
    PATCH_SIZE   = 128
    STRIDE       = 64
    NORM_EPSILON = 1e-3


cfg = Config()

# Verify all paths before starting
for path, label in [
    (cfg.MODEL_PATH, "U-Net model"),
    (cfg.HR_DIR,     "HR data directory"),
    (cfg.LR_DIR,     "LR data directory"),
    (cfg.SHAPEFILE,  "India shapefile"),
]:
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {status}  {label}: {path}")


# =============================================================================
# INDIA MASK — built once, reused for every sample
# =============================================================================

def build_india_mask(shapefile_path: str, hr_dir: str, var_name: str) -> np.ndarray:
    """
    Rasterize the India boundary shapefile onto the HR grid (500×500).
    Returns a boolean array: True = India land pixel.

    The rasterization uses the lat/lon extent of the first HR file so the
    mask is perfectly aligned with every HR field.
    """
    # Load one HR file to get grid extent and shape
    sample_file = sorted(
        [os.path.join(hr_dir, f) for f in os.listdir(hr_dir) if f.endswith(".nc")]
    )[0]

    with xr.open_dataset(sample_file) as ds:
        da  = ds[var_name]
        if "time" in da.dims:
            da = da.isel(time=0)
        lat = da.lat.values   # descending (37.65 → 8.45)
        lon = da.lon.values   # ascending  (68.75 → 97.15)
        H, W = len(lat), len(lon)

    # rasterio uses (west, south, east, north)
    west, east = float(lon.min()), float(lon.max())
    south, north = float(lat.min()), float(lat.max())

    # Half-pixel buffer so the transform covers the full cell extent
    d_lat = abs(lat[1] - lat[0])
    d_lon = abs(lon[1] - lon[0])

    transform = from_bounds(
        west  - d_lon / 2,
        south - d_lat / 2,
        east  + d_lon / 2,
        north + d_lat / 2,
        W, H
    )

    # Load shapefile and reproject to WGS84 if needed
    gdf = gpd.read_file(shapefile_path)
    if gdf.crs is not None and str(gdf.crs) != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")

    geoms = list(gdf.geometry)

    # geometry_mask returns True where geometry is ABSENT → invert
    mask_2d = ~geometry_mask(
        geoms,
        transform=transform,
        invert=False,
        out_shape=(H, W),
    )

    # The HR grid has lat descending; rasterio rasterises north→south by default
    # Verify orientation by checking a known land pixel (e.g. ~20°N 80°E)
    lat_idx = np.argmin(np.abs(lat - 20.0))
    lon_idx = np.argmin(np.abs(lon - 80.0))
    if not mask_2d[lat_idx, lon_idx]:
        print("  ⚠ Mask orientation check failed — flipping rows")
        mask_2d = mask_2d[::-1, :]

    n_land = int(mask_2d.sum())
    print(f"  India mask built: {n_land} land pixels out of {H}×{W} = {H*W} total")
    return mask_2d


# =============================================================================
# UTILITY FUNCTIONS (unchanged from original)
# =============================================================================

def nearest_fill_nan(arr: np.ndarray) -> np.ndarray:
    if np.isnan(arr).sum() == 0:
        return arr
    valid = ~np.isnan(arr)
    coords = np.array(np.nonzero(valid)).T
    if coords.size == 0:
        return np.zeros_like(arr)
    grid_coords = np.array(np.nonzero(np.ones_like(arr))).T
    tree = cKDTree(coords)
    _, idx = tree.query(grid_coords, k=1)
    mapped_coords = coords[idx]
    filled = np.zeros_like(arr)
    filled.flat[:] = arr[tuple(mapped_coords.T)]
    return filled


def safe_load(path: str, var_name: str) -> Optional[np.ndarray]:
    try:
        with xr.open_dataset(path) as ds:
            return np.squeeze(ds[var_name].values).astype(np.float32)
    except Exception as e:
        print(f"  Failed to load {path}: {e}")
        return None


def list_pairs(hr_dir: str, lr_dir: str) -> List[Tuple[str, str]]:
    import re
    def _key(f):
        return [int(t) if t.isdigit() else t.lower()
                for t in re.split(r"([0-9]+)", f)]
    hr_files = sorted(
        [f for f in os.listdir(hr_dir) if f.endswith(".nc")], key=_key
    )
    lr_set = set(os.listdir(lr_dir))
    pairs = []
    for hr in hr_files:
        base = os.path.splitext(hr)[0]
        lr   = f"{base}_1deg_to_hr.nc"
        if lr in lr_set:
            pairs.append((os.path.join(hr_dir, hr), os.path.join(lr_dir, lr)))
    return pairs


def split_pairs(pairs, train_frac=0.70, val_frac=0.15):
    n       = len(pairs)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    return pairs[:n_train], pairs[n_train:n_train + n_val], pairs[n_train + n_val:]


# =============================================================================
# NORMALIZER (unchanged from original)
# =============================================================================

class EfficientNormalizer:
    def __init__(self, mean_map, std_map):
        self.mean_map  = mean_map
        self.scale_map = np.where(std_map < cfg.NORM_EPSILON, cfg.NORM_EPSILON, std_map)

    def normalize_patch(self, patch, top, left, ps):
        return ((patch - self.mean_map[top:top+ps, left:left+ps])
                / self.scale_map[top:top+ps, left:left+ps]).astype(np.float32)

    def denormalize_patch(self, patch, top, left, ps):
        return (patch * self.scale_map[top:top+ps, left:left+ps]
                + self.mean_map[top:top+ps, left:left+ps]).astype(np.float32)


# =============================================================================
# PATCH PROCESSING (unchanged from original)
# =============================================================================

def extract_patches(img, patch_size=128, stride=64):
    h, w = img.shape
    patches, positions = [], []
    for top in range(0, h - patch_size + 1, stride):
        for left in range(0, w - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (w - patch_size) % stride != 0:
        left = w - patch_size
        for top in range(0, h - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (h - patch_size) % stride != 0:
        top = h - patch_size
        for left in range(0, w - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (h - patch_size) % stride != 0 and (w - patch_size) % stride != 0:
        patches.append(img[h-patch_size:h, w-patch_size:w])
        positions.append((h-patch_size, w-patch_size))
    return patches, positions


def gaussian_weights(patch_size, sigma_factor=0.3):
    c = patch_size // 2
    y, x = np.ogrid[:patch_size, :patch_size]
    sig  = sigma_factor * patch_size
    w    = np.exp(-((x-c)**2 + (y-c)**2) / (2*sig**2))
    return (w / w.max()).astype(np.float32)


def stitch_gaussian(patches, positions, full_shape, patch_size=128, sigma_factor=0.3):
    h, w = full_shape
    out  = np.zeros((h, w), np.float32)
    wsum = np.zeros((h, w), np.float32)
    gw   = gaussian_weights(patch_size, sigma_factor)
    for patch, (top, left) in zip(patches, positions):
        ph, pw = patch.shape
        wc = gw[:ph, :pw]
        out[top:top+ph, left:left+pw]  += patch * wc
        wsum[top:top+ph, left:left+pw] += wc
    return out / np.where(wsum < 1e-8, 1.0, wsum)


def predict_image(model, lr_image, normalizer, patch_size=128, stride=64, sigma_factor=0.3):
    import tensorflow as tf
    patches_lr, positions = extract_patches(lr_image, patch_size, stride)
    pred_patches = []
    for lr_patch, (top, left) in zip(patches_lr, positions):
        lr_norm  = np.clip(normalizer.normalize_patch(lr_patch, top, left, patch_size), -5, 5)
        lr_input = lr_norm[np.newaxis, :, :, np.newaxis]
        hr_norm  = np.clip(model.predict(lr_input, verbose=0).squeeze(), -5, 5)
        pred_patches.append(hr_norm)
    sr_norm = stitch_gaussian(pred_patches, positions, lr_image.shape,
                              patch_size=patch_size, sigma_factor=sigma_factor)
    h, w    = lr_image.shape
    result  = sr_norm * normalizer.scale_map[:h, :w] + normalizer.mean_map[:h, :w]
    return result


# =============================================================================
# METRICS — applied to India land pixels only
# =============================================================================

def compute_metrics(pred: np.ndarray, target: np.ndarray,
                    india_mask: np.ndarray) -> Dict[str, float]:
    """
    Compute all metrics using ONLY India land pixels.
    india_mask: boolean array, True = land pixel to include.
    """
    m = {}

    # --- apply mask ---
    pred_v   = pred[india_mask].astype(np.float64)
    target_v = target[india_mask].astype(np.float64)

    # Remove any residual NaN (should be none after nearest_fill_nan)
    valid    = np.isfinite(pred_v) & np.isfinite(target_v)
    pred_v   = pred_v[valid]
    target_v = target_v[valid]

    if len(pred_v) == 0:
        return {}

    # --- point-wise accuracy ---
    diff     = pred_v - target_v
    m["rmse"] = float(np.sqrt(np.mean(diff**2)))
    m["mae"]  = float(np.mean(np.abs(diff)))
    m["bias"] = float(np.mean(diff))

    # --- correlation / R² ---
    r, _       = pearsonr(pred_v, target_v)
    m["pearson_r"] = float(r)
    m["r2"]        = float(r**2)

    # --- KGE ---
    mu_s  = np.mean(pred_v);  mu_o  = np.mean(target_v)
    sig_s = np.std(pred_v);   sig_o = np.std(target_v)
    kge   = 1 - np.sqrt((r-1)**2 + (sig_s/sig_o - 1)**2 + (mu_s/mu_o - 1)**2)
    m["kge"] = float(kge)

    # --- Willmott d ---
    num = np.sum(diff**2)
    den = np.sum((np.abs(pred_v - np.mean(target_v)) + np.abs(target_v - np.mean(target_v)))**2)
    m["willmott_d"] = float(1 - num/den) if den > 0 else np.nan

    # --- variance / sigma ratio ---
    m["sigma_ratio"] = float(sig_s / sig_o) if sig_o > 0 else np.nan

    # --- distributional fidelity ---
    ks_stat, _ = ks_2samp(pred_v, target_v)
    m["ks_statistic"]    = float(ks_stat)
    m["wasserstein_1"]   = float(wasserstein_distance(pred_v, target_v))
    m["skewness_diff"]   = float(skew(pred_v) - skew(target_v))

    # --- percentile errors ---
    for p in [5, 25, 50, 75, 95]:
        m[f"p{p}_error"] = float(np.percentile(pred_v, p) - np.percentile(target_v, p))

    # --- SSIM / PSNR on masked 2-D field ---
    # Use full 2D arrays (masked region); normalise to [0,1] for skimage
    try:
        p2d  = pred.copy();   p2d[~india_mask]  = np.nan
        t2d  = target.copy(); t2d[~india_mask]  = np.nan
        vmin = np.nanmin(target); vmax = np.nanmax(target); rng = vmax - vmin
        if rng > 0:
            p_n = (p2d - vmin) / rng
            t_n = (t2d - vmin) / rng
            p_n = np.nan_to_num(p_n, nan=0.0)
            t_n = np.nan_to_num(t_n, nan=0.0)
            m["ssim"] = float(ssim(t_n, p_n, data_range=1.0))
            m["psnr"] = float(psnr(t_n, p_n, data_range=1.0))
    except Exception:
        m["ssim"] = np.nan
        m["psnr"] = np.nan

    # --- spatial texture: Sobel gradient ---
    try:
        from scipy.ndimage import sobel
        def grad_mag(arr):
            gx = sobel(np.nan_to_num(arr, nan=0.0), axis=1)
            gy = sobel(np.nan_to_num(arr, nan=0.0), axis=0)
            return np.sqrt(gx**2 + gy**2)

        gp = grad_mag(pred)[india_mask]
        gt = grad_mag(target)[india_mask]

        m["gradient_rmse"]    = float(np.sqrt(np.mean((gp - gt)**2)))
        m["roughness_ratio"]  = float(np.mean(gp) / np.mean(gt)) if np.mean(gt) > 0 else np.nan
        m["texture_corr"], _  = pearsonr(gp, gt)
        m["texture_corr"]     = float(m["texture_corr"])
    except Exception:
        m["gradient_rmse"]   = np.nan
        m["roughness_ratio"] = np.nan
        m["texture_corr"]    = np.nan

    # --- MaxAE ---
    m["max_ae"] = float(np.max(np.abs(diff)))

    return m


# =============================================================================
# MAIN
# =============================================================================

def main():
    import re
    import tensorflow as tf

    print("\n" + "="*70)
    print("U-NET EVALUATION  —  INDIA SHAPEFILE MASKED")
    print("="*70)

    # ── build India mask ──────────────────────────────────────────────────
    print("\n[1] Building India land mask ...")
    india_mask = build_india_mask(cfg.SHAPEFILE, cfg.HR_DIR, cfg.VAR_NAME)

    # ── file pairs ────────────────────────────────────────────────────────
    print("\n[2] Loading file pairs ...")
    all_pairs   = list_pairs(cfg.HR_DIR, cfg.LR_DIR)
    _, _, test_pairs = split_pairs(all_pairs, cfg.TRAIN_FRACTION, cfg.VAL_FRACTION)
    print(f"    Total pairs: {len(all_pairs)} | Test pairs: {len(test_pairs)}")

    if not test_pairs:
        raise RuntimeError("No test pairs found — check HR_DIR / LR_DIR paths")

    # ── climatology & normalizer ──────────────────────────────────────────
    print("\n[3] Loading climatology maps ...")
    clim_path = os.path.join(cfg.RESULTS_DIR, "climatology_maps.nc")
    with xr.open_dataset(clim_path) as ds:
        mean_map = ds["mean_map"].values
        std_map  = ds["std_map"].values
    normalizer = EfficientNormalizer(mean_map, std_map)
    print("    Climatology loaded ✓")

    # ── model ─────────────────────────────────────────────────────────────
    print(f"\n[4] Loading U-Net model ...")
    model = tf.keras.models.load_model(cfg.MODEL_PATH)
    print("    Model loaded ✓")

    m = re.search(r"epoch_(\d+)-val_loss_([\d.]+)\.keras", cfg.MODEL_PATH)
    epoch    = int(m.group(1))   if m else "?"
    val_loss = float(m.group(2)) if m else "?"
    print(f"    Epoch: {epoch}  |  Val loss: {val_loss}")

    # ── evaluate ──────────────────────────────────────────────────────────
    print(f"\n[5] Evaluating {len(test_pairs)} test samples (masked to India) ...")
    all_metrics = []

    for hr_path, lr_path in tqdm(test_pairs, desc="U-Net inference"):
        hr = safe_load(hr_path, cfg.VAR_NAME)
        lr = safe_load(lr_path, cfg.VAR_NAME)
        if hr is None or lr is None:
            continue

        hr = nearest_fill_nan(hr)
        lr = nearest_fill_nan(lr)

        pred = predict_image(model, lr, normalizer,
                             patch_size=cfg.PATCH_SIZE,
                             stride=cfg.STRIDE,
                             sigma_factor=cfg.SIGMA_FACTOR)

        m_day = compute_metrics(pred, hr, india_mask)
        if m_day:
            all_metrics.append(m_day)

    n = len(all_metrics)
    print(f"\n    Processed: {n} / {len(test_pairs)} samples")

    # ── aggregate ─────────────────────────────────────────────────────────
    print("\n[6] Aggregating metrics ...")
    agg = {}
    all_keys = sorted({k for m in all_metrics for k in m.keys()})
    for key in all_keys:
        vals = [m[key] for m in all_metrics
                if key in m and m[key] is not None and np.isfinite(m[key])]
        if vals:
            agg[f"{key}_mean"] = float(np.mean(vals))
            agg[f"{key}_std"]  = float(np.std(vals))
            agg[f"{key}_min"]  = float(np.min(vals))
            agg[f"{key}_max"]  = float(np.max(vals))

    # ── print summary ──────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("RESULTS  —  India land pixels only")
    print("="*70)
    print(f"{'Metric':<22}  {'Mean':>10}  {'Std':>10}")
    print("-"*46)
    key_order = [
        ("rmse",          "RMSE (°C)"),
        ("mae",           "MAE (°C)"),
        ("bias",          "Mean Bias (°C)"),
        ("max_ae",        "MaxAE (°C)"),
        ("psnr",          "PSNR (dB)"),
        ("ssim",          "SSIM"),
        ("pearson_r",     "Pearson r"),
        ("r2",            "R²"),
        ("kge",           "KGE"),
        ("willmott_d",    "Willmott d"),
        ("sigma_ratio",   "σ ratio"),
        ("skewness_diff", "Skewness Δ"),
        ("p5_error",      "P05 error (°C)"),
        ("p95_error",     "P95 error (°C)"),
        ("ks_statistic",  "KS statistic"),
        ("wasserstein_1", "Wasserstein-1 (°C)"),
        ("roughness_ratio","Roughness ratio"),
        ("texture_corr",  "Texture corr."),
        ("gradient_rmse", "Gradient RMSE (°C/px)"),
    ]
    for key, label in key_order:
        mk = f"{key}_mean"
        sk = f"{key}_std"
        if mk in agg:
            print(f"  {label:<22}  {agg[mk]:>10.4f}  {agg.get(sk, 0):>10.4f}")

    print(f"\n  India land pixels : {int(india_mask.sum())}")
    print(f"  Test samples      : {n}")

    # ── save ───────────────────────────────────────────────────────────────
    out_path = os.path.join(cfg.RESULTS_DIR, "unet_masked_evaluation_results.json")
    save_obj = {
        "timestamp":       datetime.now().isoformat(),
        "model_type":      "Simple U-Net (India-masked)",
        "model_path":      cfg.MODEL_PATH,
        "epoch":           epoch,
        "val_loss":        val_loss,
        "n_samples":       n,
        "n_land_pixels":   int(india_mask.sum()),
        "masking":         "India IMD boundary shapefile",
        "shapefile":       cfg.SHAPEFILE,
        "aggregated":      agg,
        "individual":      all_metrics,
    }
    with open(out_path, "w") as f:
        json.dump(save_obj, f, indent=2)

    print(f"\n  Results saved → {out_path}")
    print("="*70)
    print("✅ DONE")
    print("="*70)

    return agg


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive Evaluation Metrics: U-Net vs ResU-Net vs DDPM
=============================================================
All metrics computed exclusively over India land pixels using the official
India political boundary shapefile — identical mask for all three models.

Column order (weakest → strongest): U-Net | ResU-Net | DDPM

U-Net   : run on-the-fly inference (no saved downscaled files)
ResU-Net: loaded from pre-saved .nc files
DDPM    : loaded from pre-saved .nc files

India mask uses 0.05° buffer (matches original metrics_comprehensive.py)
→ all three models evaluated on exactly the same pixel set.

Outputs:
  metrics_table_3model.csv   — machine-readable
  metrics_table_3model.tex   — LaTeX booktabs, camera-ready
  metrics_summary_3model.txt — plain-text inspection
"""

# ── standard ────────────────────────────────────────────────────────────────
import os, gc, warnings
import glob
import json
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
from datetime import datetime
warnings.filterwarnings("ignore")

# ── geo / stats ──────────────────────────────────────────────────────────────
import geopandas as gpd
import rasterio.features
import rasterio.transform
from shapely.geometry import mapping
from scipy.stats import ks_2samp, wasserstein_distance, skew as sp_skew
from scipy.ndimage import sobel
from scipy.spatial import cKDTree

# ── image quality ─────────────────────────────────────────────────────────────
from skimage.metrics import structural_similarity as ssim_func
from skimage.metrics import peak_signal_noise_ratio as psnr_func

# ── deep learning (U-Net inference) ──────────────────────────────────────────
import tensorflow as tf


# =============================================================================
# PATHS  — edit if layout differs
# =============================================================================
SHAPEFILE    = "<SET YOUR PATH HERE>"
HR_TEMPLATE  = "<SET YOUR PATH HERE>"

DIR_MSWX     = "<SET YOUR PATH HERE>"
DIR_LR_UNET  = "<SET YOUR PATH HERE>"
DIR_RESUNET  = "<SET YOUR PATH HERE>"
DIR_DDPM     = "<SET YOUR PATH HERE>"

UNET_MODEL   = "<SET YOUR PATH HERE>"
UNET_CLIM    = "<SET YOUR PATH HERE>"

OUTPUT_DIR   = "<SET YOUR PATH HERE>"
VAR          = "air_temperature"

# U-Net patch config — must match original training/evaluation
PATCH_SIZE   = 128
STRIDE       = 64
SIGMA_FACTOR = 0.3
NORM_EPS     = 1e-3

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# INDIA MASK  (identical function to original metrics_comprehensive.py)
# =============================================================================

def build_india_mask(shapefile_path, lat, lon):
    gdf  = gpd.read_file(shapefile_path)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf  = gdf.to_crs(epsg=4326)

    geom = gdf.dissolve().geometry.values[0].buffer(0.05)

    nrows, ncols = len(lat), len(lon)
    lon_res = (lon[-1] - lon[0]) / (ncols - 1)
    lat_res = (lat[-1] - lat[0]) / (nrows - 1)

    west  = lon.min() - lon_res / 2
    north = lat.max() + abs(lat_res) / 2
    tf    = rasterio.transform.from_origin(west, north,
                                            abs(lon_res), abs(lat_res))

    outside = rasterio.features.geometry_mask(
        [mapping(geom)], out_shape=(nrows, ncols),
        transform=tf, invert=False, all_touched=True)

    mask = ~outside
    if lat[0] < lat[-1]:          # ascending lat — flip to match rasterio
        mask = mask[::-1, :]

    n = int(mask.sum())
    print(f"  India mask: {n:,} pixels  "
          f"({100*n/(nrows*ncols):.1f}% of {nrows}×{ncols} domain)")
    return mask


def get_india_bbox(mask):
    rows, cols = np.where(mask)
    return (slice(rows.min(), rows.max() + 1),
            slice(cols.min(), cols.max() + 1))


# =============================================================================
# FILE MAP HELPERS
# =============================================================================

def build_file_map(directory, suffix_strip=None):
    """
    stem  →  filepath.
    suffix_strip: e.g. "_ddpm" strips that suffix before using as key.
    """
    result = {}
    for f in glob.glob(os.path.join(directory, "*.nc")):
        stem = os.path.basename(f).rsplit(".", 1)[0]
        if suffix_strip and stem.endswith(suffix_strip):
            stem = stem[: -len(suffix_strip)]
        result[stem] = f
    return result


def build_lr_map_unet(lr_dir):
    """
    LR files are named  {YYYYDOY}_1deg_to_hr.nc.
    Key = YYYYDOY string (matches MSWX HR stem).
    """
    result = {}
    for f in glob.glob(os.path.join(lr_dir, "*_1deg_to_hr.nc")):
        stem = os.path.basename(f).replace("_1deg_to_hr.nc", "")
        result[stem] = f
    return result


# =============================================================================
# FIELD EXTRACTION
# =============================================================================

def extract(ds, var=VAR):
    v = ds[var].values
    if v.ndim == 3:
        v = v[0]
    if np.nanmean(v) > 200:
        v = v - 273.15
    return v.astype(np.float32)


# =============================================================================
# U-NET INFERENCE UTILITIES
# =============================================================================

def nearest_fill_nan(arr):
    if not np.isnan(arr).any():
        return arr
    valid  = ~np.isnan(arr)
    coords = np.array(np.nonzero(valid)).T
    if coords.size == 0:
        return np.zeros_like(arr)
    gc_    = np.array(np.nonzero(np.ones_like(arr))).T
    tree   = cKDTree(coords)
    _, idx = tree.query(gc_, k=1)
    filled = np.zeros_like(arr)
    filled.flat[:] = arr[tuple(coords[idx].T)]
    return filled


class Normalizer:
    def __init__(self, mean_map, std_map):
        self.mean  = mean_map
        self.scale = np.where(std_map < NORM_EPS, NORM_EPS, std_map)

    def norm(self, patch, top, left, ps):
        return ((patch - self.mean[top:top+ps, left:left+ps])
                / self.scale[top:top+ps, left:left+ps]).astype(np.float32)


def extract_patches(img, patch_size=128, stride=64):
    h, w = img.shape
    patches, positions = [], []
    for top in range(0, h - patch_size + 1, stride):
        for left in range(0, w - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (w - patch_size) % stride != 0:
        left = w - patch_size
        for top in range(0, h - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (h - patch_size) % stride != 0:
        top = h - patch_size
        for left in range(0, w - patch_size + 1, stride):
            patches.append(img[top:top+patch_size, left:left+patch_size])
            positions.append((top, left))
    if (h - patch_size) % stride != 0 and (w - patch_size) % stride != 0:
        patches.append(img[h-patch_size:h, w-patch_size:w])
        positions.append((h-patch_size, w-patch_size))
    return patches, positions


def gaussian_weights(ps, sf=0.3):
    c = ps // 2
    y, x = np.ogrid[:ps, :ps]
    sig  = sf * ps
    w    = np.exp(-((x-c)**2 + (y-c)**2) / (2*sig**2))
    return (w / w.max()).astype(np.float32)


def stitch_gaussian(patches, positions, shape, ps=128, sf=0.3):
    h, w = shape
    out  = np.zeros((h, w), np.float32)
    wsum = np.zeros((h, w), np.float32)
    gw   = gaussian_weights(ps, sf)
    for patch, (top, left) in zip(patches, positions):
        ph, pw = patch.shape
        wc = gw[:ph, :pw]
        out[top:top+ph, left:left+pw]  += patch * wc
        wsum[top:top+ph, left:left+pw] += wc
    return out / np.where(wsum < 1e-8, 1.0, wsum)


def unet_predict(model, lr_image, normalizer):
    """Full-image inference with Gaussian patch blending."""
    patches_lr, positions = extract_patches(lr_image, PATCH_SIZE, STRIDE)
    pred_patches = []
    for lr_p, (top, left) in zip(patches_lr, positions):
        norm_p = np.clip(normalizer.norm(lr_p, top, left, PATCH_SIZE), -5, 5)
        inp    = norm_p[np.newaxis, :, :, np.newaxis]
        hr_n   = np.clip(model.predict(inp, verbose=0).squeeze(), -5, 5)
        pred_patches.append(hr_n)

    sr_norm = stitch_gaussian(pred_patches, positions,
                              lr_image.shape, PATCH_SIZE, SIGMA_FACTOR)
    h, w    = lr_image.shape
    # Denormalize using full-field climatology
    return (sr_norm * normalizer.scale[:h, :w]
            + normalizer.mean[:h, :w]).astype(np.float32)


# =============================================================================
# METRIC FUNCTIONS  (all operate on India-masked 1-D vectors)
# =============================================================================

def pointwise_metrics(rv, pv):
    diff = pv - rv
    return {
        "RMSE":      float(np.sqrt(np.mean(diff**2))),
        "MAE":       float(np.mean(np.abs(diff))),
        "Mean Bias": float(np.mean(diff)),
        "MaxAE":     float(np.max(np.abs(diff))),
    }


def psnr_metric(rv, pv):
    mse = np.mean((pv - rv)**2)
    if mse == 0:
        return np.inf
    return float(10.0 * np.log10(np.ptp(rv)**2 / mse))


def calc_ssim(ref_2d, pred_2d, mask_crop):
    """SSIM over India bounding-box crop; non-India pixels set to India mean."""
    im_ref  = ref_2d.copy();  im_ref[~mask_crop]  = float(np.nanmean(ref_2d[mask_crop]))
    im_pred = pred_2d.copy(); im_pred[~mask_crop] = float(np.nanmean(pred_2d[mask_crop]))
    dr = float(np.nanmax(ref_2d[mask_crop]) - np.nanmin(ref_2d[mask_crop]))
    return float(ssim_func(im_ref, im_pred, data_range=dr))


def correlation_and_skill(rv, pv):
    r     = float(np.corrcoef(rv, pv)[0, 1])
    ss_res = np.sum((pv - rv)**2)
    ss_tot = np.sum((rv - rv.mean())**2)
    r2    = float(1.0 - ss_res / (ss_tot + 1e-12))
    alpha = np.std(pv) / (np.std(rv) + 1e-12)
    beta  = np.mean(pv) / (np.mean(rv) + 1e-12)
    kge   = float(1.0 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2))
    num   = np.sum((pv - rv)**2)
    den   = np.sum((np.abs(pv - rv.mean()) + np.abs(rv - rv.mean()))**2)
    d     = float(1.0 - num / (den + 1e-12))
    return {"Pearson r": r, "R2": r2, "KGE": kge, "Willmott d": d}


def distributional_metrics(rv, pv):
    ks_stat, _ = ks_2samp(rv, pv)
    return {
        "sigma ratio":      float(np.std(pv) / (np.std(rv) + 1e-12)),
        "Skewness delta":   float(sp_skew(pv) - sp_skew(rv)),
        "P05 error (degC)": float(np.percentile(pv, 5)  - np.percentile(rv, 5)),
        "P95 error (degC)": float(np.percentile(pv, 95) - np.percentile(rv, 95)),
        "KS statistic":     float(ks_stat),
        "Wasserstein-1":    float(wasserstein_distance(rv, pv)),
    }


def texture_metrics(ref_2d, pred_2d, mask_crop):
    """Sobel gradient metrics over India bounding-box crop."""
    ir = ref_2d.copy();  ir[~mask_crop] = float(np.nanmean(ref_2d[mask_crop]))
    ip = pred_2d.copy(); ip[~mask_crop] = float(np.nanmean(pred_2d[mask_crop]))
    gr = np.sqrt(sobel(ir, axis=0)**2 + sobel(ir, axis=1)**2)
    gp = np.sqrt(sobel(ip, axis=0)**2 + sobel(ip, axis=1)**2)
    gr_f = gr[mask_crop]; gp_f = gp[mask_crop]
    return {
        "Roughness ratio": float(np.mean(gp_f) / (np.mean(gr_f) + 1e-12)),
        "Texture corr.":   float(np.corrcoef(gr_f, gp_f)[0, 1]),
        "Gradient RMSE":   float(np.sqrt(np.mean((gp_f - gr_f)**2))),
    }


def evaluate_day(ref_2d, pred_2d, india_flat, india_crop_mask,
                 ref_crop, pred_crop):
    rv = ref_2d.ravel()[india_flat]
    pv = pred_2d.ravel()[india_flat]
    if len(rv) < 100:
        return None
    out = {}
    out.update(pointwise_metrics(rv, pv))
    out["PSNR (dB)"] = psnr_metric(rv, pv)
    out["SSIM"]      = calc_ssim(ref_crop, pred_crop, india_crop_mask)
    out.update(correlation_and_skill(rv, pv))
    out.update(distributional_metrics(rv, pv))
    out.update(texture_metrics(ref_crop, pred_crop, india_crop_mask))
    out["Variance ratio"] = float(np.std(pv) / (np.std(rv) + 1e-12))
    return out


# =============================================================================
# INITIALISE MASK
# =============================================================================
print("\n[1] Building India mask ...")
with xr.open_dataset(HR_TEMPLATE) as ds:
    for k in ("lat", "latitude"):
        if k in ds.coords: lat = ds[k].values; break
    for k in ("lon", "longitude"):
        if k in ds.coords: lon = ds[k].values; break

india_mask      = build_india_mask(SHAPEFILE, lat, lon)
bbox_row, bbox_col = get_india_bbox(india_mask)
india_crop_mask = india_mask[bbox_row, bbox_col]
india_flat      = india_mask.ravel()

# Save mask so it can be reused independently if needed
np.save(os.path.join(OUTPUT_DIR, "india_mask_31k.npy"), india_mask)
print(f"  Mask saved → {OUTPUT_DIR}/india_mask_31k.npy")


# =============================================================================
# LOAD U-NET MODEL + NORMALIZER  (once, before the loop)
# =============================================================================
print("\n[2] Loading U-Net model ...")
unet_model = tf.keras.models.load_model(UNET_MODEL)
print("    Model loaded ✓")

with xr.open_dataset(UNET_CLIM) as ds:
    mean_map = ds["mean_map"].values
    std_map  = ds["std_map"].values
normalizer = Normalizer(mean_map, std_map)
print("    Climatology loaded ✓")


# =============================================================================
# FILE MAPS
# =============================================================================
print("\n[3] Building file maps ...")
map_mswx    = build_file_map(DIR_MSWX)
map_resunet = build_file_map(DIR_RESUNET)
map_ddpm    = build_file_map(DIR_DDPM,  suffix_strip="_ddpm")
map_lr_unet = build_lr_map_unet(DIR_LR_UNET)

print(f"    MSWX HR    : {len(map_mswx):,} files")
print(f"    ResU-Net   : {len(map_resunet):,} files")
print(f"    DDPM       : {len(map_ddpm):,} files")
print(f"    U-Net LR   : {len(map_lr_unet):,} files")

common = sorted(
    set(map_mswx) & set(map_resunet) & set(map_ddpm) & set(map_lr_unet))
N = len(common)
print(f"\n    Common dates (all 4 sources): {N}")
assert N > 0, "No common dates — check directory paths."


# =============================================================================
# MAIN EVALUATION LOOP
# =============================================================================
print(f"\n[4] Evaluating {N} days ...")
rec_unet    = []
rec_resunet = []
rec_ddpm    = []

for i, dk in enumerate(tqdm(common, desc="Evaluating")):
    try:
        # ── reference ────────────────────────────────────────────────────
        with xr.open_dataset(map_mswx[dk]) as ds:
            ref = extract(ds)

        # ── ResU-Net (pre-saved) ─────────────────────────────────────────
        with xr.open_dataset(map_resunet[dk]) as ds:
            pred_resunet = extract(ds)

        # ── DDPM (pre-saved) ─────────────────────────────────────────────
        with xr.open_dataset(map_ddpm[dk]) as ds:
            pred_ddpm = extract(ds)

        # ── U-Net (on-the-fly inference) ─────────────────────────────────
        with xr.open_dataset(map_lr_unet[dk]) as ds:
            lr_field = extract(ds)
        lr_field   = nearest_fill_nan(lr_field)
        pred_unet  = unet_predict(unet_model, lr_field, normalizer)

        # ── shape check ──────────────────────────────────────────────────
        assert ref.shape == india_mask.shape, \
            f"Shape mismatch: {ref.shape} vs {india_mask.shape}"

        # ── bbox crops ───────────────────────────────────────────────────
        ref_crop      = ref[bbox_row, bbox_col]
        crop_unet     = pred_unet[bbox_row, bbox_col]
        crop_resunet  = pred_resunet[bbox_row, bbox_col]
        crop_ddpm     = pred_ddpm[bbox_row, bbox_col]

        # ── metrics ───────────────────────────────────────────────────────
        mu = evaluate_day(ref, pred_unet,    india_flat, india_crop_mask,
                          ref_crop, crop_unet)
        mr = evaluate_day(ref, pred_resunet, india_flat, india_crop_mask,
                          ref_crop, crop_resunet)
        md = evaluate_day(ref, pred_ddpm,    india_flat, india_crop_mask,
                          ref_crop, crop_ddpm)

        if mu is not None: rec_unet.append(mu)
        if mr is not None: rec_resunet.append(mr)
        if md is not None: rec_ddpm.append(md)

    except Exception as e:
        print(f"  skip {dk}: {e}")

    if i % 200 == 0:
        gc.collect()

df_unet    = pd.DataFrame(rec_unet)
df_resunet = pd.DataFrame(rec_resunet)
df_ddpm    = pd.DataFrame(rec_ddpm)

print(f"\n    Processed: U-Net {len(df_unet)} | "
      f"ResU-Net {len(df_resunet)} | DDPM {len(df_ddpm)} days")


# =============================================================================
# METRIC TABLE DEFINITION
# =============================================================================
METRIC_GROUPS = {
    "Point-wise accuracy": [
        ("RMSE",             "degC",    ".3f"),
        ("MAE",              "degC",    ".3f"),
        ("Mean Bias",        "degC",    ".3f"),
        ("MaxAE",            "degC",    ".2f"),
    ],
    "Image quality": [
        ("PSNR (dB)",        "dB",      ".2f"),
        ("SSIM",             "-",       ".4f"),
    ],
    "Correlation and skill": [
        ("Pearson r",        "-",       ".4f"),
        ("R2",               "-",       ".4f"),
        ("KGE",              "-",       ".4f"),
        ("Willmott d",       "-",       ".4f"),
    ],
    "Distributional fidelity": [
        ("sigma ratio",      "-",       ".4f"),
        ("Skewness delta",   "-",       ".4f"),
        ("P05 error (degC)", "degC",    ".3f"),
        ("P95 error (degC)", "degC",    ".3f"),
        ("KS statistic",     "-",       ".4f"),
        ("Wasserstein-1",    "degC",    ".3f"),
    ],
    "Spatial texture": [
        ("Roughness ratio",  "-",       ".4f"),
        ("Texture corr.",    "-",       ".4f"),
        ("Gradient RMSE",    "degC/px", ".3f"),
        ("Variance ratio",   "-",       ".4f"),
    ],
}

DISPLAY_NAMES = {
    "RMSE":             "RMSE",
    "MAE":              "MAE",
    "Mean Bias":        "Mean Bias",
    "MaxAE":            "MaxAE",
    "PSNR (dB)":        "PSNR",
    "SSIM":             "SSIM",
    "Pearson r":        "Pearson r",
    "R2":               "R\u00b2",
    "KGE":              "KGE",
    "Willmott d":       "Willmott d",
    "sigma ratio":      "\u03c3 ratio",
    "Skewness delta":   "Skewness \u0394",
    "P05 error (degC)": "P05 error",
    "P95 error (degC)": "P95 error",
    "KS statistic":     "KS statistic",
    "Wasserstein-1":    "Wasserstein-1",
    "Roughness ratio":  "Roughness ratio",
    "Texture corr.":    "Texture corr.",
    "Gradient RMSE":    "Gradient RMSE",
    "Variance ratio":   "Variance ratio",
}

UNIT_DISPLAY = {
    "degC": "\u00b0C", "dB": "dB",
    "-": "\u2013", "degC/px": "\u00b0C/px",
}

LOWER_BETTER  = {"RMSE", "MAE", "MaxAE", "KS statistic", "Wasserstein-1",
                 "Gradient RMSE"}
HIGHER_BETTER = {"PSNR (dB)", "SSIM", "Pearson r", "R2", "KGE",
                 "Willmott d", "Texture corr."}
RATIO_ONE     = {"sigma ratio", "Roughness ratio", "Variance ratio"}
BIAS_ZERO     = {"Mean Bias", "Skewness delta",
                 "P05 error (degC)", "P95 error (degC)"}


def improvement_str(ref_mean, pred_mean, metric):
    """DDPM improvement relative to reference model."""
    if metric in RATIO_ONE:
        rd  = abs(ref_mean  - 1.0)
        pd_ = abs(pred_mean - 1.0)
        if rd < 1e-9: return "-"
        pct = (rd - pd_) / rd * 100
    elif metric in BIAS_ZERO:
        rd  = abs(ref_mean)
        pd_ = abs(pred_mean)
        if rd < 1e-9: return "-"
        pct = (rd - pd_) / rd * 100
    elif metric in LOWER_BETTER:
        rd  = abs(ref_mean)
        pd_ = abs(pred_mean)
        if rd < 1e-9: return "-"
        pct = (rd - pd_) / rd * 100
    elif metric in HIGHER_BETTER:
        if abs(ref_mean) < 1e-9: return "-"
        pct = (pred_mean - ref_mean) / abs(ref_mean) * 100
    else:
        return "-"
    return f"+{pct:.1f}%" if pct >= 0 else f"{pct:.1f}%"


def fmt(mean, std, spec):
    return f"{mean:{spec}} \u00b1 {std:{spec}}"


# =============================================================================
# BUILD SUMMARY TABLE
# Columns: U-Net | ResU-Net | DDPM | Δ vs U-Net | Δ vs ResU-Net
# =============================================================================
rows = []
for group, metrics in METRIC_GROUPS.items():
    for mname, unit, spec in metrics:
        if mname not in df_resunet.columns:
            continue

        u_m, u_s = df_unet[mname].mean(),    df_unet[mname].std()
        r_m, r_s = df_resunet[mname].mean(), df_resunet[mname].std()
        d_m, d_s = df_ddpm[mname].mean(),    df_ddpm[mname].std()

        rows.append({
            "Category":        group,
            "Metric":          mname,
            "Display":         DISPLAY_NAMES.get(mname, mname),
            "Unit":            UNIT_DISPLAY.get(unit, unit),
            "Unit_raw":        unit,
            "Spec":            spec,
            "U-Net":           fmt(u_m, u_s, spec),
            "ResU-Net":        fmt(r_m, r_s, spec),
            "DDPM":            fmt(d_m, d_s, spec),
            "d vs U-Net":      improvement_str(u_m, d_m, mname),
            "d vs ResU-Net":   improvement_str(r_m, d_m, mname),
            # raw means for downstream inspection
            "U-Net_mean":      u_m,
            "ResU-Net_mean":   r_m,
            "DDPM_mean":       d_m,
            "U-Net_std":       u_s,
            "ResU-Net_std":    r_s,
            "DDPM_std":        d_s,
        })

df_table = pd.DataFrame(rows)


# =============================================================================
# OUTPUT: CSV
# =============================================================================
csv_path = os.path.join(OUTPUT_DIR, "metrics_table_3model.csv")
df_table.drop(columns=["Unit_raw", "Spec",
                        "U-Net_mean",    "ResU-Net_mean",    "DDPM_mean",
                        "U-Net_std",     "ResU-Net_std",     "DDPM_std",
                        ]).to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")


# =============================================================================
# OUTPUT: PLAIN TEXT
# =============================================================================
txt_path = os.path.join(OUTPUT_DIR, "metrics_summary_3model.txt")
W = 140
with open(txt_path, "w") as f:
    f.write("Comprehensive Evaluation: U-Net | ResU-Net | DDPM\n")
    f.write(f"Test period  : {N} days\n")
    f.write(f"India pixels : {int(india_mask.sum()):,}  "
            f"(0.05° buffered IMD boundary shapefile)\n")
    f.write("=" * W + "\n")
    current = None
    for _, row in df_table.iterrows():
        if row["Category"] != current:
            current = row["Category"]
            f.write(f"\n{'-'*W}\n  {current.upper()}\n{'-'*W}\n")
            f.write(f"  {'Metric':<22} {'Unit':<8} {'U-Net':<24} "
                    f"{'ResU-Net':<24} {'DDPM':<24} "
                    f"{'d_UNet':>9} {'d_ResUNet':>10}\n")
            f.write(f"  {'-'*22} {'-'*8} {'-'*24} {'-'*24} "
                    f"{'-'*24} {'-'*9} {'-'*10}\n")
        f.write(f"  {row['Display']:<22} {row['Unit']:<8} "
                f"{row['U-Net']:<24} {row['ResU-Net']:<24} "
                f"{row['DDPM']:<24} "
                f"{row['d vs U-Net']:>9} {row['d vs ResU-Net']:>10}\n")
print(f"Saved → {txt_path}")


# =============================================================================
# OUTPUT: LaTeX booktabs
# =============================================================================
def to_latex(df, n_days, n_pixels):
    def tex(s):
        return (s.replace("\u03c3",  r"$\sigma$")
                 .replace("\u0394",  r"$\Delta$")
                 .replace("R\u00b2", r"$R^2$")
                 .replace("\u00b2",  r"$^2$")
                 .replace("\u00b0C", r"$^\circ$C")
                 .replace("\u00b1",  r"$\pm$")
                 .replace("\u2013",  "--"))

    lines = [
        r"\begin{table*}[ht]",
        r"\centering",
        (r"\caption{Comprehensive performance comparison of U-Net, Residual U-Net "
         r"(ResU-Net), and the proposed DDPM over the full test period "
         f"({n_days} days, August 2009--December 2014). "
         r"Values report mean $\pm$ standard deviation across all test days. "
         r"All metrics are computed exclusively over India land pixels "
         f"({n_pixels:,} grid cells at $0.1^\\circ$ resolution, "
         r"defined by the IMD political boundary shapefile with 0.05$^\circ$ buffer). "
         r"$\Delta$ columns report DDPM improvement relative to each baseline; "
         r"positive values indicate DDPM outperformance in all cases.}"),
        r"\label{tab:comprehensive_metrics}",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{4pt}",
        r"\begin{tabular}{@{}ll rrr rr@{}}",
        r"\toprule",
        (r"\textbf{Metric} & \textbf{Unit} & "
         r"\textbf{U-Net} & \textbf{ResU-Net} & \textbf{DDPM} & "
         r"\textbf{$\Delta$\,U-Net} & \textbf{$\Delta$\,ResU-Net} \\"),
        r"\midrule",
    ]
    current = None
    for _, row in df.iterrows():
        if row["Category"] != current:
            if current is not None:
                lines.append(r"\addlinespace[4pt]")
            current = row["Category"]
            lines.append(r"\multicolumn{7}{@{}l}{\textit{"
                         + current + r"}} \\")
            lines.append(r"\addlinespace[2pt]")
        lines.append(
            f"\\quad {tex(row['Display'])} & {tex(row['Unit'])} & "
            f"{tex(row['U-Net'])} & {tex(row['ResU-Net'])} & "
            f"{tex(row['DDPM'])} & "
            f"{row['d vs U-Net']} & {row['d vs ResU-Net']} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
    return "\n".join(lines)


tex_path = os.path.join(OUTPUT_DIR, "metrics_table_3model.tex")
with open(tex_path, "w") as f:
    f.write(to_latex(df_table, N, int(india_mask.sum())))
print(f"Saved → {tex_path}")


# =============================================================================
# CONSOLE SUMMARY
# =============================================================================
print("\n" + "=" * 130)
print("  U-Net | ResU-Net | DDPM  —  India land pixels only")
print(f"  {N} test days  |  {int(india_mask.sum()):,} pixels at 0.1°")
print("=" * 130)

current = None
for _, row in df_table.iterrows():
    if row["Category"] != current:
        current = row["Category"]
        print(f"\n  {current.upper()}")
        print(f"  {'Metric':<22} {'Unit':<8} {'U-Net':<24} "
              f"{'ResU-Net':<24} {'DDPM':<24} "
              f"{'d_UNet':>9} {'d_ResUNet':>10}")
        print(f"  {'-'*22} {'-'*8} {'-'*24} {'-'*24} "
              f"{'-'*24} {'-'*9} {'-'*10}")
    print(f"  {row['Display']:<22} {row['Unit']:<8} "
          f"{row['U-Net']:<24} {row['ResU-Net']:<24} "
          f"{row['DDPM']:<24} "
          f"{row['d vs U-Net']:>9} {row['d vs ResU-Net']:>10}")

print(f"\nOutputs:")
print(f"  {csv_path}")
print(f"  {txt_path}")
print(f"  {tex_path}")
print("\nDone.")